In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import h5py
from scipy.integrate import simpson
singlecol = 3.5
plt.rcParams.update({'font.size': 8})
#plt.rcParams.update({'font.sans-serif': 'Helvetica'})|
plt.rcParams.update({'lines.linewidth':1})
plt.rcParams.update({'axes.linewidth':1})
plt.rcParams.update({'lines.markersize':5})
plt.rcParams.update({'savefig.dpi':200})
plt.rcParams.update({'legend.fontsize': 8})

In [ ]:
# At the very end of your plotting cell
plt.show()
plt.close('all')

## QENS

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from lmfit import Model # for fit

from math import pi, sin, exp, sqrt

title_font = {'fontname':'Arial', 'size':'16', 'color':'black', 'weight':'normal',
             'verticalalignment':'bottom'} # Bottom vertical alignment for more space
axis_font = {'fontname':'Arial', 'size':'14'}




def Gaussian_Model(s = 1., Dloc = 1e-6, Taumi = 1., Dlr = 0., Plot = False):
    '''
    fit (I(q,t)_loc - EISF)*I(q,t)_lr with 1 lorentzian
    export HWHM of Lor and EISF
    EISF should be exp(-Q**2 * s**2)
    
    fit is not perfect for all Q values, for instance s = 1., Dloc = 1e-5, Taumi = 1 at Q = 0.4
    G is found to be 8 µeV
    I check with qensh for data on IN6 : gaussian model can be fitted by 1 lorentzian, G = 10µeV. fit is very good
    Is it due to the convolution with the resolution ? 
    '''
    
    def Lorentz(t, tau = 1.):    # Lor*(1-EISF)
        return (np.exp(-t / tau)*(1-np.exp(-q**2*s**2)))

    Dloc = Dloc * 1e4 # A2 / ps
    Dlr = Dlr * 1e4 # A2 / ps
    
    t = np.linspace(1, 1000, 10000)  # ps
    t = np.array(t)
    Q = np.linspace(0.1,2.2,15)     # A-1
    G = np.empty_like(Q)
    #EISF = np.empty_like(Q)
    rho_t = np.empty_like(t)
    
    for i, q in enumerate(Q):
        rho_t = np.exp(-t/( s**2 / Dloc *(1 + 2 * Dloc * q**2 * Taumi)))        # tau_gaussian = 2 taumi (jump time)
        I = (np.exp(-q**2*s**2*(1-rho_t))-exp(-q**2*s**2))*np.exp(-Dlr*q**2*t)
                            
        gmodel = Model(Lorentz)
        Lor_res = gmodel.fit(I, t = t, tau = 1, EISF = 0.5, nan_policy = 'propagate')
        tau_temp = Lor_res.best_values['tau']
        G[i] = 0.658 / tau_temp
        #EISF[i] = Lor_res.best_values['EISF']
        
        #plt.close()
        if Plot is True:
            plt.figure()
            plt.plot(t, I, label = 'gaussian model')
            plt.plot(t, Lor_res.best_fit, label = 'fit Q = %0.2f'%q)
            plt.legend()
        #plt.pause(1)

            plt.show()
        
    return G, Q
sigma = 1
dloc = 0.35e-5
taumi = 2
G, Q = Gaussian_Model(s = sigma, Dloc = dloc, Taumi = taumi)
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})
plt.figure(figsize=(singlecol*2,singlecol*2))
plt.plot(Q, G*2, color='green')#, label = 'Gaussian Model $\sigma$ = %0.2f A,\nDloc = %0.2e cm^2/s, Tau = %0.2f ps '%(sigma, dloc, taumi))

plt.errorbar(x, y*2, yerr=error*2, fmt='o', color='purple', ecolor='purple', capsize=3, label='IN16b Data')
plt.plot(x,y*2, color='purple')
plt.xlabel('q (1/Å)')
plt.ylabel('FWHM (meV)')
plt.tight_layout()
plt.xlim(0, 1.75)
plt.ylim(-0.005, 0.09)
#plt.legend()



In [ ]:
IN16b = np.genfromtxt('./hydration/QENS_exp/PBA_2H2O_373K_in16B_L1free_Dloc.blk',skip_header=1, delimiter=',')
'''  Qensh parameters with fit 3 lor
bkdg : [:,16]
Iel : [:,18]
Iqel1 : [:,20]
G1 : [:,22]
Iqel2 : [:,26]
G2 : [:,28]
Iqel3 : [:32]
G3 : [:34]
'''
plt.figure(figsize=(singlecol*2,singlecol*2))
x = IN16b[:, 0]
y = IN16b[:, 22]
error = IN16b[:, 23]

# Create the plot
plt.errorbar(x, y*2, yerr=error*2, fmt='o', color='purple', ecolor='purple', capsize=3, label='IN16b Data')
plt.plot(x,y*2, color='purple')

# teixeira    
def jump_diff(x, D=1.014e-5, t=1):
     return (6.58e3*D*x**2/(1+6.58e3*D*x**2*t/0.658))
# d = (6Dt)^0.5

def SS(x, tau = 1, r = 2):
     return (0.658/(6*tau)*((x**2*r**2)/(1+x**2*r**2/6)))
# distribution of jump distance. real param is <r>^2
# D = <r>^2/(6tau)

def HR(x, tau = 1., r = 2.):  
    return 0.658/tau*(1-np.exp(-(x**2*r**2/3)/2))
# distribution of jump distance. real param is <r>^2
# <r>2 = 3r0^2
# D = <r>^2/(6tau)

Q = np.linspace(0.1, 2, 100)
G= jump_diff(Q, D = 2e-7, t = 47)
plt.plot(Q, G*2, color='orange')


plt.xlabel('q (1/Å)')
plt.ylabel('FWHM (meV)')
plt.tight_layout()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})
title_font = {'fontname':'Arial', 'size':'16', 'color':'black', 'weight':'normal', 'verticalalignment':'bottom'} # Bottom vertical alignment for more space
axis_font = {'fontname':'Arial', 'size':'14'}

'''  Qensh parameters with fit 3 lor
bkdg : [:,16]
Iel : [:,18]
Iqel1 : [:,20]
G1 : [:,22]
Iqel2 : [:,26]|
G2 : [:,28]
Iqel3 : [:32]
G3 : [:34]
'''

sharper_2H2O_1 = np.genfromtxt('./hydration/QENS_exp/PBA_2H2O_373K_grouped2_NormToMass.txt_resvana_resolution_corr_grouped2_29.68.txt_Buff__Lyonnard_Gaussian_Model__qenshfit_result_sbs6_q1.43_TueMar0315_59_452026.dat',skip_header=1, delimiter=',')

plt.figure(figsize=(singlecol*2,singlecol*2))
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,1], 'o', mfc='none', label = 'data', color='black')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,2], label = 'fit', color='r')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,4], label = 'Elastic')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,6], label = 'Lorentzian  2', color='purple')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,5], label = 'Lorentzian  3', color='green')
#plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,7], label='x')
#plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,7], label = 'Lor 3')

plt.xlabel("Energy transfer (meV)", **axis_font)
plt.ylabel("Intensity (a.u.)", **axis_font)
plt.semilogy()
plt.legend(frameon=False)
plt.xlim(-0.015,0.015)
plt.ylim(1,2000)
plt.tight_layout()
plt.show()
plt.savefig('QENS_IN16b.svg')


In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'lines.linewidth':1.5})
plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})
title_font = {'fontname':'Arial', 'size':'16', 'color':'black', 'weight':'normal', 'verticalalignment':'bottom'} # Bottom vertical alignment for more space
axis_font = {'fontname':'Arial', 'size':'14'}

'''  Qensh parameters with fit 3 lor
bkdg : [:,16]
Iel : [:,18]
Iqel1 : [:,20]
G1 : [:,22]
Iqel2 : [:,26]
G2 : [:,28]
Iqel3 : [:32]
G3 : [:34]
'''

sharper_2H2O_1 = np.genfromtxt('./hydration/QENS_exp/SQW_PB~1.DAT',skip_header=1, delimiter=',')

plt.figure(figsize=(singlecol*2,singlecol*2))
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,1], 'o', mfc='none', label = 'data', color='black')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,2],  label = 'fit', color='r')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,4], label = 'Elastic')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,6], '--', label = 'Lorentzian 1', color='red')
plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,5], label = 'Lorentzian 2', color='green')
#plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,7], label='x')
#plt.plot(sharper_2H2O_1[:,0], sharper_2H2O_1[:,7], label = 'Lor 3')

plt.xlabel("Energy transfer (meV)", **axis_font)
plt.ylabel("Intensity (a.u.)", **axis_font)
plt.semilogy()
plt.legend(frameon=False)
#plt.xlim(-0.015,0.015)
#plt.ylim(1,2000)
plt.tight_layout()
plt.show()
plt.savefig('SHARPER.svg')


In [ ]:
plt.figure(figsize=(singlecol*2,singlecol*2))

plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})

data = np.loadtxt('./hydration/QENS_exp/gdos_PBA_1H2O_SofQW_018270_Ei2meV_T300.0K')
col1, col2, col3 = data[:, 0], data[:, 1], data[:, 2]
col1 = col1[::-1]
plt.plot(col1+abs(col1[-1]), col2, label='PBA•0H2O 300 K')

data = np.loadtxt('./hydration/QENS_exp/gdos_PBA_2H2O_300_SofQW_017136_Ei2meV_T298.0K')
col1, col2, col3 = data[:, 0], data[:, 1], data[:, 2]
col1 = col1[::-1]
plt.plot(col1+abs(col1[-1]), col2, label='PBA•2H2O 300 K')

data = np.loadtxt('./hydration/QENS_exp/gdos_PBA_2H2O_373_SofQW_017147_Ei2meV_T380.0K')
col1, col2, col3 = data[:, 0], data[:, 1], data[:, 2]
col1 = col1[::-1]
#plt.plot(col1+abs(col1[-1]), col2, label='PBA•2H2O 373 K')

plt.xlabel("Energy (meV)")
plt.ylabel("Intensity (a.u.)")

plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()
plt.show()
plt.savefig('vDOS.svg')

In [ ]:
import numpy as np

def load_spectra(filename):
    spectra = {}
    current_meta = {}
    current_data = []

    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('#'):
                # Parse metadata
                if 'Spectrum No.' in line:
                    # Save previous spectrum if exists
                    if current_meta and current_data:
                        current_meta['data'] = np.array(current_data)
                        spectra[current_meta['spectrum']] = current_meta
                    # Start new spectrum
                    current_meta = {'spectrum': int(line.split()[-1])}
                    current_data = []
                elif 'q(Angstrom^-1)' in line:
                    current_meta['q'] = float(line.split('=')[-1])
                elif 'Scattering Angle' in line:
                    current_meta['angle'] = float(line.split('=')[-1])
            else:
                current_data.append([float(x) for x in line.split()])

    # Don't forget the last spectrum
    if current_meta and current_data:
        current_meta['data'] = np.array(current_data)
        spectra[current_meta['spectrum']] = current_meta

    return spectra

In [ ]:
plt.figure(figsize=(singlecol*2,singlecol*2))

file_ls = ['PBA_2H2O_heating_grouped2_2.00.dat',
           'PBA_2D2O_heating_grouped2_2.00.dat',
           'PBA_0H2O_heating_grouped2_2.00.dat',
           'PBA_0D2O_heating_grouped2_2.00.dat']

name_ls = ['PBA•2H2O','PBA•2D2O','PBA•0H2O','PBA•0D2O']

for i in range(4):
    spectra = load_spectra(f'./hydration/QENS_exp/{file_ls[i]}')
    s0 = spectra[8]
    temperature = s0['data'][:, 0]
    intensity  = s0['data'][:, 1]
    error      = s0['data'][:, 2]
    q          = s0['q']      # 0.19
    angle      = s0['angle']  # 10.94
    
    plt.plot(temperature, intensity, label=f'{name_ls[i]}')

plt.xlabel("Temperature (K)")
plt.ylabel("Intensity (a.u.)")

plt.legend(frameon=False)
plt.xlim(20,375)
plt.tight_layout()
plt.savefig('IFWS.svg')

In [ ]:
spectra.keys()

In [ ]:
plt.figure(figsize=(singlecol*2,singlecol*1.5))

s0 = spectra[8]
temperature = s0['data'][:, 0]
intensity  = s0['data'][:, 1]
error      = s0['data'][:, 2]
q          = s0['q']      # 0.19
angle      = s0['angle']  # 10.94

plt.plot(temperature, intensity)

plt.xlabel("Temperature (K)")
plt.ylabel("Intensity (a.u.)")

plt.legend(frameon=False)
plt.xlim(20,375)
plt.tight_layout()

### Convert

In [ ]:
#!/usr/bin/env python

import os

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    VECLIB_MAXIMUM_THREADS="1",
    NUMEXPR_NUM_THREADS="1",
)

########################################################
# This is an automatically generated MDANSE run script #
########################################################

from MDANSE.Framework.Converters.Converter import Converter

########################################################
# Job parameters                                       #
########################################################

parameters = {
    #"atom_aliases": "",
    "config_file": "./hydration/400_diffusion_4/400K_Na_16_O_32.data",  # LAMMPS configuration file
    "fold": False,  # Fold coordinates in to box
    "lammps_units": "metal",  # LAMMPS unit system
    "n_steps": "0",  # Number of time steps (0 for automatic detection)
    "output_files": (
        "./hydration/QENS/400K_9_convert_trj",
        32,
        128,
        "gzip",
        "INFO",
    ),  # MDANSE trajectory (filename, datatype, chunk size, compression, logfile output)
    "time_step": "0.001",  # Time step (lammps units, depends on unit system)
    "trajectory_file": "./hydration/400_diffusion_4/dump.400_diffuse_9.trj",  # LAMMPS trajectory file
    "trajectory_format": "custom",  # LAMMPS trajectory format
}

########################################################
# Setup and run the analysis                           #
########################################################

if __name__ == "__main__":
    lammps = Converter.create("LAMMPS")
    # Progress bars only available if tqdm available.
    # Install with `cli` optional dependency.
    lammps.run(parameters, status=True, prog_bar=True)

In [ ]:
with h5py.File("./hydration/QENS/400K_9/400K_9_convert_trj_long.mdt", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
file_path = "./hydration/QENS/400K_9_convert_trj.mdt"

with h5py.File(file_path, "r") as f:
    C = f["metadata/inputs/atom_aliases"][:]
    #rmsd_o = f["rmsd/O"][:] # read Na RMSD array
    #time = f["rmsd/axes/time"][:]    # read time axis
C

### MSD

In [ ]:
#!/usr/bin/env python

import os

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    VECLIB_MAXIMUM_THREADS="1",
    NUMEXPR_NUM_THREADS="1",
)

########################################################
# This is an automatically generated MDANSE run script #
########################################################

from MDANSE.Framework.Jobs.IJob import IJob

########################################################
# Job parameters                                       #
########################################################

parameters = {
    "atom_transmutation": "{}",  # atom_transmutation
    "frames": [35, 2035, 1],  # frames
    "grouping_level": "atom",  # grouping_level
    "output_files": (
        "./hydration/QENS/400K_9_displacement",
        ["MDAFormat"],
        "INFO",
    ),  # output_files
    "reference_frame": "0",  # reference_frame
    "running_mode": ("single-core",),  # running_mode
    "trajectory": "./hydration/QENS/400K_9_convert_trj.mdt",  # trajectory
}

########################################################
# Setup and run the analysis                           #
########################################################

if __name__ == "__main__":
    rootmeansquaredeviation = IJob.create("RootMeanSquareDeviation")
    # Progress bars only available if tqdm available.
    # Install with `cli` optional dependency.
    rootmeansquaredeviation.run(parameters, status=True, prog_bar=True)

In [ ]:
with h5py.File("./hydration/QENS/400K_9_displacement.mda", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
file_path = "./hydration/QENS/400K_9_displacement.mda"

with h5py.File(file_path, "r") as f:
    rmsd_na = f["rmsd/Na"][:]
    rmsd_o = f["rmsd/O"][:] # read Na RMSD array
    time = f["rmsd/axes/time"][:]    # read time axis

# Plot
plt.plot(time*1000, rmsd_na*100)
plt.plot(time*1000, rmsd_o*100)
plt.xlabel("Time")
plt.ylabel("RMSD (Na)")
plt.title("RMSD of Na vs Time")
plt.grid(True)
plt.show()

### van Hove Gs(r,t)

In [ ]:
#!/usr/bin/env python
os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    VECLIB_MAXIMUM_THREADS="1",
    NUMEXPR_NUM_THREADS="1",
)

########################################################
# This is an automatically generated MDANSE run script #
########################################################

from MDANSE.Framework.Jobs.IJob import IJob

########################################################
# Job parameters                                       #
########################################################

parameters = {
    "frames": [35, 2035, 1, 1000],  # frames
    "grouping_level": "atom",  # grouping_level
    "output_files": (
        "./hydration/QENS/400K_9_vanhovefunctionself_1000",
        ["MDAFormat"],
        "INFO",
    ),  # output_files
    "r_values": [0.0, 0.5, 0.005],  # r values (nm)
    #"running_mode": ("single-core",),  # running_mode
    "running_mode": ("multicore", 60),
    #"atom_selection": ("./hydration/QENS/400K_9_convert_trj.mdt", {'Na'}),
    #"atom_selection": ([:10]),
    "trajectory": "./hydration/QENS/400K_9_convert_trj.mdt",  # trajectory
    "weights": "equal",  # weights
}

########################################################
# Setup and run the analysis                           #
########################################################

if __name__ == "__main__":
    vanhovefunctionself = IJob.create("VanHoveFunctionSelf")
    # Progress bars only available if tqdm available.
    # Install with `cli` optional dependency.
    vanhovefunctionself.run(parameters, status=True, prog_bar=True)

In [ ]:
with h5py.File("./hydration/QENS/400K_9/400K_9_vanhovefunctionself_1000.mda", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
r

In [ ]:
file_path = "./hydration/QENS/400K_9_vanhovefunctionself_1000.mda"

with h5py.File(file_path, "r") as f:
    C_vhs_1 = f["vh/g(r,t)/C"][:]
    Mn_vhs_1 = f["vh/g(r,t)/Mn"][:]
    Fe_vhs_1 = f["vh/g(r,t)/Fe"][:]
    N_vhs_1 = f["vh/g(r,t)/N"][:]
    Na_vhs_1 = f["vh/g(r,t)/Na"][:]
    O_vhs_1 = f["vh/g(r,t)/O"][:]
    H_vhs_1 = f["vh/g(r,t)/H"][:]

In [ ]:
file_path = "./hydration/QENS/400K_9_vanhovefunctionself_1000.mda"

with h5py.File(file_path, "r") as f:
    total_vhs_1000 = f["vh/g(r,t)/total"][:]
    C_vhs_1000 = f["vh/g(r,t)/C"][:]
    Mn_vhs_1000 = f["vh/g(r,t)/Mn"][:]
    Fe_vhs_1000 = f["vh/g(r,t)/Fe"][:]
    N_vhs_1000 = f["vh/g(r,t)/N"][:]
    Na_vhs_1000 = f["vh/g(r,t)/Na"][:]
    O_vhs_1000 = f["vh/g(r,t)/O"][:]
    H_vhs_1000 = f["vh/g(r,t)/H"][:]
    r = f["vh/axes/r"][:]
    t = f["vh/axes/time"][:]


In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_10_SVH_select.mda"

with h5py.File(file_path, "r") as f:
    total_vhs_1000 = f["vh/4_pi_r2_g(r,t)/total"][:]
    #C_vhs_1000 = f["vh/4_pi_r2_g(r,t)/C"][:]
    #Mn_vhs_1000 = f["vh/4_pi_r2_g(r,t)/Mn"][:]
    #Fe_vhs_1000 = f["vh/4_pi_r2_g(r,t)/Fe"][:]
    #N_vhs_1000 = f["vh/4_pi_r2_g(r,t)/N"][:]
    #Na_vhs_1000 = f["vh/4_pi_r2_g(r,t)/Na"][:]
    O_vhs_1000 = f["vh/4_pi_r2_g(r,t)/O"][:]
    H_vhs_1000 = f["vh/4_pi_r2_g(r,t)/H"][:]
    r = f["vh/axes/r"][:]
    t = f["vh/axes/time"][:]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Your existing code...
datasets = [
    (total_vhs_1000, "total"),
    #(Na_vhs_1000,  "Na"),
    (O_vhs_1000,  "O"),
    (H_vhs_1000, "H"),
    #(Fe_vhs_1000,  "Fe"),
    #(N_vhs_1000, "N"),
]

indices = range(1,len(t),2)
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2), sharex=True, dpi=100)
fig.subplots_adjust(wspace=0.25)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color)
    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel(r"$4\pi r^2\, G_s(r,t)$ (a.u.)")
    ax.set_xlim(0, 6)
    ax.set_yticks([])
    #ax.set_yscale('log')
    #ax.grid(True)
plt.tight_layout()
#plt.savefig('./hydration/VH_total_Na_O_H.svg')

In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_1_SVH.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_1 = f["vh/4_pi_r2_g(r,t)/H"][:]

file_path = "./hydration/QENS/400K_vac/400K_2_SVH.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_2 = f["vh/4_pi_r2_g(r,t)/H"][:]

file_path = "./hydration/QENS/400K_vac/400K_5_SVH.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_5 = f["vh/4_pi_r2_g(r,t)/H"][:]

file_path = "./hydration/QENS/400K_vac/400K_4_SVH.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_4 = f["vh/4_pi_r2_g(r,t)/H"][:]

file_path = "./hydration/QENS/400K_vac/400K_10_SVH.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_10 = f["vh/4_pi_r2_g(r,t)/H"][:]

file_path = "./hydration/QENS/400K_vac/400K_10_SVH_select.mda"
with h5py.File(file_path, "r") as f:
    H_vhs_10_select = f["vh/4_pi_r2_g(r,t)/H"][:]
    r = f["vh/axes/r"][:]
    t = f["vh/axes/time"][:]


# Your existing code...
datasets = [
    (H_vhs_1, "1"),
    (H_vhs_2, "2"),
    (H_vhs_4, "4"),
    (H_vhs_5, "5"),
    (H_vhs_10,  "10"),
    (H_vhs_10_select, "10_vac"),
]

indices = range(1,len(t),2)
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(3, 2, figsize=(singlecol*2, singlecol*3), sharex=True, dpi=100)
fig.subplots_adjust(wspace=0.25)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color)
    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel(r"$4\pi r^2\, G_s(r,t)$ (a.u.)")
    ax.set_xlim(0, 7)
    ax.set_yticks([])
    #ax.set_yscale('log')
    #ax.grid(True)
plt.tight_layout()
#plt.savefig('./hydration/VH_total_Na_O_H.svg')

In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_5_SVH.mda"

with h5py.File(file_path, "r") as f:
    total_vhs_1000 = f["vh/4_pi_r2_g(r,t)/total"][:]
    O_vhs_1000 = f["vh/4_pi_r2_g(r,t)/O"][:]
    H_vhs_1000 = f["vh/4_pi_r2_g(r,t)/H"][:]
    r = f["vh/axes/r"][:]
    t = f["vh/axes/time"][:]

# Your existing code...
datasets = [
    (total_vhs_1000, "total"),
    #(Na_vhs_1000,  "Na"),
    (O_vhs_1000,  "O"),
    (H_vhs_1000, "H"),
    #(Fe_vhs_1000,  "Fe"),
    #(N_vhs_1000, "N"),
]

indices = range(1,len(t),2)
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2), sharex=True, dpi=100)
fig.subplots_adjust(wspace=0.25)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color)
    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel(r"$4\pi r^2\, G_s(r,t)$ (a.u.)")
    ax.set_xlim(0, 6)
    ax.set_yticks([])
    #ax.set_yscale('log')
    #ax.grid(True)
plt.tight_layout()
#plt.savefig('./hydration/VH_total_Na_O_H.svg')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth': 2})
plt.rcParams.update({'axes.linewidth': 1})
plt.rcParams.update({'lines.markersize': 5})
plt.rcParams.update({'savefig.dpi': 300})
plt.rcParams.update({'legend.fontsize': 8})

# Your existing code...
datasets = [
    (Fe_vhs_1000,  "Fe"),
    (Mn_vhs_1000,  "Mn"),
    (C_vhs_1000,  "C"),
    (N_vhs_1000,  "N")
    #(total_vhs_1000, "total"),
    #(Na_vhs_1000,  "Na"),
    #(O_vhs_1000,  "O"),
    #(H_vhs_1000, "H"),
    #(Fe_vhs_1000,  "Fe"),
    #(N_vhs_1000, "N"),
]

indices = range(1,len(t),10)
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2), sharex=True, dpi=300,)
fig.subplots_adjust(wspace=0.25)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color)
    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel(r"$4\pi r^2\, G_s(r,t)$")
    ax.set_xlim(0, 4)
    #ax.set_yscale('log')
    #ax.grid(True)
plt.tight_layout()
#plt.savefig('./hydration/VH_Fe_Mn_C_N.svg')

In [ ]:
Na_vhs_1000[15:25, 50].argmax(),r2

In [ ]:
Na_vhs_1000[25:40, 50].argmax(), r3

In [ ]:
plt.plot(Na_vhs_1000[:, 900])
#plt.xlim(0,80)

In [ ]:
r1,r2,r3,r4,r5

In [ ]:
plt.plot(r*10, Na_vhs_1000[:, 800], color=color)
r1 = r[Na_vhs_1000[:20, 100].argmax()]*10
r2 = 2.1#r[Na_vhs_1000[15:23, 100].argmax()+15]*10
r3 = r[Na_vhs_1000[25:40, 100].argmax()+25]*10
r4 = r[Na_vhs_1000[45:55, 500].argmax()+45]*10
r5 = r[Na_vhs_1000[57:66, 800].argmax()+57]*10
r6 = 7.24
r7 = 8.1

plt.vlines(r1,0,1)
plt.vlines(r2,0,1)
plt.vlines(r3,0,1)
plt.vlines(r4,0,1)
plt.vlines(r5,0,1)
plt.vlines(r6,0,1)
plt.vlines(r7,0,1)
#plt.xlim(0,5)

In [ ]:
%matplotlib widget
plt.figure( figsize=(singlecol*2, singlecol*2))
cmap = plt.get_cmap("jet")

for i in range(1,490,10):
    color = cmap(i / (490 - 1))
    plt.plot(r*10, Na_vhs_1000[:, i], color=color)

r_centers = [r1, r2, r3, r4, r5, r6, r7]

cmap = plt.get_cmap("plasma")
for i in range(len(r_centers)):
    color = cmap(i / len(r_centers))
    plt.axvspan(r_centers[i]-0.25, r_centers[i]+0.25, 0, 2, alpha=0.2, color=color)

plt.xlabel("r (Å)")
plt.ylabel(r"$4\pi r^2\, G_s(r,t)$")
#plt.yscale('log')
plt.xlim(0,10)

In [ ]:
H_vhs_1000[0, i]

In [ ]:
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth': 2})
plt.rcParams.update({'axes.linewidth': 1})
plt.rcParams.update({'lines.markersize': 5})
plt.rcParams.update({'savefig.dpi': 300})
plt.rcParams.update({'legend.fontsize': 8})
plt.figure( figsize=(singlecol*2, singlecol*2))
cmap = plt.get_cmap("jet")

for i in range(1,1000,5):
    color = cmap(i / (1000 - 1))
    plt.plot(r*10, H_vhs_1000[:, i], color=color)
plt.xlabel("r (Å)")
plt.ylabel("g(r, t)")
#plt.xlim(0,4)

In [ ]:
t[250]

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})
plt.rcParams.update({'lines.linewidth': 2})
plt.rcParams.update({'axes.linewidth': 1})
plt.rcParams.update({'lines.markersize': 5})
plt.rcParams.update({'savefig.dpi': 300})
plt.rcParams.update({'legend.fontsize': 8})
plt.figure( figsize=(singlecol*1.2, singlecol*1.2))
cmap = plt.get_cmap("jet")

for i in range(1,250,1):
    color = cmap(i / (250 - 1))
    plt.plot(r*10, Na_vhs_1000[:, i], color=color)
plt.xlabel("r (Å)")
plt.ylabel(r"$4\pi r^2\, G_s(r,t)$")
plt.xlim(0,10)
plt.tight_layout()
#plt.savefig('./hydration/VH_400K_9.svg')

In [ ]:
plt.plot(Na_vhs_1000[0, 1:])

popt, pcov = curve_fit(exponential_with_bg, t[1:], H_vhs_1000[0, 1:], 
                       p0=[initial_tau, 0], 
                       bounds=([0, 0], [np.inf, 10.0]),
                       maxfev=10000)

plt.plot(t, exponential_with_bg(t, *popt), 'r-', linewidth=2, 
               label=f'KWW Fit (τ={tau_extracted:.3f}, β={beta_extracted:.3f})')

In [ ]:
rc_idx_1

In [ ]:
cmap = plt.get_cmap("plasma")
for i in range(len(r_centers)):
    color = cmap(i / len(r_centers))
    
    # Find indices for radial window
    rc_idx_1 = np.searchsorted(r, 0.1*r_centers[i] - 0.025)
    rc_idx_2 = np.searchsorted(r, 0.1*r_centers[i] + 0.025)
    
    # Extract data within radial window
    r_clipped_1 = r[rc_idx_1:rc_idx_2]
    gs_clipped_1 = Na_vhs_1000[rc_idx_1:rc_idx_2, :]
    
    # Integrate over radius (axis=0) for each time point
    P_t_1 = simpson(gs_clipped_1, x=r_clipped_1, axis=0)
    P_t_1 /= P_t_1.max()
    
    # Remove 'r' from format string since we're using color parameter
    plt.plot(t[1:], P_t_1[1:], '.-', alpha=0.6, 
             label=f'{r_centers[i]:.2f} Å (±0.25 Å)', 
             color=color)

plt.xlabel('Correlation time (ps)')
plt.ylabel('Normalized P(t)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2))#, dpi=300)

# Top plot: Gs(r,t) profiles at different time

r_centers = [0.45,2.1,3.55,5.15,6.35,7.24,8.1,9.5]
#[r1, r2, r3, r4, r5, r6, r7]
cmap_plasma = plt.get_cmap("plasma")
for i in range(len(r_centers)):
    color = cmap_plasma(i / len(r_centers))
    axes[0].axvspan(r_centers[i]-0.25, r_centers[i]+0.25, 0, 2, alpha=0.3, color=color)
    axes[0].vlines(r_centers[i], -0.1, 2.3, color=color, alpha=0.5,
                   label=f'{r_centers[i]:.2f} Å (±0.25 Å)', linewidth=2)

cmap = plt.get_cmap("jet")
for i in range(1, 1000, 10):
    color = cmap(i / (1000 - 1))
    axes[0].plot(r*10, Na_vhs_1000[:, i], color=color)
    
axes[0].set_xlabel("r (Å)")
axes[0].set_ylabel(r"$4\pi r^2\, G_s(r,t)$")
#axes[0].set_xlim(-0.05, 9)
axes[0].set_ylim(-0.1, 2.3)

# Bottom plot: P(t) for different radial shells
for i in range(len(r_centers)):
    color = cmap_plasma(i / len(r_centers))
    
    # Find indices for radial window
    rc_idx_1 = np.searchsorted(r, 0.1*r_centers[i] - 0.025)
    rc_idx_2 = np.searchsorted(r, 0.1*r_centers[i] + 0.025)
    
    # Extract data within radial window
    r_clipped_1 = r[rc_idx_1:rc_idx_2]
    gs_clipped_1 = Na_vhs_1000[rc_idx_1:rc_idx_2, :]
    
    # Integrate over radius (axis=0) for each time point
    P_t_1 = simpson(gs_clipped_1, x=r_clipped_1, axis=0)
    P_t_1 /= P_t_1.max()
    
    axes[1].plot(t[1:], P_t_1[1:], '.-', alpha=0.6, 
                 label=f'{r_centers[i]:.2f} Å (±0.25 Å)', 
                 color=color)

axes[1].set_xlabel('Correlation time (ps)')
axes[1].set_ylabel('Normalized P(t)')
#axes[1].set_xlim(-20, 1020)
axes[0].legend()#frameon=False)

plt.tight_layout()
#plt.savefig('./hydration/VH_Na.svg')

In [ ]:
def stretched_exponential(t, tau, beta, A=1.0):
    """
    Kohlrausch-Williams-Watts (KWW) function
    tau: characteristic time
    beta: stretching exponent (0 < beta <= 1)
    A: amplitude (fixed to 1.0 for normalized data)
    """
    return A * np.exp(-(t / tau) ** beta)

def exponential_with_bg(t, tau, BG):
    """
    Simple exponential decay with background
    tau: characteristic decay time
    BG: background/plateau level
    """
    return np.exp(-t / tau) + BG

i=0
rc_idx_1 = np.searchsorted(r, 0.1*r_centers[i] - 0.025)
rc_idx_2 = np.searchsorted(r, 0.1*r_centers[i] + 0.025)

# Extract data within radial window
r_clipped_1 = r[rc_idx_1:rc_idx_2]
gs_clipped_1 = Na_vhs_1000[rc_idx_1:rc_idx_2, :]

# Integrate over radius (axis=0) for each time point
P_t = simpson(gs_clipped_1, x=r_clipped_1, axis=0)


popt_1, pcov_1 = curve_fit(stretched_exponential, t[1:], P_t[1:]/P_t.max(),
                       p0=[50, 0.7, 1.0], 
                       bounds=([0, 0, 0.9], [np.inf, 1.0, 2]),
                       maxfev=10000)

popt_2, pcov_2 = curve_fit(exponential_with_bg, t[1:], P_t[1:]/P_t.max(), 
                           p0=[50, 0], 
                           bounds=([0, 0], [np.inf, 1.0]),
                           maxfev=10000)
tau_extracted_2, BG_extracted_2 = popt

plt.plot(t[1:], P_t[1:]/P_t.max(),'k.', alpha=0.4, label='Data')

plt.plot(t[1:], stretched_exponential(t[1:], *popt_1), 'r-', linewidth=2, 
           label=f'KWW Fit (τ={tau_extracted:.3f}, β={beta_extracted:.3f})')

plt.plot(t[1:], exponential_with_bg(t[1:], *popt_2), 'g-', linewidth=2, 
           label=f'KWW Fit (τ={tau_extracted_2:.3f}, β={BG_extracted_2:.3f})')

print(f"Extracted Residence Time (τ): {tau_extracted:.4f} ± {perr[0]:.4f}")
print(f"Stretching Exponent (β): {beta_extracted:.4f} ± {perr[1]:.4f}")
print(f"Amplitude (A): {A_extracted:.4f} ± {perr[2]:.4f}")
    
plt.legend()

In [ ]:
def stretched_exponential(t, tau, beta, A=1.0):
    """
    Kohlrausch-Williams-Watts (KWW) function
    tau: characteristic time
    beta: stretching exponent (0 < beta <= 1)
    A: amplitude (fixed to 1.0 for normalized data)
    """
    return A * np.exp(-(t / tau) ** beta)

def exponential_with_bg(t, tau, BG):
    """
    Simple exponential decay with background
    tau: characteristic decay time
    BG: background/plateau level
    """
    return np.exp(-t / tau) + BG

i = 0
rc_idx_1 = np.searchsorted(r, 0.1*r_centers[i] - 0.025)
rc_idx_2 = np.searchsorted(r, 0.1*r_centers[i] + 0.025)

# Extract data within radial window
r_clipped_1 = r[rc_idx_1:rc_idx_2]
gs_clipped_1 = Na_vhs_1000[rc_idx_1:rc_idx_2, :]

# Integrate over radius (axis=0) for each time point
P_t = simpson(gs_clipped_1, x=r_clipped_1, axis=0)
P_t_normalized = P_t / P_t.max()

# Fit 1: Stretched exponential (KWW)
popt_1, pcov_1 = curve_fit(stretched_exponential, t[1:], P_t_normalized[1:],
                           p0=[50, 0.7, 1.0], 
                           bounds=([0, 0, 0.9], [np.inf, 1.0, 2]),
                           maxfev=10000)
tau_extracted_1, beta_extracted_1, A_extracted_1 = popt_1
perr_1 = np.sqrt(np.diag(pcov_1))

# Fit 2: Exponential with background
popt_2, pcov_2 = curve_fit(exponential_with_bg, t[1:], P_t_normalized[1:], 
                           p0=[50, 0], 
                           bounds=([0, 0], [np.inf, 1.0]),
                           maxfev=10000)
tau_extracted_2, BG_extracted_2 = popt_2
perr_2 = np.sqrt(np.diag(pcov_2))

# Calculate R-squared for both fits
residuals_1 = P_t_normalized[1:] - stretched_exponential(t[1:], *popt_1)
ss_res_1 = np.sum(residuals_1**2)
ss_tot = np.sum((P_t_normalized[1:] - np.mean(P_t_normalized[1:]))**2)
r_squared_1 = 1 - (ss_res_1 / ss_tot)

residuals_2 = P_t_normalized[1:] - exponential_with_bg(t[1:], *popt_2)
ss_res_2 = np.sum(residuals_2**2)
r_squared_2 = 1 - (ss_res_2 / ss_tot)

# Print results
print("=" * 60)
print("Stretched Exponential (KWW) Fit:")
print(f"  Residence Time (τ): {tau_extracted_1:.4f} ± {perr_1[0]:.4f}")
print(f"  Stretching Exponent (β): {beta_extracted_1:.4f} ± {perr_1[1]:.4f}")
print(f"  Amplitude (A): {A_extracted_1:.4f} ± {perr_1[2]:.4f}")
print(f"  R-squared: {r_squared_1:.4f}")
print()
print("Exponential + Background Fit:")
print(f"  Residence Time (τ): {tau_extracted_2:.4f} ± {perr_2[0]:.4f}")
print(f"  Background (BG): {BG_extracted_2:.4f} ± {perr_2[1]:.4f}")
print(f"  R-squared: {r_squared_2:.4f}")
print("=" * 60)

# Plot
plt.figure(figsize=(singlecol*1.5,singlecol*1.5))
plt.plot(t[1:], P_t_normalized[1:], 'k.', alpha=0.4, label='Data')
plt.plot(t[1:], stretched_exponential(t[1:], *popt_1), 'r-', linewidth=2, 
         label=f'KWW: τ={tau_extracted_1:.2f}, β={beta_extracted_1:.2f} (R²={r_squared_1:.3f})')
plt.plot(t[1:], exponential_with_bg(t[1:], *popt_2), 'g-', linewidth=2, 
         label=f'Exp+BG: τ={tau_extracted_2:.2f}, BG={BG_extracted_2:.2f} (R²={r_squared_2:.3f})')

plt.xlabel('Correlation time (ps)')
plt.ylabel('Normalized P(t)')
#plt.title(f'Fitting Comparison at r = {r_centers[i]:.2f} Å')
plt.legend(frameon=False)
#plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('./hydration/fit.svg')

# Semi-log plot to better see differences
plt.figure(figsize=(singlecol*1, singlecol*1))
plt.semilogy(t[1:], P_t_normalized[1:], 'k.', alpha=0.4, label='Data')
plt.semilogy(t[1:], stretched_exponential(t[1:], *popt_1), 'r-', linewidth=2, 
             label=f'KWW: τ={tau_extracted_1:.2f}, β={beta_extracted_1:.2f}')
plt.semilogy(t[1:], exponential_with_bg(t[1:], *popt_2), 'g-', linewidth=2, 
             label=f'Exp+BG: τ={tau_extracted_2:.2f}, BG={BG_extracted_2:.2f}')

plt.xlabel('Time (ps)')
plt.ylabel('log P(t)')
plt.title(f'Fitting Comparison (Log Scale) at r = {r_centers[i]:.2f} Å')
plt.legend(frameon=False)
plt.grid(True, which="both", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.optimize import curve_fit
rc1 = 0.1
rc2 = 0.25
rc3 = 0.43

# 1. Identify the spatial cutoff index
rc_idx_1 = np.searchsorted(r, rc1)
rc_idx_2 = np.searchsorted(r, rc2)
rc_idx_3 = np.searchsorted(r, rc3)
r_clipped = r[rc_idx_2:rc_idx_3]

# Select the spatial rows up to rc_idx for all time columns
gs_clipped = Na_vhs_1000[rc_idx_2:rc_idx_3, :] 

# 2. Calculate P(t)
# We integrate over the r-axis (axis=0)
integrand = gs_clipped
P_t = simpson(integrand, x=r_clipped, axis=0)

# 3. Normalization
# P(t) should represent the fraction remaining in the cage
P_t /= P_t[1]

# 4. Fitting for Tau (Stretched exponential / KWW)
def stretched_exponential(t, tau, beta, A=1.0):
    """
    Kohlrausch-Williams-Watts (KWW) function
    tau: characteristic time
    beta: stretching exponent (0 < beta <= 1)
    A: amplitude (fixed to 1.0 for normalized data)
    """
    return A * np.exp(-(t / tau) ** beta)

# Fit only the meaningful decay (e.g., until 10% remains)
mask = P_t > 0.1

try:
    # Initial guess: [tau, beta]
    # tau: estimate from time when P drops to ~1/e
    # beta: typically between 0.5-1.0 for glassy systems
    initial_tau = t[len(t)//10] if len(t)//10 < len(t) else t[len(t)//2]
    
    popt, pcov = curve_fit(stretched_exponential, t[mask], P_t[mask], 
                           p0=[initial_tau, 0.7, 1.0], 
                           bounds=([0, 0, 0.9], [np.inf, 1.0, 2]),
                           maxfev=10000)
    
    tau_extracted, beta_extracted, A_extracted = popt
    perr = np.sqrt(np.diag(pcov))
    
    print(f"Extracted Residence Time (τ): {tau_extracted:.4f} ± {perr[0]:.4f}")
    print(f"Stretching Exponent (β): {beta_extracted:.4f} ± {perr[1]:.4f}")
    print(f"Amplitude (A): {A_extracted:.4f} ± {perr[2]:.4f}")
    
    # Calculate R-squared
    residuals = P_t[mask] - stretched_exponential(t[mask], *popt)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((P_t[mask] - np.mean(P_t[mask]))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"R-squared: {r_squared:.4f}")
    
    # --- Visualization ---
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Linear Plot
    ax[0].plot(t, P_t, 'k.', alpha=0.4, label='Data')
    ax[0].plot(t, stretched_exponential(t, *popt), 'r-', linewidth=2, 
               label=f'KWW Fit (τ={tau_extracted:.3f}, β={beta_extracted:.3f})')
    ax[0].set_xlabel('Time (t)')
    ax[0].set_ylabel('Survival Probability P(t)')
    ax[0].legend()
    ax[0].set_title('Residence Probability (Linear)')
    ax[0].grid(True, alpha=0.3)
    
    # Semi-log Plot
    ax[1].semilogy(t, P_t, 'k.', alpha=0.4, label='Data')
    ax[1].semilogy(t, stretched_exponential(t, *popt), 'r-', linewidth=2,
                   label=f'KWW Fit')
    ax[1].set_xlabel('Time (t)')
    ax[1].set_ylabel('log P(t)')
    ax[1].set_title('Residence Probability (Log)')
    ax[1].legend()
    ax[1].grid(True, which="both", alpha=0.2)
    
    plt.tight_layout()
    plt.show()
    
    # Additional: Plot on log-log scale to check stretching
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.loglog(t, P_t, 'k.', alpha=0.4, label='Data')
    ax.loglog(t, stretched_exponential(t, *popt), 'r-', linewidth=2,
              label=f'KWW Fit (τ={tau_extracted:.3f}, β={beta_extracted:.3f})')
    ax.set_xlabel('Time (t)')
    ax.set_ylabel('P(t)')
    ax.set_title('Residence Probability (Log-Log)')
    ax.legend()
    ax.grid(True, which="both", alpha=0.2)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Fitting failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
initial_BG

In [ ]:
rc1 = 0.1
rc2 = 0.25
rc3 = 0.43

# 1. Identify the spatial cutoff index
rc_idx_1 = np.searchsorted(r, rc1)
rc_idx_2 = np.searchsorted(r, rc2)
rc_idx_3 = np.searchsorted(r, rc3)
r_clipped = r[:rc_idx_1]

# Select the spatial rows up to rc_idx for all time columns
gs_clipped = Na_vhs_1000[:rc_idx_1, :] 

# 2. Calculate P(t)
# We integrate over the r-axis (axis=0)
integrand = gs_clipped
P_t = simpson(integrand, x=r_clipped, axis=0)

# 3. Normalization
# P(t) should represent the fraction remaining in the cage
P_t /= P_t.max()

# 4. Fitting for Tau (Exponential decay with background)
def exponential_with_bg(t, tau, BG):
    """
    Simple exponential decay with background
    tau: characteristic decay time
    BG: background/plateau level
    """
    return np.exp(-t / tau) + BG

# Fit only the meaningful decay (e.g., until 10% remains)
mask = P_t > 0.1

try:
    # Initial guess: [tau, BG]
    # tau: estimate from time when P drops significantly
    # BG: estimate from the plateau at long times
    initial_tau = t[len(t)//10] if len(t)//10 < len(t) else t[len(t)//2]
    initial_BG = P_t[-1] if len(P_t) > 0 else 0.1
    
    popt, pcov = curve_fit(exponential_with_bg, t[mask], P_t[mask], 
                           p0=[initial_tau, 0], 
                           bounds=([0, 0], [np.inf, 1.0]),
                           maxfev=10000)
    
    tau_extracted, BG_extracted = popt
    perr = np.sqrt(np.diag(pcov))
    
    print(f"Extracted Residence Time (τ): {tau_extracted:.4f} ± {perr[0]:.4f}")
    print(f"Background (BG): {BG_extracted:.4f} ± {perr[1]:.4f}")
    
    # Calculate R-squared
    residuals = P_t[mask] - exponential_with_bg(t[mask], *popt)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((P_t[mask] - np.mean(P_t[mask]))**2)
    r_squared = 1 - (ss_res / ss_tot)
    print(f"R-squared: {r_squared:.4f}")
    
    # Calculate effective decay amplitude
    amplitude = 1.0 - BG_extracted
    print(f"Decay amplitude (1 - BG): {amplitude:.4f}")
    
    # --- Visualization ---
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    
    # Linear Plot
    ax[0].plot(t, P_t, 'k.', alpha=0.4, label='Data')
    ax[0].plot(t, exponential_with_bg(t, *popt), 'r-', linewidth=2, 
               label=f'Fit (τ={tau_extracted:.3f}, BG={BG_extracted:.3f})')
    ax[0].axhline(y=BG_extracted, color='b', linestyle='--', alpha=0.5, 
                  label=f'Background={BG_extracted:.3f}')
    ax[0].set_xlabel('Time (t)')
    ax[0].set_ylabel('Survival Probability P(t)')
    ax[0].legend()
    ax[0].set_title('Residence Probability (Linear)')
    ax[0].grid(True, alpha=0.3)
    
    # Semi-log Plot
    ax[1].semilogy(t, P_t, 'k.', alpha=0.4, label='Data')
    ax[1].semilogy(t, exponential_with_bg(t, *popt), 'r-', linewidth=2,
                   label=f'Fit (τ={tau_extracted:.3f})')
    ax[1].axhline(y=BG_extracted, color='b', linestyle='--', alpha=0.5,
                  label=f'Background')
    ax[1].set_xlabel('Time (t)')
    ax[1].set_ylabel('log P(t)')
    ax[1].set_title('Residence Probability (Log)')
    ax[1].legend()
    ax[1].grid(True, which="both", alpha=0.2)
    
    plt.tight_layout()
    plt.show()
    
    # Additional: Plot residuals
    fig, ax = plt.subplots(figsize=(8, 4))
    residuals_all = P_t - exponential_with_bg(t, *popt)
    ax.plot(t, residuals_all, 'k.', alpha=0.4)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('Time (t)')
    ax.set_ylabel('Residuals')
    ax.set_title('Fit Residuals')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Fitting failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
pcov

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

# Your existing code...
datasets = [
    (total_vhs_1000, "total"),
    (Na_vhs_1000,  "Na"),
    (O_vhs_1000,  "O"),
    (H_vhs_1000, "H"),
    (Fe_vhs_1000,  "Fe"),
    (N_vhs_1000, "N"),
]

indices = range(1,len(t),5)
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(3, 2, figsize=(singlecol*2, singlecol*3), sharex=True)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color)
    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel("g(r, t)")
    ax.set_xlim(0, 5)
    #ax.set_yscale('log')
    ax.grid(True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (Na_vhs, "Na"),
    (O_vhs,  "O"),
    (H_vhs,  "H"),
    (Fe_vhs, "Fe")
]

indices = range(1, 100)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("viridis")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(r*10, data[:, i], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("r (Å)")
    ax.set_ylabel("g(r, t)")
    ax.set_xlim(-0.1,3)
    ax.grid(True)


plt.tight_layout()
plt.show()


### QENS

In [ ]:
#!/usr/bin/env python
import os

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
    VECLIB_MAXIMUM_THREADS="1",
    NUMEXPR_NUM_THREADS="1",
)

########################################################
# This is an automatically generated MDANSE run script #
########################################################

from MDANSE.Framework.Jobs.IJob import IJob

########################################################
# Job parameters                                       #
########################################################


parameters = {
    "atom_transmutation": "{}",  # atom_transmutation
    "frames": [35, 2035, 1, 100],  # frames
    "grouping_level": "atom",  # grouping_level
    "instrument_resolution": ("ideal", {}),  # instrument_resolution
    "output_files": (
        "./hydration/QENS/400K_9_DISF_1000.mda",
        ["MDAFormat"],
        "INFO",
    ),  # output_files
    "projection": ("NullProjector", []),  # project coordinates
    "q_vectors": (
        "SphericalQVectors",
        {
            "seed": 0,
            "shells": [5.0, 20.0, 0.5],
            "n_vectors": 500,
            "width": 0.0,
        },
    ),  # q_vectors
    "running_mode": ("multicore", 60),  # running_mode
    "trajectory": "./hydration/QENS/400K_9_convert_trj.mdt",  # trajectory
    "weights": "b_incoherent",  # weights
}

########################################################
# Setup and run the analysis                           #
########################################################

if __name__ == "__main__":
    dynamicincoherentstructurefactor = IJob.create("DynamicIncoherentStructureFactor")
    # Progress bars only available if tqdm available.
    # Install with `cli` optional dependency.
    dynamicincoherentstructurefactor.run(parameters, status=True, prog_bar=True)

In [ ]:
with h5py.File("./hydration/QENS/400K_9/400K_9_DISF_1000.mda", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
import numpy as np

# Incoherent scattering lengths (in fm) from NIST data
# Based on natural isotopic abundances
INCOHERENT_SCATTERING_LENGTHS = {
    'C': 0.0892,       # Carbon (predominantly 12C)
    'Mn': 1.79,     # Manganese
    'Fe': 1.784,      # Iron (predominantly 56Fe)
    'N': 1.994,       # Nitrogen (predominantly 14N)
    'Na': 3.59,     # Sodium
    'O': 0.0798,       # Oxygen (predominantly 16O)
    'H': 25.252#4.04#25.272     # Hydrogen (predominantly 1H)
}

def calculate_incoherent_cross_sections(scattering_lengths):
    """
    Calculate incoherent scattering cross sections from scattering lengths.
    σ_inc = 4π * b_inc²
    
    Parameters:
    -----------
    scattering_lengths : dict
        Dictionary of element symbols to incoherent scattering lengths (fm)
    
    Returns:
    --------
    dict : Incoherent cross sections in barns (1 barn = 100 fm²)
    """
    cross_sections = {}
    for element, b_inc in scattering_lengths.items():
        # σ_inc = 4π * b_inc² (in fm²), then convert to barns
        sigma_inc = 4 * np.pi * b_inc**2 / 100  # Convert fm² to barns
        cross_sections[element] = sigma_inc
    return cross_sections

def calculate_single_atom_weights(element_counts, scattering_lengths):
    """
    Calculate single atom-type weights for DISF analysis.
    
    For single atom-type weighting:
    w_α = (N_α * b_inc,α²) / Σ_β(N_β * b_inc,β²)
    
    where:
    - N_α is the number of atoms of type α
    - b_inc,α is the incoherent scattering length of type α
    
    Parameters:
    -----------
    element_counts : dict
        Dictionary of element symbols to number of atoms
        Example: {'H': 100, 'O': 50, 'C': 30}
    
    scattering_lengths : dict
        Dictionary of element symbols to incoherent scattering lengths (fm)
    
    Returns:
    --------
    dict : Normalized weights for each element type
    """
    # Calculate weighted contributions
    weighted_contributions = {}
    total_weighted = 0.0
    
    for element, count in element_counts.items():
        if element not in scattering_lengths:
            raise ValueError(f"Element '{element}' not found in scattering length database")
        
        b_inc = scattering_lengths[element]
        contribution = count * b_inc**2
        weighted_contributions[element] = contribution
        total_weighted += contribution
    
    # Normalize to get weights
    weights = {}
    for element, contribution in weighted_contributions.items():
        if total_weighted > 0:
            weights[element] = contribution / total_weighted
        else:
            weights[element] = 0.0
    
    return weights

def apply_weights_to_disf(disf_data, weights, element_order):
    """
    Apply single atom-type weights to DISF data.
    
    Parameters:
    -----------
    disf_data : numpy.ndarray
        DISF data with shape (n_q_points, n_atom_types) or (n_time_points, n_q_points, n_atom_types)
    
    weights : dict
        Dictionary of element symbols to weights
    
    element_order : list
        List of element symbols in the order they appear in disf_data
        Example: ['C', 'Mn', 'Fe', 'N', 'Na', 'O', 'H']
    
    Returns:
    --------
    numpy.ndarray : Weighted DISF data
    """
    # Create weight array in the correct order
    weight_array = np.array([weights[element] for element in element_order])
    
    # Apply weights (broadcasting will handle different dimensions)
    weighted_disf = disf_data * weight_array
    
    return weighted_disf

# Example usage
if __name__ == "__main__":
    # Example: Define your system composition
    # (number of atoms of each type in your system)
    element_counts = {
        'C': 5184,
        'Mn': 864,
        'Fe': 864,
        'N': 5184,
        'Na': 864,#1512,
        'O': 1728, #864,#1728,
        'H': 3456
    }
    
    # Calculate incoherent cross sections
    print("Incoherent Scattering Cross Sections:")
    print("=" * 50)
    cross_sections = calculate_incoherent_cross_sections(INCOHERENT_SCATTERING_LENGTHS)
    for element in ['C', 'Mn', 'Fe', 'N', 'Na', 'O', 'H']:
        print(f"{element:3s}: {cross_sections[element]:10.4f} barns")
    
    print("\n" + "=" * 50)
    print("Single Atom-Type Weights:")
    print("=" * 50)
    
    # Calculate weights
    weights = calculate_single_atom_weights(element_counts, INCOHERENT_SCATTERING_LENGTHS)
    
    # Display weights
    for element in ['C', 'Mn', 'Fe', 'N', 'Na', 'O', 'H']:
        print(f"{element:3s}: {weights[element]:10.6f} ({weights[element]*100:6.3f}%)")
    
    print(f"\nSum of weights: {sum(weights.values()):.6f}")
    
    # Example: Apply weights to DISF data
    print("\n" + "=" * 50)
    print("Example: Applying weights to DISF data")
    print("=" * 50)
    
    # Simulate DISF data: (n_q_points=10, n_atom_types=7)
    element_order = ['C', 'Mn', 'Fe', 'N', 'Na', 'O', 'H']
    dummy_disf = np.random.rand(10, 7)  # 10 q-points, 7 atom types
    
    # Apply weights
    weighted_disf = apply_weights_to_disf(dummy_disf, weights, element_order)
    
    print(f"Original DISF shape: {dummy_disf.shape}")
    print(f"Weighted DISF shape: {weighted_disf.shape}")
    print(f"\nFirst q-point, original: {dummy_disf[0]}")
    print(f"First q-point, weighted: {weighted_disf[0]}")
    
    # You can sum over atom types to get total weighted DISF
    total_weighted_disf = np.sum(weighted_disf, axis=1)
    print(f"\nTotal weighted DISF (summed over atom types): {total_weighted_disf}")

In [ ]:
file_path = "./hydration/QENS/400K_9/400K_9_DISF_1000.mda"
#file_path = "./hydration/QENS/400K_vac/400K_1_DISF_1000.mda"

with h5py.File(file_path, "r") as f:
    total_disf = f["disf/s(q,f)/total"][:]
    C_disf = f["disf/s(q,f)/C"][:]#*weights['C']
    Mn_disf = f["disf/s(q,f)/Mn"][:]#*weights['Mn']
    Na_disf = f["disf/s(q,f)/Na"][:]*0.008
    N_disf = f["disf/s(q,f)/N"][:]#*weights['N']
    O_disf = f["disf/s(q,f)/O"][:]#*weights['O']
    H_disf = f["disf/s(q,f)/H"][:]*0.97
    Fe_disf = f["disf/s(q,f)/Fe"][:]#*weights['Fe']
    
    total_F = f["disf/f(q,t)/C"][:]
    Na_F = f["disf/f(q,t)/Na"][:]
    O_F = f["disf/f(q,t)/O"][:]
    H_F = f["disf/f(q,t)/H"][:]
    Fe_F = f["disf/f(q,t)/Fe"][:]
    Mn_F = f["disf/f(q,t)/Mn"][:]
    C_F = f["disf/f(q,t)/C"][:]
    N_F = f["disf/f(q,t)/N"][:]
    
    omega = f["disf/axes/omega"][:]
    q = f["disf/axes/q"][:]
    time = f["disf/axes/time"][:]

    weight = f["metadata/inputs/weights"]
omega = np.linspace(-0.51675,0.51675,2499)

In [ ]:
#vac_1
#Na: 0.007272388091947219
#H: 0.9795321603396179

400K_9
008649299524081392
9797229325774742

400K_8
004977985359196792
9833511851003275

In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_12_DISF.mda"

with h5py.File(file_path, "r") as f:
    total_disf = f["disf/s(q,f)/total"][:]*1
    C_disf = f["disf/s(q,f)/C"][:]#*weights['C']
    Mn_disf = f["disf/s(q,f)/Mn"][:]#*weights['Mn']
    Na_disf = f["disf/s(q,f)/Na"][:]*0.004888914600695902 #0.004977985359196792 #0.008649299524081392
    N_disf = f["disf/s(q,f)/N"][:]#*weights['N']
    O_disf = f["disf/s(q,f)/O"][:]#*weights['O']
    H_disf = f["disf/s(q,f)/H"][:]*0.9842518036306152
    Fe_disf = f["disf/s(q,f)/Fe"][:]#*weights['Fe']
    
    total_F = f["disf/f(q,t)/total"][:]
    Na_F = f["disf/f(q,t)/Na"][:]
    O_F = f["disf/f(q,t)/O"][:]
    H_F = f["disf/f(q,t)/H"][:]
    Fe_F = f["disf/f(q,t)/Fe"][:]
    Mn_F = f["disf/f(q,t)/Mn"][:]
    C_F = f["disf/f(q,t)/C"][:]
    N_F = f["disf/f(q,t)/N"][:]

    q = f["disf/axes/q"][:]
    time = f["disf/axes/time"][:]

    weight = f["metadata/inputs/weights"]

In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_12_DISF_select.mda"

with h5py.File(file_path, "r") as f:
    total_disf = f["disf/s(q,f)/total"][:]


In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_12_DISF_select.mda"

with h5py.File(file_path, "r") as f:
    total_select = f["disf/s(q,f)/total"][:]
    H_select = f["disf/s(q,f)/H"][:]#*0.9795321603396179# 0.9833511851003275 #0.9797229325774742
    
    total_F = f["disf/f(q,t)/C"][:]
    H_F = f["disf/f(q,t)/H"][:]
    q = f["disf/axes/q"][:]
    time = f["disf/axes/time"][:]

    weight = f["metadata/inputs/weights"]

In [ ]:
516.75*2/1500

In [ ]:
total_disf.shape

In [ ]:
total_disf[0,1]

In [ ]:
total_disf_error = total_disf*

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})

#omega = np.linspace(-0.51675,0.51675,1799)
omega = np.linspace(-0.51675,0.51675,2499)
i=25
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_disf, "Total"),
    (Na_disf,    "Na"),
    (H_disf,    "H")
]

indices = range(25, 190,5)   # frames to plot
n = len(indices)
cmap = plt.get_cmap("jet")
fig, axes = plt.subplots(1, 3, figsize=(singlecol*3, singlecol*1), dpi=100)
axes = axes.ravel()

def remove_peak_outlier(S_raw):
    """Remove the single highest peak by linear interpolation."""
    S_clean = S_raw.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 1, 0)
    hi = min(i0 + 2, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    return S_clean

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        S_clean = remove_peak_outlier(data[i, :])
        ax.plot(omega, S_clean, color=color, label=f"t={i}")
    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w) (a.u.)")
    ax.set_xlim(-0.03, 0.03)
    ax.set_yscale("log")
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib widget
def remove_peak_outlier(S_raw):
    """Remove the single highest peak by linear interpolation."""
    S_clean = S_raw.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    return S_clean
#omega = np.linspace(-0.51675,0.51675,1499)
i=20
plt.figure( figsize=(singlecol*2, singlecol*2))
total_clean = remove_peak_outlier(total_disf[i, :])
Na_clean = remove_peak_outlier(Na_disf[i, :])
H_clean = remove_peak_outlier(H_disf[i, :])
plt.plot(omega, total_clean, label='total', lw=3)
plt.plot(omega, Na_clean*20, label='Na')
plt.plot(omega, H_clean, label='H')
plt.xlim(-0.2,0.2)
plt.plot()
plt.legend()
plt.yscale('log')
plt.xlabel("Energy transfer (meV)")
plt.ylabel('S(q,w)')

In [ ]:
indices = range(25, 190)   # frames to plot
n = len(indices)
cmap = plt.get_cmap("jet")
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

In [ ]:
S_clean.shape

In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})

import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_disf, "Total"),
    (Na_disf,    "Na"),
    (O_disf,     "O"),
    (H_disf,     "H")
]

indices = range(25, 190,5)   # frames to plot
n = len(indices)
cmap = plt.get_cmap("jet")
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        #S_clean = remove_peak_outlier(data[i, :])
        ax.plot(omega, data[i, :], color=color, label=f"t={i}")
    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w)")
    ax.set_xlim(-0.03, 0.03)
    ax.set_yscale("log")
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib widget
plt.figure()
i=20
S = H_disf[i]
S_raw = S.copy()
i0 = np.argmax(S_raw)
lo = max(i0 - 0, 0)
hi = min(i0 + 1, len(S_raw))
good_idx = np.r_[0:lo, hi:len(S_raw)]
S_clean = S_raw.copy()
S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
H = S_clean
plt.plot(omega, H_disf[i], label='H')

S = Na_disf[i]
S_raw = S.copy()
i0 = np.argmax(S_raw)
lo = max(i0 - 0, 0)
hi = min(i0 + 1, len(S_raw))
good_idx = np.r_[0:lo, hi:len(S_raw)]
S_clean = S_raw.copy()
S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
total = S_clean
plt.plot(omega, Na_disf[i], label='Na')

S = total_disf[i]
S_raw = S.copy()
i0 = np.argmax(S_raw)
lo = max(i0 - 0, 0)
hi = min(i0 + 1, len(S_raw))
good_idx = np.r_[0:lo, hi:len(S_raw)]
S_clean = S_raw.copy()
S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
total = S_clean
plt.plot(omega, total_disf[i], label='total')

plt.xlim(-0.03, 0.03)
plt.yscale("log")
plt.legend()

In [ ]:
q[90]

In [ ]:
i = 90

def clean_peak(S_raw):
    S_clean = S_raw.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    return S_clean

H     = clean_peak(H_disf[i])
total = clean_peak(total_disf[i])

plt.rcParams.update({'font.size': 12, 'lines.linewidth': 2})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(omega, total,       label='total', lw=3)#, color=color1)
ax1.plot(omega, Na_disf[i],  label='Na')#,          color=color2)
ax1.plot(omega, H,           label='H')#,            color=color3)
ax1.set_ylabel('S(Q, ω)')
ax1.set_yscale('log')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-0.02,0.02)

ax2.plot(omega, total - H, color=color1)
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xlabel('Energy transfer (meV)')
ax2.set_ylabel('total − H')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12})
plt.rcParams.update({'lines.linewidth':2})
plt.figure( figsize=(singlecol*2, singlecol*2))

cmap1 = plt.get_cmap("Blues")
cmap2 = plt.get_cmap("Greens")
cmap3 = plt.get_cmap("Oranges")

for i in range(25,190):
    color1 = cmap1(i / 190)
    color2 = cmap2(i / 190)
    color3 = cmap3(i / 190)
    
    S = H_disf[i]
    S_raw = S.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean = S_raw.copy()
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    H = S_clean

    S = total_disf[i]
    S_raw = S.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean = S_raw.copy()
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    total = S_clean
    
    plt.plot(omega, total-H, label='total', lw=3, color=color1)
    #plt.plot(omega, Na_disf[i], label='Na', color=color2)
    plt.plot(omega, H, label='H', color=color3)
    plt.xlim(-0.02,0.02)
    plt.ylim(1e-2,5)
#plt.legend(fontsize=16, frameon=False)
plt.yscale('log')
plt.xlabel("Energy transfer (meV)")
plt.ylabel('S(q,w)')

In [ ]:
%matplotlib inline
plt.rcParams.update({'font.size': 12})
plt.rcParams.update({'lines.linewidth':2})
plt.figure( figsize=(singlecol*2, singlecol*2))
plt.plot(omega, total, label='total', lw=3)
#plt.plot(omega, Na_disf[i], label='Na')
#plt.plot(omega, H, label='H')
plt.xlim(-0.02,0.02)
plt.ylim(1e-2,5)
plt.plot()
plt.legend(fontsize=16, frameon=False)
plt.yscale('log')
plt.xlabel("Energy transfer (meV)")
plt.ylabel('S(q,w)')

In [ ]:
plt.figure( figsize=(singlecol*3, singlecol*3))
i=190
plt.plot(omega, total_disf[i,:], label='total', lw=3)
plt.plot(omega, Na_disf[i,:], label='Na')
#plt.plot(omega, C_disf[i,:], label='C')
plt.plot(omega, N_disf[i,:], label='N')
#plt.plot(omega, Mn_disf[i,:], label='Mn')
#plt.plot(omega, Fe_disf[i,:], label='Fe')
#plt.plot(omega, O_disf[i,:], label='O')
plt.plot(omega, H_disf[i,:], label='H')
I_sum = (C_disf[0,:]+ Mn_disf[0,:]+Fe_disf[0,:]+N_disf[0,:]+Na_disf[0,:]+O_disf[0,:]+H_disf[0,:])
plt.plot()
plt.legend()
plt.yscale('log')

In [ ]:
plt.figure( figsize=(singlecol*3, singlecol*3))
i=25
plt.plot(omega, total_disf[i,:], label='total', lw=3)
plt.plot(omega, Na_disf[i,:], label='Na')
#plt.plot(omega, C_disf[i,:], label='C')
plt.plot(omega, N_disf[i,:], label='N')
#plt.plot(omega, Mn_disf[i,:], label='Mn')
#plt.plot(omega, Fe_disf[i,:], label='Fe')
#plt.plot(omega, O_disf[i,:], label='O')
plt.plot(omega, H_disf[i,:], label='H')
I_sum = (C_disf[0,:]+ Mn_disf[0,:]+Fe_disf[0,:]+N_disf[0,:]+Na_disf[0,:]+O_disf[0,:]+H_disf[0,:])
plt.plot()
plt.legend()
plt.yscale('log')

In [ ]:
q.shape[0]

In [ ]:
plt.figure()
for i in range(q.shape[0]):
    color = cmap(i / q.shape[0])
    plt.plot(omega, Fe_disf[i,:]*i*100, color=color)
plt.yscale('log')

In [ ]:
plt.figure( figsize=(singlecol*2, singlecol*2))
plt.plot(total_disf[0,:], label='total', lw=3)
plt.plot(Na_disf[0,:], label='Na')
plt.plot(C_disf[0,:], label='C')
plt.plot(N_disf[0,:], label='N')
plt.plot(Mn_disf[0,:], label='Mn')
plt.plot(Fe_disf[0,:], label='Fe')
plt.plot(O_disf[0,:], label='O')
plt.plot(H_disf[0,:], label='H')
plt.plot()
plt.legend()
plt.yscale('log')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_disf, "Total"),
    (Na_disf,  "Na"),
    (O_disf,  "O"),
    (H_disf, "H")
]

indices = range(25,190)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(omega, data[i, :], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w)")
    ax.set_xlim(-0.01,0.01)
    ax.set_yscale("log")
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

i=10

# copy original data
x_orig = Na_disf[i, :].copy()
x_clean = x_orig.copy()

# locate noisy peak
i0 = x_orig.argmax()

# define noisy window
lo = max(i0 - 5, 0)
hi = min(i0 + 5 + 1, x_orig.size)

# indices used for interpolation
good_idx = np.r_[0:lo, hi:x_orig.size]

# interpolate over the noisy region
x_clean[lo:hi] = np.interp(
    np.arange(lo, hi),
    good_idx,
    x_orig[good_idx]
)

# store back
Na_disf[i, :] = x_clean

# ---- plot ----
plt.figure()
plt.plot(x_orig, label="Original", alpha=0.6)
plt.plot(x_clean, label="Noise-free (interpolated)", linewidth=2)
plt.axvspan(lo, hi-1, color="red", alpha=0.15, label="Removed region")

plt.xlabel("Index")
plt.ylabel("Na_disf")
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

i=10
# Store original data before cleaning
Na_disf_original = Na_disf[i,:].copy()

# Find the index of maximum
max_idx = Na_disf[i,:].argmax()

# Create mask for valid data (exclude noisy region)
valid_mask = np.ones(Na_disf.shape[1], dtype=bool)
valid_mask[max_idx - 3:max_idx + 4] = False

# Interpolate over the noisy region
x = np.arange(Na_disf.shape[1])
f = interp1d(x[valid_mask], Na_disf[0, valid_mask], kind='cubic')
Na_disf[0, max_idx - 3:max_idx + 4] = f(x[max_idx - 3:max_idx + 4])

# Plot comparison
plt.figure()#figsize=(12, 5))

# Subplot 1: Full signal comparison
plt.subplot(1, 2, 1)
plt.plot(Na_disf_original, label='Original (with noise)', alpha=0.7, linewidth=2)
plt.plot(Na_disf[i,:], label='Noise-free (cubic interp.)', linewidth=2)
plt.axvline(max_idx, color='r', linestyle='--', alpha=0.5, label='Max position')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Full Signal: Before and After Noise Removal')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 2: Zoomed view of the cleaned region
plt.subplot(1, 2, 2)
window = 100  # points to show on each side
x_range = range(max(0, max_idx - window), min(len(Na_disf[i,:]), max_idx + window + 1))
plt.plot(x_range, Na_disf_original[x_range], 'o-', label='Original', alpha=0.7, markersize=4)
plt.plot(x_range, Na_disf[0, x_range], 's-', label='Cleaned', alpha=0.7, markersize=4)
plt.axvspan(max_idx - 3, max_idx + 3, alpha=0.2, color='red', label='Noisy region')
plt.xlabel('Index')
plt.ylabel('Value')
plt.yscale('log')
plt.title('Zoomed View: Noise Removal Region')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
i=10
# Store original data before cleaning
Na_disf_original = Na_disf[i,:].copy()

# Find the index of maximum
max_idx = Na_disf[i,:].argmax()

Na_disf_original

In [ ]:
q[15]

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_disf, "Total"),
    (Na_disf,  "Na"),
    (O_disf,  "O"),
    (H_disf, "H")
]

indices = range(15,190)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(omega, data[i, :], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w)")
    ax.set_xlim(-0.03,0.03)
    ax.set_yscale("log")
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (Fe_disf,  "Fe"),
    (Mn_disf,  "Mn"),
    (C_disf,  "C"),
    (N_disf,  "N"),
]

indices = range(15,190)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(omega, data[i, :], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w)")
    #ax.set_xlim(-0.03,0.03)
    ax.set_yscale("log")
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import fftconvolve  # only needed if doing convolution with resolution

hbar_mev_ps = 0.6582119514

# ---- Define model components ----
def lorentzian(omega, A, Gamma):
    # A * (Gamma / (pi * (omega^2 + Gamma^2)))
    return A * (Gamma / (np.pi * (omega**2 + Gamma**2)))

def gaussian(omega, A, sigma):
    return A * np.exp(-0.5 * (omega / sigma)**2) / (sigma * np.sqrt(2*np.pi))

def linear_bg(omega, b0, b1):
    return b0 + b1 * omega

# ---- Full model: elastic Gaussian (resolution) + Lorentzian (quasielastic) + linear background ----
def model_with_elastic(omega, A_el, sigma_res, A_q, Gamma, b0, b1):
    return gaussian(omega, A_el, sigma_res) + lorentzian(omega, A_q, Gamma) + linear_bg(omega, b0, b1)

# ---- Alternative: convolve Lorentzian+inelastic with known resolution R(omega) ----
def convolved_model(omega, R, params):
    # R: resolution array sampled on same omega grid and normalized (area=1)
    # params: tuple (A_el, sigma_res, A_q, Gamma, b0, b1)
    A_el, sigma_res, A_q, Gamma, b0, b1 = params
    elastic = gaussian(omega, A_el, sigma_res)
    qens = lorentzian(omega, A_q, Gamma)
    # convolve qens + elastic with R (assuming R already normalized)
    tot = fftconvolve(elastic + qens, R, mode='same')
    tot += linear_bg(omega, b0, b1)
    return tot

# ---- Fitting wrapper for a single spectrum ----
def fit_spectrum(omega, S, p0=None, bounds=None, use_resolution=False, R=None):
    """
    Fit S(omega) with the chosen model.
    If use_resolution and R is provided, the code will fit by convolving model with R.
    Returns popt and pcov and derived values (Gamma, FWHM, tau).
    """
    # initial guesses
    if p0 is None:
        p0 = [S.max(), 0.05, 0.5*S.max(), 0.1, np.median(S[:5]), 0.0]  # [A_el, sigma_res, A_q, Gamma, b0, b1]
    if bounds is None:
        bounds = ([-np.inf, 1e-6, 0, 1e-6, -np.inf, -np.inf], np.inf)

    if use_resolution:
        if R is None:
            raise ValueError("Resolution array R must be provided for convolution fit.")
        # simple brute-force optimization wrapper: minimize residuals via curve_fit on parameters,
        # but we need a function that takes parameters and returns convolved model
        def fitfunc(omega, A_el, sigma_res, A_q, Gamma, b0, b1):
            params = (A_el, sigma_res, A_q, Gamma, b0, b1)
            return convolved_model(omega, R, params)
    else:
        fitfunc = model_with_elastic

    popt, pcov = curve_fit(fitfunc, omega, S, p0=p0, bounds=bounds, maxfev=10000)

    # Extract Gamma and compute FWHM and tau
    A_el, sigma_res, A_q, Gamma, b0, b1 = popt
    FWHM = 2.0 * Gamma
    tau_ps = hbar_mev_ps / Gamma  # ps, if Gamma in meV
    return popt, pcov, {'Gamma': Gamma, 'FWHM': FWHM, 'tau_ps': tau_ps}


In [ ]:
S = Na_disf[30]
w = omega
# ---- Noise removal around spike (Option 1) ----
S_raw = S.copy()

i0 = np.argmax(S_raw)
lo = max(i0 - 3, 0)
hi = min(i0 + 4, len(S_raw))

good_idx = np.r_[0:lo, hi:len(S_raw)]

S_clean = S_raw.copy()
S_clean[lo:hi] = np.interp(
    np.arange(lo, hi),
    good_idx,
    S_raw[good_idx]
)

# Use noise-free data from here on
S = S_clean
S = Na_disf[10]

p0_bg = [np.median(S_bg), 0.0]
popt_bg, pcov_bg = curve_fit(linear_bg, w_bg, S_bg, p0=p0_bg)
b0_fit, b1_fit = popt_bg
bg_fit_full = linear_bg(w, b0_fit, b1_fit)

S_corr = S - bg_fit_full

# ---- Lorentzian fit ----
fit_mask = np.abs(w) <= 0.6 * np.max(np.abs(w))

A0 = np.max(S_corr) * np.pi * 0.05
gamma0 = (w.max() - w.min()) * 0.05
w0_0 = 0.0

p0_lorentz = [A0, gamma0, w0_0]

bounds = (
    [1e-12, 1e-12, -np.inf],
    [np.inf, np.inf, np.inf]
)

popt_lor, pcov_lor = curve_fit(
    lorentz,
    w[fit_mask],
    S_corr[fit_mask],
    p0=p0_lorentz,
    bounds=bounds,
    maxfev=20000
)

A_fit, gamma_fit, w0_fit = popt_lor
FWHM = 2.0 * gamma_fit

perr = np.sqrt(np.diag(pcov_lor))
A_err, gamma_err, w0_err = perr
FWHM_err = 2.0 * gamma_err

lorentz_fit = lorentz(w, *popt_lor)

plt.figure(figsize=(8, 5))

plt.plot(w, S_raw, 'k.', ms=3, alpha=0.4, label='Raw data')
plt.plot(w, S, '-', lw=2, color='blue', label='Noise-free (interpolated)')
plt.plot(w, bg_fit_full, '--', lw=2, color='orange', label='Linear background')
plt.plot(w, lorentz_fit + bg_fit_full, '-', lw=2.5, color='red',
         label='Lorentz + BG fit')

plt.axvspan(w[lo], w[hi-1], color='red', alpha=0.12, label='Removed noise')

plt.xlabel('Energy transfer ω')
plt.ylabel('S(Q, ω)')
plt.yscale('log')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("="*60)
print("LINEAR BACKGROUND:")
print(f"  b0 = {b0_fit:.6g}")
print(f"  b1 = {b1_fit:.6g}")
print()
print("LORENTZIAN FIT:")
print(f"  ω₀   = {w0_fit:.6g} ± {w0_err:.3g}")
print(f"  A    = {A_fit:.6g} ± {A_err:.3g}")
print(f"  Γ    = {gamma_fit:.6g} ± {gamma_err:.3g}")
print(f"  FWHM = {FWHM:.6g} ± {FWHM_err:.3g}")
print("="*60)


In [ ]:
Na_disf[i].min()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Assuming Na_disf and omega are already defined in your environment
for i in range(0,105,10):
    S_clean = Na_disf[i].copy()
    w = omega
    
    # --- 2) Models ---
    def lorentz_with_bg(w, A, gamma, w0, b0):
        """Single Lorentzian + Flat Background"""
        return (A/np.pi) * (gamma / ((w - w0)**2 + gamma**2)) + b0

    # --- 3) Initial Guesses ---
    # Flat background guess: median of the tails
    tail_n = int(len(S_clean) * 0.1)
    b0_guess = np.median(np.concatenate([S_clean[:tail_n], S_clean[-tail_n:]]))
    
    # Lorentzian guesses
    S_sub = S_clean - b0_guess
    A_guess = np.max(S_sub) * np.pi * 0.05 
    w0_guess = w[np.argmax(S_sub)]
    gamma_guess = (w.max() - w.min()) * 0.05
    
    p0 = [A_guess, gamma_guess, w0_guess, b0_guess]
    bounds = ([0, 0, -np.inf, 0], [np.inf, np.inf, np.inf, Na_disf[i].min()*2])

    # --- 4) Fit ---
    try:
        popt, pcov = curve_fit(lorentz_with_bg, w, S_clean, p0=p0, bounds=bounds)
        A_f, gamma_f, w0_f, b0_f = popt
        perr = np.sqrt(np.diag(pcov))
        
        fwhm = 2.0 * gamma_f
        fwhm_err = 2.0 * perr[1]
        
    except Exception as e:
        print(f"Fit failed for index {i}: {e}")
        continue

    # --- 5) Plotting ---
    fig, axes = plt.subplots(2, 1, figsize=(8, 8), gridspec_kw={'height_ratios':[2, 1]})
    
    # Main Plot
    ax = axes[0]
    ax.plot(w, S_raw, 'k.', alpha=0.3, label='Original Data')
    ax.plot(w, S_clean, 'b.', ms=4, label='Cleaned Data')
    ax.plot(w, lorentz_with_bg(w, *popt), 'r-', lw=2, label='Fit (Lorentz + Flat BG)')
    ax.axhline(b0_f, color='orange', linestyle='--', label=f'Flat BG ({b0_f:.3e})')
    
    ax.set_yscale('log')
    ax.set_ylabel('S(Q, ω)')
    ax.set_title(f'Index {i}: Single Lorentz Fit')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Residuals
    ax2 = axes[1]
    residuals = S_clean - lorentz_with_bg(w, *popt)
    ax2.plot(w, residuals, 'g.')
    ax2.axhline(0, color='black', lw=1)
    ax2.set_ylabel('Residuals')
    ax2.set_xlabel('ω')
    
    plt.tight_layout()
    plt.show()

    print(f"RESULTS FOR INDEX {i}:")
    print(f"FWHM: {fwhm:.6e} ± {fwhm_err:.3e}")
    print(f"Center: {w0_f:.6e}")
    print(f"Background: {b0_f:.6e}")
    print("-" * 30)

In [ ]:
q

### 1 Lorentz without Linear BG

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit

indices = []
FWHM1_list = []
FWHM1_err_list = []

for i in range(25, 190, 5):
    S = H_disf[i]
    w = omega

    # ---- Remove elastic peak ----
    S_raw = S.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean = S_raw.copy()
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    S = S_clean

    # ---- Model ----
    def lorentz(w, A, gamma, w0):
        return (A / np.pi) * (gamma / ((w - w0)**2 + gamma**2))

    # ---- Fit window ----
    fit_window = 0.6 * (w.max() - w.min())
    w_span = fit_window / 2.0
    fit_mask = np.abs(w) <= w_span

    # ---- Initial guesses ----
    peak_height = np.max(S)
    A0 = peak_height * np.pi * 0.05
    p0 = [A0, 0.002, 0]
    bounds_lower = [1e-10, 1e-10, -np.inf]
    bounds_upper = [np.inf,  np.inf,  np.inf]

    try:
        popt, pcov = curve_fit(
            lorentz,
            w[fit_mask], S[fit_mask],
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A_fit, gamma_fit, w0_fit = popt
        perr = np.sqrt(np.diag(pcov))
        A_err, gamma_err, w0_err = perr
    except Exception as e:
        raise RuntimeError(f"Fit failed: {e}")

    # ---- Fit curve ----
    fit_total = lorentz(w, *popt)

    # ---- Plot ----
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), gridspec_kw={'height_ratios': [2, 1]})
    ax = axes[0]
    ax.plot(w, S, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(w, fit_total, '-', lw=2.5, color='red', label=f'Lorentz (Γ={gamma_fit:.3g})')
    ax.set_xlabel('Energy transfer ω')
    ax.set_ylabel('S(Q, ω)')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Lorentzian Fit — q = {round(q[i]/10, 3)}')

    ax2 = axes[1]
    ax2.plot(w, S - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Energy transfer ω')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    indices.append(i)
    FWHM1_list.append(2 * gamma_fit)
    FWHM1_err_list.append(2 * gamma_err)

    # ---- Print results ----
    print(f"\n{'='*50}  Q index {i}")
    print(f"Lorentzian:  A={A_fit:.4g}±{A_err:.2g}  Γ={gamma_fit:.4g}±{gamma_err:.2g}  ω0={w0_fit:.4g}±{w0_err:.2g}  FWHM={2*gamma_fit:.4g}")

### 2 Lorentz without Linear BG

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit
indices = []
FWHM1_list = []
FWHM2_list = []
FWHM1_err_list = []
FWHM2_err_list = []
for i in range(25, 190,5):
    S = H_disf[i]
    w = np.linspace(-0.51675,0.51675,2499)

    # ---- Remove elastic peak ----
    S_raw = S.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 1, 0)
    hi = min(i0 + 2, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean = S_raw.copy()
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    S = S_clean

    # ---- Helper models ----
    def lorentz(w, A, gamma, w0):
        return (A / np.pi) * (gamma / ((w - w0)**2 + gamma**2))

    def double_lorentz(w, A1, gamma1, w01, A2, gamma2, w02):
        return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02)

    # ---- Fit window ----
    fit_window = 0.6 * (w.max() - w.min())
    w_span = fit_window / 2.0
    fit_mask = np.abs(w) <= w_span

    # ---- Initial guesses ----
    peak_height = np.max(S)
    A0 = peak_height * np.pi * 0.05
    gamma0 = (w.max() - w.min()) * 0.05

    p0 = [A0, 0.002, 0,
          A0, 0.05, 0]

    bounds_lower = [1e-10, 1e-10, -np.inf, 1e-10, 1e-10, -np.inf]
    bounds_upper = [np.inf, np.inf,  np.inf, np.inf, np.inf,  np.inf]

    try:
        popt, pcov = curve_fit(
            double_lorentz,
            w[fit_mask], S[fit_mask],
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A1_fit, gamma1_fit, w01_fit, A2_fit, gamma2_fit, w02_fit = popt
        perr = np.sqrt(np.diag(pcov))
        A1_err, gamma1_err, w01_err, A2_err, gamma2_err, w02_err = perr
    except Exception as e:
        raise RuntimeError(f"Fit failed: {e}")

    # ---- Fit curves ----
    fit_total   = double_lorentz(w, *popt)
    lorentz1_fit = lorentz(w, A1_fit, gamma1_fit, w01_fit)
    lorentz2_fit = lorentz(w, A2_fit, gamma2_fit, w02_fit)

    # ---- Plot ----
    
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), gridspec_kw={'height_ratios': [2, 1]})

    ax = axes[0]
    ax.plot(w, S, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(w, fit_total,    '-',  lw=2.5, color='red',    label='Double Lorentz (total)')
    ax.plot(w, lorentz1_fit, '--', lw=1.5, color='green',  label=f'Lorentz 1 (Γ={gamma1_fit:.3g})')
    ax.plot(w, lorentz2_fit, '--', lw=1.5, color='purple', label=f'Lorentz 2 (Γ={gamma2_fit:.3g})')
    ax.set_xlabel('Energy transfer ω')
    ax.set_ylabel('S(Q, ω)')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Double Lorentzian Fit — q = {round(q[i]/10,3)}')

    ax2 = axes[1]
    ax2.plot(w, S - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Energy transfer ω')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    
    indices.append(i)
    FWHM1_list.append(2 * gamma1_fit)
    FWHM2_list.append(2 * gamma2_fit)
    FWHM1_err_list.append(2 * gamma1_err)
    FWHM2_err_list.append(2 * gamma2_err)
    # ---- Print results ----
    print(f"\n{'='*50}  Q index {i}")
    print(f"Lorentzian 1:  A={A1_fit:.4g}±{A1_err:.2g}  Γ={gamma1_fit:.4g}±{gamma1_err:.2g}  ω0={w01_fit:.4g}±{w01_err:.2g}  FWHM={2*gamma1_fit:.4g}")
    print(f"Lorentzian 2:  A={A2_fit:.4g}±{A2_err:.2g}  Γ={gamma2_fit:.4g}±{gamma2_err:.2g}  ω0={w02_fit:.4g}±{w02_err:.2g}  FWHM={2*gamma2_fit:.4g}")

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =====================================================
# Storage lists
# =====================================================
indices = []

FWHM1_list = []
FWHM2_list = []

FWHM1_err_list = []
FWHM2_err_list = []

background_list = []
background_err_list = []

# Set this if it is not already defined
singlecol = 3.54

# =====================================================
# Model functions
# =====================================================
def lorentz(w, A, gamma, w0):
    """
    Lorentzian function.

    A     : integrated area
    gamma : half width at half maximum (HWHM)
    w0    : peak center
    """
    return (A / np.pi) * (
        gamma / ((w - w0)**2 + gamma**2)
    )


def double_lorentz_background(
    w,
    A1, gamma1, w01,
    A2, gamma2, w02,
    background
):
    """Two Lorentzian functions plus a constant background."""
    return (
        lorentz(w, A1, gamma1, w01)
        + lorentz(w, A2, gamma2, w02)
        + background
    )


# =====================================================
# Loop over Q indices
# =====================================================
for i in range(25, 190, 5):

    S = np.asarray(total_disf[i], dtype=float)
    w = np.linspace(-0.51675, 0.51675, 2499)

    # =================================================
    # Remove elastic peak
    # =================================================
    S_raw = S.copy()

    i0 = np.argmax(S_raw)

    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))

    good_idx = np.r_[
        0:lo,
        hi:len(S_raw)
    ]

    S_clean = S_raw.copy()

    S_clean[lo:hi] = np.interp(
        np.arange(lo, hi),
        good_idx,
        S_raw[good_idx]
    )

    S = S_clean

    # =================================================
    # Define fitting window
    # =================================================
    fit_window = 0.6 * (w.max() - w.min())
    w_span = fit_window / 2.0

    fit_mask = np.abs(w) <= w_span

    w_fit = w[fit_mask]
    S_fit = S[fit_mask]

    # Remove non-finite values if present
    finite_mask = np.isfinite(w_fit) & np.isfinite(S_fit)

    w_fit = w_fit[finite_mask]
    S_fit = S_fit[finite_mask]

    # =================================================
    # Initial parameter estimates
    # =================================================

    # Estimate background from the outer 20% of the
    # selected fitting window
    edge_mask = (
        np.abs(w_fit)
        > 0.8 * np.max(np.abs(w_fit))
    )

    background0 = np.median(S_fit[edge_mask])

    # The background is constrained to be non-negative
    background0 = max(background0, 1e-10)

    peak_height = max(
        np.max(S_fit) - background0,
        1e-10
    )

    # Lorentzian peak height at w = w0 is A / (pi*gamma).
    # Therefore A is approximately:
    # A = peak_height * pi * gamma
    gamma1_0 = 0.002
    gamma2_0 = 0.050

    A1_0 = 0.5 * peak_height * np.pi * gamma1_0
    A2_0 = 0.5 * peak_height * np.pi * gamma2_0

    p0 = [
        A1_0, gamma1_0, 0.0,
        A2_0, gamma2_0, 0.0,
        background0
    ]

    # =================================================
    # Parameter bounds
    # =================================================
    bounds_lower = [
        1e-10, 1e-10, -np.inf,
        1e-10, 1e-10, -np.inf,
        0.0
    ]

    bounds_upper = [
        np.inf, np.inf, np.inf,
        np.inf, np.inf, np.inf,
        np.inf
    ]

    # =================================================
    # Perform fit
    # =================================================
    try:
        popt, pcov = curve_fit(
            double_lorentz_background,
            w_fit,
            S_fit,
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )

        (
            A1_fit,
            gamma1_fit,
            w01_fit,
            A2_fit,
            gamma2_fit,
            w02_fit,
            background_fit
        ) = popt

        perr = np.sqrt(np.diag(pcov))

        (
            A1_err,
            gamma1_err,
            w01_err,
            A2_err,
            gamma2_err,
            w02_err,
            background_err
        ) = perr

    except Exception as e:
        raise RuntimeError(
            f"Fit failed for Q index {i}: {e}"
        )

    # =================================================
    # Calculate fitted curves over the complete w range
    # =================================================
    lorentz1_fit = lorentz(
        w,
        A1_fit,
        gamma1_fit,
        w01_fit
    )

    lorentz2_fit = lorentz(
        w,
        A2_fit,
        gamma2_fit,
        w02_fit
    )

    background_curve = np.full_like(
        w,
        background_fit
    )

    fit_total = double_lorentz_background(
        w,
        *popt
    )

    residual = S - fit_total

    # =================================================
    # Plot
    # =================================================
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(singlecol * 2, singlecol * 2),
        gridspec_kw={
            'height_ratios': [2, 1]
        },
        sharex=True
    )

    # Main fit plot
    ax = axes[0]

    ax.plot(
        w,
        S,
        'k.',
        ms=4,
        alpha=0.6,
        label='Data'
    )

    ax.plot(
        w,
        fit_total,
        '-',
        lw=2.5,
        color='red',
        label='Double Lorentz + background'
    )

    ax.plot(
        w,
        lorentz1_fit,
        '--',
        lw=1.5,
        color='green',
        label=(
            f'Lorentz 1 '
            f'(Γ={gamma1_fit:.3g})'
        )
    )

    ax.plot(
        w,
        lorentz2_fit,
        '--',
        lw=1.5,
        color='purple',
        label=(
            f'Lorentz 2 '
            f'(Γ={gamma2_fit:.3g})'
        )
    )

    ax.plot(
        w,
        background_curve,
        ':',
        lw=2,
        color='orange',
        label=(
            f'Flat background '
            f'({background_fit:.3g})'
        )
    )

    # Show fitting-window boundaries
    ax.axvline(
        -w_span,
        color='grey',
        linestyle=':',
        linewidth=1
    )

    ax.axvline(
        w_span,
        color='grey',
        linestyle=':',
        linewidth=1
    )

    ax.set_ylabel(r'$S(Q,\omega)$')
    ax.set_yscale('log')

    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    ax.set_title(
        f'Double Lorentzian Fit with Flat Background'
        f'\nq = {q[i] / 10:.3f}'
    )

    # Residual plot
    ax2 = axes[1]

    ax2.plot(
        w,
        residual,
        '.',
        ms=4,
        color='blue',
        alpha=0.6
    )

    ax2.axhline(
        0,
        color='black',
        lw=0.8,
        ls='--'
    )

    ax2.axvline(
        -w_span,
        color='grey',
        linestyle=':',
        linewidth=1
    )

    ax2.axvline(
        w_span,
        color='grey',
        linestyle=':',
        linewidth=1
    )

    ax2.set_xlabel(r'Energy transfer $\omega$')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # =================================================
    # Store fitted results
    # =================================================
    indices.append(i)

    FWHM1_list.append(2 * gamma1_fit)
    FWHM2_list.append(2 * gamma2_fit)

    FWHM1_err_list.append(2 * gamma1_err)
    FWHM2_err_list.append(2 * gamma2_err)

    background_list.append(background_fit)
    background_err_list.append(background_err)

    # =================================================
    # Print results
    # =================================================
    print(f"\n{'=' * 60}")
    print(
        f"Q index {i}, "
        f"q = {q[i] / 10:.4f}"
    )

    print(
        f"Lorentzian 1:\n"
        f"  A    = {A1_fit:.4g} ± {A1_err:.2g}\n"
        f"  Γ    = {gamma1_fit:.4g} ± {gamma1_err:.2g}\n"
        f"  ω0   = {w01_fit:.4g} ± {w01_err:.2g}\n"
        f"  FWHM = {2 * gamma1_fit:.4g} "
        f"± {2 * gamma1_err:.2g}"
    )

    print(
        f"Lorentzian 2:\n"
        f"  A    = {A2_fit:.4g} ± {A2_err:.2g}\n"
        f"  Γ    = {gamma2_fit:.4g} ± {gamma2_err:.2g}\n"
        f"  ω0   = {w02_fit:.4g} ± {w02_err:.2g}\n"
        f"  FWHM = {2 * gamma2_fit:.4g} "
        f"± {2 * gamma2_err:.2g}"
    )

    print(
        f"Flat background:\n"
        f"  B = {background_fit:.4g} "
        f"± {background_err:.2g}"
    )


# =====================================================
# Convert results to NumPy arrays
# =====================================================
indices = np.asarray(indices)

FWHM1_array = np.asarray(FWHM1_list)
FWHM2_array = np.asarray(FWHM2_list)

FWHM1_err_array = np.asarray(FWHM1_err_list)
FWHM2_err_array = np.asarray(FWHM2_err_list)

background_array = np.asarray(background_list)
background_err_array = np.asarray(background_err_list)

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='o-', color='green',
            capsize=4, label='FWHM Lorentz 1')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='purple',
            capsize=4, label='FWHM Lorentz 2')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
plt.ylim(0,0.1)

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='o-', color='green',
            capsize=4, label='FWHM Lorentz 1')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='purple',
            capsize=4, label='FWHM Lorentz 2')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
#plt.ylim(0,0.1)

### 3 Lorentz without Linear BG

In [ ]:
plt.figure()
for i in range(25,190,5):
    plt.plot(omega,H_disf[i,:])
plt.xlim(-0.1,0.1)
plt.yscale('log')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---- Axes: verify these match your data ----
omega = np.linspace(-0.51675, 0.51675, total_disf.shape[1])  # meV
q_values = np.asarray(q, dtype=float)  # Å^-1; use q / 10 if q is in nm^-1

spectra = np.asarray(total_disf, dtype=float)
assert spectra.shape == (len(q_values), len(omega))

# ---- Models: all components centred at zero ----
def lorentz(w, A, gamma):
    return (A / np.pi) * gamma / (w**2 + gamma**2)

def triple_lorentz(w, A1, g1, A2, g2, A3, g3):
    return (
        lorentz(w, A1, g1)
        + lorentz(w, A2, g2)
        + lorentz(w, A3, g3)
    )

# ---- Storage: columns = narrow, intermediate, broad ----
nq = spectra.shape[0]
Gamma = np.full((nq, 3), np.nan)
Gamma_err = np.full((nq, 3), np.nan)
Amplitude = np.full((nq, 3), np.nan)
fit_parameters = np.full((nq, 6), np.nan)

# Exclude the central elastic bin.
# Set half_window = 1 to exclude the central bin and one on either side.
half_window = 1
i0 = np.argmin(np.abs(omega))
elastic_mask = np.ones(len(omega), dtype=bool)
elastic_mask[max(0, i0-half_window):i0+half_window+1] = False

failed_indices = []

fit_indices = range(25, 190)

for i in fit_indices:
    S_raw = spectra[i]

    mask = elastic_mask & np.isfinite(S_raw)
    x = omega[mask]
    y = S_raw[mask]

    if len(y) <= 6 or np.max(y) <= 0:
        failed_indices.append(i)
        continue

    # Normalize intensity for numerical stability; widths are unchanged.
    scale = np.max(y)
    y = y / scale

    # Positive starting estimate for the integrated intensity.
    area0 = np.sum(np.clip(y, 0, None)) * abs(omega[1] - omega[0])

    best = None

    # Try several initial widths to reduce sensitivity to initial guesses.
    for factor in (0.5, 1.0, 2.0):
        g0 = factor * np.array([0.001, 0.01, 0.1])  # meV
        p0 = [
            area0 / 3, g0[0],
            area0 / 3, g0[1],
            area0 / 3, g0[2],
        ]

        try:
            popt, pcov = curve_fit(
                triple_lorentz,
                x, y,
                p0=p0,
                bounds=(
                    [0, 1e-8, 0, 1e-8, 0, 1e-8],
                    [np.inf] * 6,
                ),
                method="trf",
                x_scale="jac",
                max_nfev=10000,
            )

            rss = np.sum((y - triple_lorentz(x, *popt))**2)
            if best is None or rss < best[0]:
                best = (rss, popt, pcov)

        except (RuntimeError, ValueError):
            continue

    if best is None:
        failed_indices.append(i)
        continue

    _, popt, pcov = best
    perr = np.sqrt(np.maximum(np.diag(pcov), 0))

    # Sort components by Gamma at each q.
    order = np.argsort(popt[1::2])
    Gamma[i] = popt[1::2][order]
    Gamma_err[i] = perr[1::2][order]
    Amplitude[i] = popt[0::2][order] * scale

    fit_parameters[i, 0::2] = Amplitude[i]
    fit_parameters[i, 1::2] = Gamma[i]

# ---- Requested lists: retain alignment with q_values ----
Gamma1_list, Gamma2_list, Gamma3_list = Gamma.T.tolist()
Gamma1_err_list, Gamma2_err_list, Gamma3_err_list = Gamma_err.T.tolist()

In [ ]:
%matplotlib widget
plot_indices = fit_indices

for i in plot_indices:
    if not np.all(np.isfinite(fit_parameters[i])):
        continue

    # Same mask used during fitting
    mask = elastic_mask & np.isfinite(spectra[i])
    x = omega[mask]
    y = spectra[i, mask]

    A1, g1, A2, g2, A3, g3 = fit_parameters[i]

    L1 = lorentz(x, A1, g1)
    L2 = lorentz(x, A2, g2)
    L3 = lorentz(x, A3, g3)
    total_fit = L1 + L2 + L3

    residual = y - total_fit
    rmse = np.sqrt(np.mean(residual**2))

    fig, (ax, ax_err) = plt.subplots(
        2, 1, figsize=(7, 6),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08}
    )

    # ---- Masked spectrum and fitted components ----
    ax.plot(x, y, ".", color="0.5", markersize=3,
            label="Spectrum (elastic bin excluded)")
    ax.plot(x, total_fit, "k-", linewidth=2, label="Total fit")

    for L, gamma, color, label in zip(
        [L1, L2, L3],
        [g1, g2, g3],
        ["tab:blue", "tab:orange", "tab:green"],
        ["L1", "L2", "L3"],
    ):
        ax.plot(
            x, L, "--", color=color,
            label=fr"{label}: $\Gamma$ = {gamma:.4g} meV"
        )

    ax.set_ylabel(r"$S(q,\omega)$")
    ax.set_title(
        fr"$q = {q_values[i]/10:.3f}\ \mathrm{{\AA}}^{{-1}}$  (i = {i})"
    )
    ax.legend(fontsize=9)
    ax.set_yscale('log')
    ax.set_xlim(-0.2,0.2)

    # ---- Fitting residuals, evaluated only at unmasked points ----
    ax_err.axhline(0, color="black", linewidth=1)
    ax_err.plot(x, residual, ".", color="tab:red", markersize=3)
    ax_err.set_ylabel("Data − fit")
    ax_err.set_xlabel("Energy transfer (meV)")
    ax_err.text(
        0.98, 0.93, f"RMSE = {rmse:.3g}",
        transform=ax_err.transAxes,
        ha="right", va="top", fontsize=9
    )

In [ ]:
colors = ["tab:purple", "tab:green", "tab:purple", "tab:purple"]
labels = ["Lorentzian 1", "Lorentzian 2", "Lorentzian 3", "Lorentzian 4"]

# Plot only selected, successful fits, ordered by q.
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(Gamma[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))

for k in [1]:#range(4):
    ax.errorbar(
        q_values[idx]/10, Gamma[idx, k],
        yerr=Gamma_err[idx, k],
        fmt="o-", markersize=3, capsize=2,
        color=colors[k], label=labels[k],
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"$\Gamma$ (meV, HWHM)")
ax.legend(frameon=False, fontsize=16)
#ax.set_ylim(0,0.002)
fig.tight_layout()


In [ ]:
# ---- Plot Gamma vs q ----
fig, ax = plt.subplots(figsize=(7, 5))

for k, (color, label) in enumerate(zip(
    ["tab:blue", "tab:orange", "tab:green"],
    ["L1: narrow", "L2: intermediate", "L3: broad"],
)):
    ax.errorbar(
        q_values, Gamma[:, k],
        yerr=Gamma_err[:, k],
        fmt="o-", markersize=3, capsize=2,
        color=color, label=label,
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"$\Gamma$ (meV, HWHM)")
ax.legend()
fig.tight_layout()
plt.show()
plt.ylim(0,0.1)
print("Failed fit indices:", failed_indices)

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit
indices = []
FWHM1_list = []
FWHM2_list = []
FWHM3_list = []
FWHM1_err_list = []
FWHM2_err_list = []
FWHM3_err_list = []

omega = np.linspace(-0.51675,0.51675,2499)

    # ---- Helper models ----
def lorentz(w, A, gamma, w0):
    return (A / np.pi) * (gamma / ((w - w0)**2 + gamma**2))

def triple_lorentz(w, A1, gamma1, w01, A2, gamma2, w02, A3, gamma3, w03):
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02) + lorentz(w, A3, gamma3, w03)
        
for i in range(25, 190, 10):
    S = total_disf[i]
    w = omega

    # ---- Remove elastic peak ----
    S_raw = S.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean = S_raw.copy()
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    S = S_clean

    # ---- Fit window ----
    fit_window = 1 * (w.max() - w.min())
    w_span = fit_window / 2.0
    fit_mask = np.abs(w) <= w_span

    # ---- Initial guesses ----
    peak_height = np.max(S)
    A0 = peak_height * np.pi * 0.05
    gamma0 = (w.max() - w.min()) * 0.05
    w_sep = w_span / 2.0

    p0 = [
        A0, 0.001, 0,
        A0, 0.02,  0,
        A0, 1,  0
    ]

    bounds_lower = [1e-10, 1e-10, -0.005,
                    0.01, 0.02, -0.005,
                    1e-10, 0.5, -0.005]

    bounds_upper = [np.inf, 0.01, 0.005,
                    np.inf, 0.3, 0.005,
                    np.inf, np.inf, 0.005]

    try:
        popt, pcov = curve_fit(
            triple_lorentz,
            w[fit_mask], S[fit_mask],
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A1_fit, gamma1_fit, w01_fit, \
        A2_fit, gamma2_fit, w02_fit, \
        A3_fit, gamma3_fit, w03_fit = popt

        perr = np.sqrt(np.diag(pcov))
        A1_err, gamma1_err, w01_err, \
        A2_err, gamma2_err, w02_err, \
        A3_err, gamma3_err, w03_err = perr

    except Exception as e:
        print(f"Fit failed at index {i}: {e}")
        continue  # skip this index instead of crashing

    # ---- Fit curves ----
    fit_total    = triple_lorentz(w, *popt)
    lorentz1_fit = lorentz(w, A1_fit, gamma1_fit, w01_fit)
    lorentz2_fit = lorentz(w, A2_fit, gamma2_fit, w02_fit)
    lorentz3_fit = lorentz(w, A3_fit, gamma3_fit, w03_fit)

    # ---- Plot (uncomment to enable) ----
    
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2), gridspec_kw={'height_ratios': [2, 1]})
    ax = axes[0]
    ax.plot(w, S, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(w, fit_total,    '-',  lw=2.5, color='red',    label='Triple Lorentz (total)')
    ax.plot(w, lorentz1_fit, '--', lw=1.5, color='green',  label=f'Lorentzian 1 (Γ={gamma1_fit:.3g})')
    ax.plot(w, lorentz2_fit, '--', lw=1.5, color='purple', label=f'Lorentzian 2 (Γ={gamma2_fit:.3g})')
    ax.plot(w, lorentz3_fit, '--', lw=1.5, color='orange', label=f'Lorentzian 3 (Γ={gamma3_fit:.3g})')
    ax.set_xlabel('Energy transfer ω')
    ax.set_ylabel('S(Q, ω)')
    ax.set_yscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Triple Lorentzian Fit — q = {round(q[i]/10, 3)}')
    ax2 = axes[1]
    ax2.plot(w, S - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Energy transfer ω')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    

    # ---- Collect results ----
    indices.append(i)
    FWHM1_list.append(2 * gamma1_fit)
    FWHM2_list.append(2 * gamma2_fit)
    FWHM3_list.append(2 * gamma3_fit)
    FWHM1_err_list.append(2 * gamma1_err)
    FWHM2_err_list.append(2 * gamma2_err)
    FWHM3_err_list.append(2 * gamma3_err)

    print(f"\n{'='*50}  Q index {i}")
    print(f"Lorentzian 1:  A={A1_fit:.4g}±{A1_err:.2g}  Γ={gamma1_fit:.4g}±{gamma1_err:.2g}  ω0={w01_fit:.4g}±{w01_err:.2g}  FWHM={2*gamma1_fit:.4g}")
    print(f"Lorentzian 2:  A={A2_fit:.4g}±{A2_err:.2g}  Γ={gamma2_fit:.4g}±{gamma2_err:.2g}  ω0={w02_fit:.4g}±{w02_err:.2g}  FWHM={2*gamma2_fit:.4g}")
    print(f"Lorentzian 3:  A={A3_fit:.4g}±{A3_err:.2g}  Γ={gamma3_fit:.4g}±{gamma3_err:.2g}  ω0={w03_fit:.4g}±{w03_err:.2g}  FWHM={2*gamma3_fit:.4g}")

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='o-', color='green',
#            capsize=4, label='FWHM Lorentzian 2')
#ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='purple',
#            capsize=4, label='FWHM Lorentzian 3')
ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list, fmt='s-', color='orange',
           capsize=4, label='FWHM Lorentz 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
#plt.ylim(0,0.1)

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='o-', color='purple',
#           capsize=4, label='FWHM Lorentz 3')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='green',
            capsize=4, label='FWHM Lorentz 2')
#ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list, fmt='s-', color='orange',
#            capsize=4, label='FWHM Lorentz 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
plt.ylim(0,0.2)

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='o-', color='green',
            capsize=4, label='FWHM Lorentz 1')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='purple',
            capsize=4, label='FWHM Lorentz 2')
ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list, fmt='s-', color='orange',
            capsize=4, label='FWHM Lorentz 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
plt.ylim(0,2)

### 4 Lorentzian without linear BG

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ---- Axes: verify these match your data ----
omega = np.linspace(-0.51675, 0.51675, total_disf.shape[1])  # meV
q_values = np.asarray(q, dtype=float)  # Å^-1; use q / 10 if q is in nm^-1

spectra = np.asarray(H_disf, dtype=float)
assert spectra.shape == (len(q_values), len(omega))

# ---- Models ----
def lorentz(w, A, gamma):
    return (A / np.pi) * gamma / (w**2 + gamma**2)

def four_lorentz(w, A1, g1, A2, g2, A3, g3, A4, g4):
    return (
        lorentz(w, A1, g1)
        + lorentz(w, A2, g2)
        + lorentz(w, A3, g3)
        + lorentz(w, A4, g4)
    )

# Indices 190 down to 11; use range(190, 9, -1) to include 10.
fit_indices = range(25,190,10)

# ---- Reset storage for four components ----
nq = spectra.shape[0]
Gamma = np.full((nq, 4), np.nan)
Gamma_err = np.full((nq, 4), np.nan)
Amplitude = np.full((nq, 4), np.nan)
fit_parameters = np.full((nq, 8), np.nan)
RMSE = np.full(nq, np.nan)
failed_indices = []

# Parameter order: [A1, g1, A2, g2, A3, g3, A4, g4]
lower_bounds = [0, 1e-8] * 4
upper_bounds = [np.inf] * 8

# ---- Fit each selected spectrum ----
for i in fit_indices:
    S_raw = spectra[i]

    if np.isclose(q_values[i], 0):
        continue

    mask = elastic_mask & np.isfinite(S_raw)
    x = omega[mask]
    y_raw = S_raw[mask]

    if len(y_raw) <= 8 or np.max(y_raw) <= 0:
        failed_indices.append(i)
        continue

    # Normalize intensity for numerical stability.
    scale = np.max(y_raw)
    y = y_raw / scale

    area0 = np.sum(np.clip(y, 0, None)) * abs(omega[1] - omega[0])
    best = None

    # Try several initial width combinations.
    for factor in (0.5, 1.0, 2.0):
        g0 = factor * np.array([0.001, 0.005, 0.025, 0.125])

        p0 = np.empty(8)
        p0[0::2] = area0 / 4
        p0[1::2] = g0

        try:
            popt, pcov = curve_fit(
                four_lorentz,
                x, y,
                p0=p0,
                bounds=(lower_bounds, upper_bounds),
                method="trf",
                x_scale="jac",
                max_nfev=20000,
            )

            rss = np.sum((y - four_lorentz(x, *popt))**2)

            if best is None or rss < best[0]:
                best = (rss, popt, pcov)

        except (RuntimeError, ValueError):
            continue

    if best is None:
        failed_indices.append(i)
        continue

    _, popt, pcov = best
    perr = np.sqrt(np.maximum(np.diag(pcov), 0))

    # Sort: L1 narrowest → L4 broadest.
    order = np.argsort(popt[1::2])

    Gamma[i] = popt[1::2][order]
    Gamma_err[i] = perr[1::2][order]
    Amplitude[i] = popt[0::2][order] * scale

    fit_parameters[i, 0::2] = Amplitude[i]
    fit_parameters[i, 1::2] = Gamma[i]

    residual = y_raw - four_lorentz(x, *fit_parameters[i])
    RMSE[i] = np.sqrt(np.mean(residual**2))

# ---- Lists: each entry remains aligned with its spectrum index ----
Gamma1_list, Gamma2_list, Gamma3_list, Gamma4_list = Gamma.T.tolist()

(
    Gamma1_err_list,
    Gamma2_err_list,
    Gamma3_err_list,
    Gamma4_err_list,
) = Gamma_err.T.tolist()

print("Failed fit indices:", failed_indices)

In [ ]:
colors = ['orange', 'red', 'green', 'purple']

for i in fit_indices:
    if not np.all(np.isfinite(fit_parameters[i])):
        continue

    mask = elastic_mask & np.isfinite(spectra[i])
    x = omega[mask]
    y = spectra[i, mask]

    components = np.array([
        lorentz(x, Amplitude[i, k], Gamma[i, k])
        for k in range(4)
    ])
    total_fit = components.sum(axis=0)
    residual = y - total_fit

    fig, (ax, ax_err) = plt.subplots(
        2, 1, figsize=(7, 6),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    )

    ax.plot(
        x, y, ".", color="0.5", markersize=3,
        label="Masked spectrum",
    )
    ax.plot(x, total_fit, "k-", linewidth=2, label="Total fit")

    for k in range(4):
        ax.plot(
            x, components[k], "--", color=colors[k],
            label=fr"{labels[k]}: $\Gamma$ = {Gamma[i, k]:.4g} meV",
        )

    ax.set_ylabel(r"$S(q,\omega)$")
    ax.set_title(
        fr"$q = {q_values[i]:.3f}\ \mathrm{{\AA}}^{{-1}}$  (i = {i})"
    )
    ax.legend(fontsize=9)

    ax_err.axhline(0, color="black", linewidth=1)
    ax_err.plot(x, residual, ".", color="tab:red", markersize=3)
    ax_err.set_ylabel("Data − fit")
    ax_err.set_xlabel("Energy transfer (meV)")
    ax_err.text(
        0.98, 0.93, f"RMSE = {RMSE[i]:.3g}",
        transform=ax_err.transAxes,
        ha="right", va="top", fontsize=9,
    )

    ax.set_yscale('log')
    ax.set_xlim(-0.2,0.2)

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple"]
labels = ["L1", "L2", "L3", "L4"]

# Plot only selected, successful fits, ordered by q.
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(Gamma[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

fig, ax = plt.subplots(figsize=(7, 5))

for k in [1]:#range(4):
    ax.errorbar(
        q_values[idx], Gamma[idx, k],
        yerr=Gamma_err[idx, k],
        fmt="o-", markersize=3, capsize=2,
        color=colors[k], label=labels[k],
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"$\Gamma$ (meV, HWHM)")
ax.legend()
#ax.set_ylim(0,0.1)
fig.tight_layout()


In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

indices = []

FWHM1_list, FWHM2_list, FWHM3_list, FWHM4_list = [], [], [], []
FWHM1_err_list, FWHM2_err_list, FWHM3_err_list, FWHM4_err_list = [], [], [], []

FWHM_lists = [FWHM1_list, FWHM2_list, FWHM3_list, FWHM4_list]
FWHM_err_lists = [
    FWHM1_err_list, FWHM2_err_list,
    FWHM3_err_list, FWHM4_err_list
]

omega = np.linspace(-0.51675, 0.51675, 2499)
plot_fits = False


# ---- Helper models ----
def lorentz(w, A, gamma, w0):
    return (A / np.pi) * gamma / ((w - w0)**2 + gamma**2)


def quadruple_lorentz(
    w,
    A1, gamma1, w01,
    A2, gamma2, w02,
    A3, gamma3, w03,
    A4, gamma4, w04
):
    return (
        lorentz(w, A1, gamma1, w01)
        + lorentz(w, A2, gamma2, w02)
        + lorentz(w, A3, gamma3, w03)
        + lorentz(w, A4, gamma4, w04)
    )


for i in range(45, 190):
    S_raw = np.asarray(H_disf[i], dtype=float).copy()
    w = omega
    # ---- Replace the maximum-intensity bin by interpolation ----
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))

    good_idx = np.r_[0:lo, hi:len(S_raw)]

    S = S_raw.copy()
    S[lo:hi] = np.interp(
        np.arange(lo, hi),
        good_idx,
        S_raw[good_idx]
    )

    # ---- Fit window ----
    fit_window = 1.0 * (w.max() - w.min())
    w_span = fit_window / 2.0
    fit_mask = (np.abs(w) <= w_span) & np.isfinite(S)

    # ---- Initial guesses ----
    peak_height = np.max(S[fit_mask])

    gamma_guesses = [2, 0.05, 0.01, 0.001] #[0.001, 0.01, 0.05, 2.0]

    # Assign each component one quarter of the initial peak height.
    # At w = w0, the Lorentzian height is A / (pi * gamma).
    p0 = [
        value
        for gamma in gamma_guesses
        for value in [peak_height * np.pi * gamma / 4.0, gamma, 0.0]
    ]

    # Parameter order for each component: A, gamma, w0
    bounds_lower = [
        1e-10, 1e-10, -0.005,
        1e-10, 1e-10, -0.005,
        1e-10, 1e-10, -0.005,
        1e-10, 1e-10, -0.005
    ]

    bounds_upper = [
        np.inf, np.inf,   0.005,
        np.inf, 3,    0.005,
        np.inf, 0.5,  0.005,
        np.inf, 0.05, 0.005
    ]

    try:
        popt, pcov = curve_fit(
            quadruple_lorentz,
            w[fit_mask],
            S[fit_mask],
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=500000
        )

        # Rows correspond to components; columns are A, gamma, w0.
        params = popt.reshape(4, 3)
        errors = np.sqrt(np.diag(pcov)).reshape(4, 3)

    except (RuntimeError, ValueError) as e:
        print(f"Fit failed at index {i}: {e}")
        continue

    # ---- Fit curves ----
    fit_total = quadruple_lorentz(w, *popt)
    component_curves = [
        lorentz(w, *component_params)
        for component_params in params
    ]

    # ---- Optional plot ----
    if plot_fits:
        fig, axes = plt.subplots(
            2, 1,
            figsize=(8, 7),
            gridspec_kw={"height_ratios": [2, 1]},
            sharex=True
        )

        ax, ax2 = axes

        ax.plot(w, S, "k.", ms=4, alpha=0.6, label="Data")
        ax.plot(
            w, fit_total, "-", lw=2.5,
            color="red", label="Four Lorentzians (total)"
        )

        colors = ["orange", "red", "green", "purple"]

        for j, (curve, color) in enumerate(zip(component_curves, colors)):
            gamma_fit = params[j, 1]
            ax.plot(
                w, curve, "--", lw=1.5, color=color,
                label=f"Lorentzian {j + 1} (Γ={gamma_fit:.3g})"
            )

        ax.set_ylabel("S(Q, ω)")
        ax.set_yscale("log")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_title(
            f"Four-Lorentzian Fit — Q = {q[i] / 10:.3f}"
        )

        ax2.plot(
            w, S - fit_total, ".",
            ms=4, color="blue", alpha=0.6
        )
        ax2.axhline(0, color="k", lw=0.8, ls="--")
        ax2.set_xlabel("Energy transfer ω")
        ax2.set_ylabel("Residual")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # ---- Collect results ----
    indices.append(i)

    for j in range(4):
        FWHM_lists[j].append(2 * params[j, 1])
        FWHM_err_lists[j].append(2 * errors[j, 1])

    # ---- Print results ----
    print(f"\n{'=' * 50}  Q index {i}")

    for j, (component_params, component_errors) in enumerate(
        zip(params, errors), start=1
    ):
        A_fit, gamma_fit, w0_fit = component_params
        A_err, gamma_err, w0_err = component_errors

        print(
            f"Lorentzian {j}:  "
            f"A={A_fit:.4g}±{A_err:.2g}  "
            f"Γ={gamma_fit:.4g}±{gamma_err:.2g}  "
            f"ω0={w0_fit:.4g}±{w0_err:.2g}  "
            f"FWHM={2 * gamma_fit:.4g}±{2 * gamma_err:.2g}"
        )

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM4_list, yerr=FWHM4_err_list, fmt='o-', color='purple', capsize=4, label='FWHM Lorentzian 3')
ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list, fmt='s-', color='green', capsize=4, label='FWHM Lorentzian 2')
#ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list, fmt='s-', color='red',capsize=4, label='FWHM Lorentzian 1')
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list, fmt='s-', color='orange',capsize=4, label='FWHM Lorentzian 0')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
#plt.ylim(0,2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# -------------------------
# Initialize arrays to store results
# -------------------------
n_spectra = q.shape[0]
FWHM1_array = np.zeros(n_spectra)
FWHM2_array = np.zeros(n_spectra)
FWHM3_array = np.zeros(n_spectra)
FWHM1_err_array = np.zeros(n_spectra)
FWHM2_err_array = np.zeros(n_spectra)
FWHM3_err_array = np.zeros(n_spectra)

# Store other fit parameters if needed
w01_array = np.zeros(n_spectra)
w02_array = np.zeros(n_spectra)
w03_array = np.zeros(n_spectra)
A1_array = np.zeros(n_spectra)
A2_array = np.zeros(n_spectra)
A3_array = np.zeros(n_spectra)

# -------------------------
# Parameters you can change
# -------------------------
tail_fraction = 0.15   # fraction of the spectrum at each end used for BG estimation
use_indices = None     # alternatively, specify tuple (left_i, right_i) indices for background fit

# ---- Helper models ----
def linear_bg(w, b0, b1):
    return b0 + b1 * w

def lorentz(w, A, gamma, w0=0):
    """Lorentzian centered at w0: (A/pi) * gamma / ((w-w0)^2 + gamma^2)"""
    return (A/np.pi) * (gamma / ((w - w0)**2 + gamma**2))

def double_lorentz(w, A1, gamma1, w01, A2, gamma2, w02):
    """Sum of two Lorentzians"""
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02)
    
def triple_lorentz(w, A1, gamma1, w01, A2, gamma2, w02, A3, gamma3, w03):
    """Sum of three Lorentzians"""
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02) + lorentz(w, A3, gamma3, w03)

# -------------------------
# Main fitting loop
# -------------------------
for i in range(25,n_spectra):
    print(f"\n{'='*60}")
    print(f"Fitting spectrum {i+1}/{n_spectra}")
    print(f"{'='*60}")
    
    try:
        # Get data for this spectrum
        S_raw = total_disf[i].copy()
        w = omega
        
        # Remove elastic peak
        i0 = np.argmax(S_raw)
        lo = max(i0 - 3, 0)
        hi = min(i0 + 4, len(S_raw))
        
        good_idx = np.r_[0:lo, hi:len(S_raw)]
        
        S_clean = S_raw.copy()
        S_clean[lo:hi] = np.interp(
            np.arange(lo, hi),
            good_idx,
            S_raw[good_idx]
        )
        
        S = S_clean

        #S = Na_disf[i]
        # ---- 1) Select tail points for background estimation ----
        N = len(w)
        if use_indices is None:
            n_tail = max(2, int(np.round(tail_fraction * N)))
            left_idx = np.arange(0, n_tail)
            right_idx = np.arange(N - n_tail, N)
        else:
            left_i, right_i = use_indices
            left_idx = np.arange(0, left_i)
            right_idx = np.arange(right_i, N)
        
        bg_idx = np.concatenate([left_idx, right_idx])
        w_bg = w[bg_idx]
        S_bg = S[bg_idx]
        
        # ---- 2) Fit linear background to the tails ----
        p0_bg = [np.median(S_bg), 0.0]
        popt_bg, pcov_bg = curve_fit(linear_bg, w_bg, S_bg, p0=p0_bg)
        b0_fit, b1_fit = popt_bg
        bg_fit_full = linear_bg(w, b0_fit, b1_fit)
        
        # ---- 3) Subtract background ----
        S_corr = S - bg_fit_full
        
        # ---- 4) Fit triple Lorentzian to background-subtracted data ----
        fit_window = 0.6 * (w.max() - w.min())
        w_span = fit_window / 2.0
        fit_mask = np.abs(w) <= w_span
        
        # Initial guesses for three Lorentzians
        peak_height = np.max(S_corr)
        A0 = peak_height * np.pi * 0.05
        gamma0 = (w.max() - w.min()) * 0.05
        w_sep = w_span / 2.0
        
        # Initial parameters: [A1, gamma1, w01, A2, gamma2, w02, A3, gamma3, w03]
        p0_triple = [
            A0, gamma0, -w_sep,  # First Lorentzian (left)
            A0, gamma0, 0,       # Second Lorentzian (center)
            A0, gamma0, w_sep    # Third Lorentzian (right)
        ]
        
        bounds_lower = [1e-10, 1e-10, -np.inf, 
                        1e-10, 1e-10, -np.inf, 
                        1e-10, 1e-10, -np.inf]
        bounds_upper = [np.inf, np.inf, np.inf, 
                        np.inf, np.inf, np.inf, 
                        np.inf, np.inf, np.inf]
        
        popt_triple, pcov_triple = curve_fit(
            triple_lorentz, 
            w[fit_mask], 
            S_corr[fit_mask], 
            p0=p0_triple,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        
        A1_fit, gamma1_fit, w01_fit, A2_fit, gamma2_fit, w02_fit, A3_fit, gamma3_fit, w03_fit = popt_triple
        
        # Calculate FWHM
        FWHM1 = 2.0 * gamma1_fit
        FWHM2 = 2.0 * gamma2_fit
        FWHM3 = 2.0 * gamma3_fit
        
        # Estimate uncertainties
        perr_triple = np.sqrt(np.diag(pcov_triple)) if pcov_triple is not None else [np.nan]*9
        A1_err, gamma1_err, w01_err, A2_err, gamma2_err, w02_err, A3_err, gamma3_err, w03_err = perr_triple
        FWHM1_err = 2.0 * gamma1_err
        FWHM2_err = 2.0 * gamma2_err
        FWHM3_err = 2.0 * gamma3_err
        
        # Store results
        FWHM1_array[i] = FWHM1
        FWHM2_array[i] = FWHM2
        FWHM3_array[i] = FWHM3
        FWHM1_err_array[i] = FWHM1_err
        FWHM2_err_array[i] = FWHM2_err
        FWHM3_err_array[i] = FWHM3_err
        w01_array[i] = w01_fit
        w02_array[i] = w02_fit
        w03_array[i] = w03_fit
        A1_array[i] = A1_fit
        A2_array[i] = A2_fit
        A3_array[i] = A3_fit
        
        print(f"  Lorentzian 1: ω₀={w01_fit:.4g}, FWHM={FWHM1:.4g} ± {FWHM1_err:.3g}")
        print(f"  Lorentzian 2: ω₀={w02_fit:.4g}, FWHM={FWHM2:.4g} ± {FWHM2_err:.3g}")
        print(f"  Lorentzian 3: ω₀={w03_fit:.4g}, FWHM={FWHM3:.4g} ± {FWHM3_err:.3g}")
        
    except Exception as e:
        print(f"  *** Fit failed for spectrum {i}: {e}")
        FWHM1_array[i] = np.nan
        FWHM2_array[i] = np.nan
        FWHM3_array[i] = np.nan
        FWHM1_err_array[i] = np.nan
        FWHM2_err_array[i] = np.nan
        FWHM3_err_array[i] = np.nan
        w01_array[i] = np.nan
        w02_array[i] = np.nan
        w03_array[i] = np.nan
        A1_array[i] = np.nan
        A2_array[i] = np.nan
        A3_array[i] = np.nan

# -------------------------
# Summary plot
# -------------------------

### Lorentz fit

In [ ]:
"""
QENS S(Q,w) multi-Lorentzian fitting
======================================
Model: S(Q,w) = sum_n [ (A_n/pi) * (gamma_n / ((w - w0_n)^2 + gamma_n^2)) ]

- Fits 1-, 2-, and 3-component Lorentzian models for every Q value
- Removes elastic peak before fitting
- Selects best model via AIC (Akaike Information Criterion)
- Plots FWHM (meV) vs Q with error bars for all components of the best model
- Also plots AIC comparison and amplitudes

Adjust the CONFIGURATION section to match your data.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ── CONFIGURATION ────────────────────────────────────────────────────────────

TIME_UNIT = "ps"

# Q values in Å⁻¹  –  replace with your actual q array
# q_ls = q[25:190:5] / 10

# ── DATA ─────────────────────────────────────────────────────────────────────
# Expected shapes:
#   omega  : 1-D array, length W  (in meV)
#   data   : 2-D array, shape (N_Q, W)  – one row per Q value
#
# From your session:
#   omega = omega                          # shape (W,)
#   data  = H_disf[25:190:5, :]           # shape (N_Q, W)
#   q_ls  = q[25:190:5] / 10             # shape (N_Q,) in Å⁻¹

# ── MODELS ───────────────────────────────────────────────────────────────────

def lorentz(w, A, gamma, w0):
    return (A / np.pi) * (gamma / ((w - w0)**2 + gamma**2))

def model1(w, A1, gamma1, w01):
    return lorentz(w, A1, gamma1, w01)

def model2(w, A1, gamma1, w01, A2, gamma2, w02):
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02)

def model3(w, A1, gamma1, w01, A2, gamma2, w02, A3, gamma3, w03):
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02) + lorentz(w, A3, gamma3, w03)

MODELS = {
    1: (model1, ["A1", "gamma1", "w01"]),
    2: (model2, ["A1", "gamma1", "w01", "A2", "gamma2", "w02"]),
    3: (model3, ["A1", "gamma1", "w01", "A2", "gamma2", "w02", "A3", "gamma3", "w03"]),
}

# ── ELASTIC PEAK REMOVAL ─────────────────────────────────────────────────────

def remove_elastic_peak(w, S):
    S_clean = S.copy()
    i0 = np.argmax(S_clean)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_clean))
    good_idx = np.r_[0:lo, hi:len(S_clean)]
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S[good_idx])
    return S_clean

# ── INITIAL GUESSES & BOUNDS ─────────────────────────────────────────────────

def initial_guess(n_comp, w, sqw):
    peak_height = np.max(sqw)
    w_span = (w.max() - w.min())
    A0     = peak_height * np.pi * 0.05
    g_narrow = 0.001 #w_span * 0.02
    g_mid    = 0.01 #w_span * 0.05
    g_broad  = 0.1 #w_span * 0.15
    w_sep    = w_span * 0.15

    if n_comp == 1:
        return [A0, g_mid, 0.0]
    elif n_comp == 2:
        return [A0, g_narrow, -w_sep,
                A0, g_broad,   w_sep]
    else:
        return [A0, g_narrow, -w_sep,
                A0, g_mid,     0.0,
                A0, g_broad,   w_sep]

def bounds(n_comp):
    lo, hi = [], []
    for _ in range(n_comp):
        lo += [1e-10, 1e-10, -np.inf]   # A, gamma, w0
        hi += [np.inf, np.inf, np.inf]
    return lo, hi

# ── FIT WINDOW ───────────────────────────────────────────────────────────────

FIT_WINDOW_FRACTION = 1  # fraction of full omega range to fit

def get_fit_mask(w):
    w_span = FIT_WINDOW_FRACTION * (w.max() - w.min()) / 2.0
    return np.abs(w) <= w_span

# ── AIC ──────────────────────────────────────────────────────────────────────

def aic(residuals, n_params):
    n   = len(residuals)
    rss = np.sum(residuals**2)
    return n * np.log(rss / n) + 2 * n_params

# ── FIT ONE Q ROW ────────────────────────────────────────────────────────────

def fit_one_q(omega, sqw):
    sqw   = remove_elastic_peak(omega, sqw)
    mask  = get_fit_mask(omega)
    w_fit = omega[mask]
    s_fit = sqw[mask]

    results = {}
    for n_comp, (func, param_names) in MODELS.items():
        p0     = initial_guess(n_comp, w_fit, s_fit)
        lo, hi = bounds(n_comp)
        try:
            popt, pcov = curve_fit(
                func, w_fit, s_fit,
                p0=p0, bounds=(lo, hi),
                maxfev=50000, ftol=1e-10, xtol=1e-10,
            )
            perr    = np.sqrt(np.diag(pcov))
            res     = s_fit - func(w_fit, *popt)
            aic_val = aic(res, len(popt))
            results[n_comp] = dict(popt=popt, perr=perr, aic=aic_val,
                                   param_names=param_names, func=func)
        except (RuntimeError, ValueError) as e:
            print(f"    [{n_comp}-comp fit failed: {e}]")
            results[n_comp] = None

    valid  = {k: v for k, v in results.items() if v is not None}
    if not valid:
        return None
    best_n = min(valid, key=lambda k: valid[k]["aic"])
    return dict(best_n=best_n, models=results)

# ── MAIN ─────────────────────────────────────────────────────────────────────

def run_fitting(omega, data, q_vals):
    """
    omega  : 1-D array (W,)    energy transfer in meV
    data   : 2-D array (N_Q, W)
    q_vals : 1-D array (N_Q,)  in Å⁻¹
    """
    N_Q = data.shape[0]
    assert len(q_vals) == N_Q

    fit_results = []
    for i in range(N_Q):
        r = fit_one_q(omega, data[i, :])
        fit_results.append(r)
        n = r["best_n"] if r else "failed"
        print(f"  Q={q_vals[i]:.3f} Å⁻¹  →  best model: {n} component(s)")

    return fit_results

# ── PLOT RESULTS ─────────────────────────────────────────────────────────────

def plot_results(fit_results, q_vals):
    """
    Three-panel figure:
      Left   : FWHM (meV) vs Q  for all components of best model
      Middle : amplitudes vs Q
      Right  : AIC comparison across 1-, 2-, 3-component models
    """
    fig = plt.figure(figsize=(14, 5))
    ax_g   = fig.add_subplot(1, 3, 1)
    ax_a   = fig.add_subplot(1, 3, 2)
    ax_aic = fig.add_subplot(1, 3, 3)

    colors  = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    markers = ["o", "s", "^"]

    aic_1, aic_2, aic_3 = [], [], []

    for i, (r, qi) in enumerate(zip(fit_results, q_vals)):
        if r is None:
            for lst in [aic_1, aic_2, aic_3]:
                lst.append(np.nan)
            continue

        best_n = r["best_n"]
        best   = r["models"][best_n]
        popt   = best["popt"]
        perr   = best["perr"]

        # parameters are ordered (A, gamma, w0) per component
        for comp in range(best_n):
            A_val    = popt[3 * comp]
            g_val    = popt[3 * comp + 1]   # half-width (gamma)
            A_err    = perr[3 * comp]
            g_err    = perr[3 * comp + 1]

            FWHM     = 2 * g_val             # meV  (already in meV)
            FWHM_err = 2 * g_err

            ax_g.errorbar(qi, FWHM, yerr=FWHM_err,
                          fmt=markers[comp], color=colors[comp],
                          capsize=3, markersize=5,
                          label=f"FWHM {comp+1}" if i == 0 else "_")
            ax_a.errorbar(qi, A_val, yerr=A_err,
                          fmt=markers[comp], color=colors[comp],
                          capsize=3, markersize=5,
                          label=f"A{comp+1}" if i == 0 else "_")

        # AIC per model
        for n_c, aic_list in zip([1, 2, 3], [aic_1, aic_2, aic_3]):
            m       = r["models"].get(n_c)
            aic_val = m["aic"] if m is not None else np.nan
            aic_list.append(aic_val)

    q_arr = np.array(q_vals)
    for n_c, aic_list, c, lbl in zip(
            [1, 2, 3], [aic_1, aic_2, aic_3], colors, ["1-comp", "2-comp", "3-comp"]):
        ax_aic.plot(q_arr, aic_list, marker="o", color=c, label=lbl, markersize=4)

    ax_g.set_xlabel("Q (Å⁻¹)")
    ax_g.set_ylabel("FWHM (meV)")
    ax_g.set_title("Linewidths vs Q")
    ax_g.set_ylim(0,0.1)
    ax_g.legend()

    ax_a.set_xlabel("Q (Å⁻¹)")
    ax_a.set_ylabel("Amplitude")
    ax_a.set_title("Amplitudes vs Q")
    ax_a.legend()

    ax_aic.set_xlabel("Q (Å⁻¹)")
    ax_aic.set_ylabel("AIC")
    ax_aic.set_title("AIC model comparison")
    ax_aic.legend()

    fig.tight_layout()
    plt.savefig("sqw_fit_results.png", dpi=150)
    plt.show()
    print("Figure saved → sqw_fit_results.png")


def plot_fits(omega, data, fit_results, q_vals, n_cols=4):
    """Optional: plot data + best-fit curves for every Q."""
    N_Q    = len(q_vals)
    n_rows = int(np.ceil(N_Q / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(4 * n_cols, 3 * n_rows),
                             sharey=True, sharex=True)
    axes = axes.flatten()
    mask = get_fit_mask(omega)

    colors_comp = ["green", "purple", "orange"]

    for i, (r, qi) in enumerate(zip(fit_results, q_vals)):
        ax  = axes[i]
        sqw = remove_elastic_peak(omega, data[i, :])
        ax.semilogy(omega, sqw, "k.", ms=3, label="data")

        if r is not None:
            best  = r["models"][r["best_n"]]
            y_fit = best["func"](omega, *best["popt"])
            ax.semilogy(omega, y_fit, "r-", lw=1.5,
                        label=f"{r['best_n']}-comp")

            # individual components
            popt   = best["popt"]
            best_n = r["best_n"]
            for comp in range(best_n):
                y_comp = lorentz(omega,
                                 popt[3*comp],
                                 popt[3*comp+1],
                                 popt[3*comp+2])
                ax.semilogy(omega, y_comp, "--", lw=1,
                            color=colors_comp[comp],
                            label=f"L{comp+1}")

        ax.set_title(f"Q = {qi:.3f} Å⁻¹", fontsize=9)
        ax.set_xlabel("ω (meV)", fontsize=8)
        ax.set_ylabel("S(Q,ω)", fontsize=8)
        ax.legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.tight_layout()
    plt.savefig("sqw_fit_curves.png", dpi=150)
    plt.show()
    print("Figure saved → sqw_fit_curves.png")

# ── ENTRY POINT ───────────────────────────────────────────────────────────────

omega  = np.linspace(-0.51675,0.51675,2499)                      # your energy axis (meV)
data   = H_disf[30:190:5, :]        # shape (N_Q, W)
q_ls   = q[30:190:5] / 10          # Å⁻¹

fit_results = run_fitting(omega, data, q_ls)
plot_results(fit_results, q_ls)
plot_fits(omega, data, fit_results, q_ls)   # optional per-Q panel

In [ ]:
%matplotlib widget
fig, ax1 = plt.subplots(1, 1, figsize=(singlecol*2, singlecol*2))

# Plot FWHM vs spectrum index
ax1.errorbar(q[range(n_spectra)]/10, FWHM1_array, yerr=FWHM1_err_array, 
             fmt='o-', label='Lorentzian 1', capsize=3)
ax1.errorbar(q[range(n_spectra)]/10, FWHM2_array, yerr=FWHM2_err_array, 
             fmt='s-', label='Lorentzian 2', capsize=3)
ax1.errorbar(q[range(n_spectra)]/10, FWHM3_array, yerr=FWHM3_err_array, 
             fmt='^-', label='Lorentzian 3', capsize=3)
ax1.set_xlabel('q (1/A)')
ax1.set_ylabel('FWHM')
ax1.set_ylim(0,0.75)
ax1.set_title('FWHM of Three Lorentzians vs Spectrum Index')
ax1.legend()
ax1.grid(True, alpha=0.3)


# -------------------------
# Print summary
# -------------------------
print("\n" + "="*80)
print("SUMMARY OF ALL FITS")
print("="*80)
print(f"{'Index':<8} {'FWHM1':<15} {'FWHM2':<15} {'FWHM3':<15} {'ω₀1':<10} {'ω₀2':<10} {'ω₀3':<10}")
print("-"*80)
for i in range(n_spectra):
    print(f"{i:<8} {FWHM1_array[i]:>7.4g}±{FWHM1_err_array[i]:<5.2g} "
          f"{FWHM2_array[i]:>7.4g}±{FWHM2_err_array[i]:<5.2g} "
          f"{FWHM3_array[i]:>7.4g}±{FWHM3_err_array[i]:<5.2g} "
          f"{w01_array[i]:>9.4g} {w02_array[i]:>9.4g} {w03_array[i]:>9.4g}")
print("="*80)

In [ ]:
# The arrays are now available for further analysis:
# FWHM1_array, FWHM2_array, FWHM3_array contain the FWHM values
# FWHM1_err_array, FWHM2_err_array, FWHM3_err_array contain the uncertainties
# w01_array, w02_array, w03_array contain the peak centers
# A1_array, A2_array, A3_array contain the amplitudes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# -------------------------
# Initialize arrays to store results
# -------------------------
n_spectra = q.shape[0]
FWHM1_array = np.zeros(n_spectra)
FWHM2_array = np.zeros(n_spectra)
FWHM1_err_array = np.zeros(n_spectra)
FWHM2_err_array = np.zeros(n_spectra)

# Store other fit parameters if needed
w01_array = np.zeros(n_spectra)
w02_array = np.zeros(n_spectra)

# -------------------------
# Parameters you can change
# -------------------------
tail_fraction = 0.15   # fraction of the spectrum at each end used for BG estimation
use_indices = None     # alternatively, specify tuple (left_i, right_i) indices for background fit

# ---- Helper models ----
def linear_bg(w, b0, b1):
    return b0 + b1 * w

def lorentz(w, A, gamma, w0=0):
    """Lorentzian centered at w0: (A/pi) * gamma / ((w-w0)^2 + gamma^2)"""
    return (A/np.pi) * (gamma / ((w - w0)**2 + gamma**2))

def double_lorentz(w, A1, gamma1, w01, A2, gamma2, w02):
    """Sum of two Lorentzians"""
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02)
    
def triple_lorentz(w, A1, gamma1, w01, A2, gamma2, w02, A3, gamma3, w03):
    """Sum of two Lorentzians"""
    return lorentz(w, A1, gamma1, w01) + lorentz(w, A2, gamma2, w02) + lorentz(w, A3, gamma3, w03)
# -------------------------
# Main fitting loop
# -------------------------
for i in range(n_spectra):
    print(f"\n{'='*60}")
    print(f"Fitting spectrum {i+1}/{n_spectra}")
    print(f"{'='*60}")
    
    try:
        # Get data for this spectrum

        S_raw = total_disf[i].copy()
        w = omega
        
        # Remove elastic peak
        i0 = np.argmax(S_raw)
        lo = max(i0 - 3, 0)
        hi = min(i0 + 4, len(S_raw))
        
        good_idx = np.r_[0:lo, hi:len(S_raw)]
        
        S_clean = S_raw.copy()
        S_clean[lo:hi] = np.interp(
            np.arange(lo, hi),
            good_idx,
            S_raw[good_idx]
        )
        
        S = S_clean

        #S = Na_disf[i]
        # ---- 1) Select tail points for background estimation ----
        N = len(w)
        if use_indices is None:
            n_tail = max(2, int(np.round(tail_fraction * N)))
            left_idx = np.arange(0, n_tail)
            right_idx = np.arange(N - n_tail, N)
        else:
            left_i, right_i = use_indices
            left_idx = np.arange(0, left_i)
            right_idx = np.arange(right_i, N)
        
        bg_idx = np.concatenate([left_idx, right_idx])
        w_bg = w[bg_idx]
        S_bg = S[bg_idx]
        
        # ---- 2) Fit linear background to the tails ----
        p0_bg = [np.median(S_bg), 0.0]
        popt_bg, pcov_bg = curve_fit(linear_bg, w_bg, S_bg, p0=p0_bg)
        b0_fit, b1_fit = popt_bg
        bg_fit_full = linear_bg(w, b0_fit, b1_fit)
        
        # ---- 3) Subtract background ----
        S_corr = S - bg_fit_full
        
        # ---- 4) Fit double Lorentzian to background-subtracted data ----
        fit_window = 0.6 * (w.max() - w.min())
        w_span = fit_window / 2.0
        fit_mask = np.abs(w) <= w_span
        
        # Initial guesses
        peak_height = np.max(S_corr)
        A0 = peak_height * np.pi * 0.05
        gamma0 = (w.max() - w.min()) * 0.05
        w_sep = w_span / 2.0
        
        p0_double = [
            A0, gamma0, -w_sep,  # First Lorentzian (left)
            A0, gamma0, w_sep    # Second Lorentzian (right)
        ]
        
        bounds_lower = [1e-10, 1e-10, -np.inf, 1e-10, 1e-10, -np.inf, 1e-10, 1e-10, -np.inf]
        bounds_upper = [np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf]
        
        popt_double, pcov_double = curve_fit(
            triple_lorentz, 
            w[fit_mask], 
            S_corr[fit_mask], 
            p0=p0_double,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        
        A1_fit, gamma1_fit, w01_fit, A2_fit, gamma2_fit, w02_fit, A3_fit, gamma3_fit, w03_fit = popt_double
        
        # Calculate FWHM
        FWHM1 = 2.0 * gamma1_fit
        FWHM2 = 2.0 * gamma2_fit
        
        # Estimate uncertainties
        perr_double = np.sqrt(np.diag(pcov_double)) if pcov_double is not None else [np.nan]*6
        A1_err, gamma1_err, w01_err, A2_err, gamma2_err, w02_err = perr_double
        FWHM1_err = 2.0 * gamma1_err
        FWHM2_err = 2.0 * gamma2_err
        
        # Store results
        FWHM1_array[i] = FWHM1
        FWHM2_array[i] = FWHM2
        FWHM1_err_array[i] = FWHM1_err
        FWHM2_err_array[i] = FWHM2_err
        w01_array[i] = w01_fit
        w02_array[i] = w02_fit
        
        print(f"  Lorentzian 1: ω₀={w01_fit:.4g}, FWHM={FWHM1:.4g} ± {FWHM1_err:.3g}")
        print(f"  Lorentzian 2: ω₀={w02_fit:.4g}, FWHM={FWHM2:.4g} ± {FWHM2_err:.3g}")
        
    except Exception as e:
        print(f"  *** Fit failed for spectrum {i}: {e}")
        FWHM1_array[i] = np.nan
        FWHM2_array[i] = np.nan
        FWHM1_err_array[i] = np.nan
        FWHM2_err_array[i] = np.nan
        w01_array[i] = np.nan
        w02_array[i] = np.nan

# -------------------------
# Summary plot
# -------------------------
fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2))

# Plot FWHM vs spectrum index
ax1 = axes[0]
ax1.errorbar(range(n_spectra), FWHM1_array, yerr=FWHM1_err_array, 
             fmt='o-', label='Lorentzian 1', capsize=3)
ax1.errorbar(range(n_spectra), FWHM2_array, yerr=FWHM2_err_array, 
             fmt='s-', label='Lorentzian 2', capsize=3)
ax1.set_xlabel('Spectrum index')
ax1.set_ylabel('FWHM')
ax1.set_title('FWHM of both Lorentzians vs Spectrum Index')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot peak centers
ax2 = axes[1]
ax2.plot(range(n_spectra), w01_array, 'o-', label='Lorentzian 1 center')
ax2.plot(range(n_spectra), w02_array, 's-', label='Lorentzian 2 center')
ax2.set_xlabel('Spectrum index')
ax2.set_ylabel('Peak center (ω₀)')
ax2.set_title('Peak Centers vs Spectrum Index')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

# -------------------------
# Print summary
# -------------------------
print("\n" + "="*60)
print("SUMMARY OF ALL FITS")
print("="*60)
print(f"{'Index':<8} {'FWHM1':<15} {'FWHM2':<15} {'ω₀1':<12} {'ω₀2':<12}")
print("-"*60)
for i in range(n_spectra):
    print(f"{i:<8} {FWHM1_array[i]:>8.4g}±{FWHM1_err_array[i]:<5.2g} "
          f"{FWHM2_array[i]:>8.4g}±{FWHM2_err_array[i]:<5.2g} "
          f"{w01_array[i]:>11.4g} {w02_array[i]:>11.4g}")
print("="*60)

# The arrays are now available for further analysis:
# FWHM1_array, FWHM2_array contain the FWHM values
# FWHM1_err_array, FWHM2_err_array contain the uncertainties

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2))

# Plot FWHM vs spectrum index
ax1 = axes[0]
ax1.errorbar(range(n_spectra), FWHM1_array, yerr=FWHM1_err_array, 
             fmt='o-', label='Lorentzian 1', capsize=3)
ax1.errorbar(range(n_spectra), FWHM2_array, yerr=FWHM2_err_array, 
             fmt='s-', label='Lorentzian 2', capsize=3)
ax1.set_xlabel('Spectrum index')
ax1.set_ylabel('FWHM')
#ax1.set_ylim(0,0.1)
ax1.set_title('FWHM of both Lorentzians vs Spectrum Index')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot peak centers
ax2 = axes[1]
ax2.plot(range(n_spectra), w01_array, 'o-', label='Lorentzian 1 center')
ax2.plot(range(n_spectra), w02_array, 's-', label='Lorentzian 2 center')
ax2.set_xlabel('Spectrum index')
ax2.set_ylabel('Peak center (ω₀)')
ax2.set_title('Peak Centers vs Spectrum Index')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', lw=0.8, ls='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Na FMHW

In [ ]:
######## Na FWHM ####

import pandas as pd

data = [
    {"Index": 0, "FWHM1": 0.005941, "err1": 0.00007, "FWHM2": 0.005941, "err2": 0.00007, "w01": 0.001946, "w02": -0.001946},
    {"Index": 1, "FWHM1": 1.324, "err1": 9.4, "FWHM2": 0.008571, "err2": 0.000047, "w01": 0.0005113, "w02": -6.485e-10},
    {"Index": 2, "FWHM1": 0.8887, "err1": 2.9, "FWHM2": 0.008728, "err2": 0.000046, "w01": 0.0003397, "w02": 4.799e-10},
    {"Index": 3, "FWHM1": 0.5642, "err1": 0.79, "FWHM2": 0.008672, "err2": 0.000047, "w01": 9.077e-08, "w02": -5.518e-10},
    {"Index": 4, "FWHM1": 0.4805, "err1": 0.5, "FWHM2": 0.008819, "err2": 0.000047, "w01": -1.112e-06, "w02": 5.27e-10},
    {"Index": 5, "FWHM1": 0.4388, "err1": 0.38, "FWHM2": 0.009088, "err2": 0.000046, "w01": -3.488e-07, "w02": -4.775e-10},
    {"Index": 6, "FWHM1": 0.3651, "err1": 0.24, "FWHM2": 0.009096, "err2": 0.000047, "w01": 1.185e-07, "w02": -4.635e-10},
    {"Index": 7, "FWHM1": 0.2822, "err1": 0.13, "FWHM2": 0.008963, "err2": 0.000048, "w01": -9.801e-08, "w02": 4.469e-10},
    {"Index": 8, "FWHM1": 0.2722, "err1": 0.11, "FWHM2": 0.009241, "err2": 0.000047, "w01": -1.001e-07, "w02": 4.119e-10},
    {"Index": 9, "FWHM1": 0.2532, "err1": 0.09, "FWHM2": 0.009424, "err2": 0.000046, "w01": 7.375e-08, "w02": -3.813e-10},
    {"Index": 10, "FWHM1": 0.1179, "err1": 0.038, "FWHM2": 0.01238, "err2": 0.000057, "w01": -0.05777, "w02": 0.0004068},
    {"Index": 11, "FWHM1": 0.2317, "err1": 0.066, "FWHM2": 0.009978, "err2": 0.000045, "w01": 4.142e-08, "w02": -3.337e-10},
    {"Index": 12, "FWHM1": 0.1866, "err1": 0.042, "FWHM2": 0.009811, "err2": 0.000049, "w01": -2.801e-08, "w02": 3.345e-10},
    {"Index": 13, "FWHM1": 0.1888, "err1": 0.04, "FWHM2": 0.01012, "err2": 0.000046, "w01": 3.101e-08, "w02": -2.982e-10},
    {"Index": 14, "FWHM1": 0.2018, "err1": 0.042, "FWHM2": 0.01067, "err2": 0.000045, "w01": -3.516e-08, "w02": 2.774e-10},
    {"Index": 15, "FWHM1": 0.1725, "err1": 0.029, "FWHM2": 0.0104, "err2": 0.000046, "w01": -2.135e-08, "w02": 2.755e-10},
    {"Index": 16, "FWHM1": 0.1595, "err1": 0.024, "FWHM2": 0.01051, "err2": 0.000047, "w01": 1.821e-08, "w02": -2.6e-10},
    {"Index": 17, "FWHM1": 0.1561, "err1": 0.022, "FWHM2": 0.01081, "err2": 0.000046, "w01": 1.536e-08, "w02": -2.306e-10},
    {"Index": 18, "FWHM1": 0.1556, "err1": 0.02, "FWHM2": 0.01101, "err2": 0.000044, "w01": -1.538e-08, "w02": 2.269e-10},
    {"Index": 19, "FWHM1": 0.1115, "err1": 0.011, "FWHM2": 0.01043, "err2": 0.000051, "w01": -9.307e-09, "w02": 2.164e-10},
    {"Index": 20, "FWHM1": 0.1358, "err1": 0.014, "FWHM2": 0.01122, "err2": 0.000045, "w01": 1.076e-08, "w02": -1.993e-10},
    {"Index": 21, "FWHM1": 0.1157, "err1": 0.011, "FWHM2": 0.01118, "err2": 0.00005, "w01": 8.16e-09, "w02": -1.806e-10},
    {"Index": 22, "FWHM1": 0.102, "err1": 0.008, "FWHM2": 0.01103, "err2": 0.000051, "w01": -6.266e-09, "w02": 1.744e-10},
    {"Index": 23, "FWHM1": 0.1254, "err1": 0.011, "FWHM2": 0.01218, "err2": 0.000046, "w01": -7.148e-09, "w02": 1.509e-10},
    {"Index": 24, "FWHM1": 0.1147, "err1": 0.0086, "FWHM2": 0.01216, "err2": 0.000046, "w01": 5.765e-09, "w02": -1.383e-10},
    {"Index": 25, "FWHM1": 0.1184, "err1": 0.0086, "FWHM2": 0.01235, "err2": 0.000047, "w01": -8.985e-09, "w02": 1.413e-10},
    {"Index": 26, "FWHM1": 0.08507, "err1": 0.0047, "FWHM2": 0.01134, "err2": 0.000053, "w01": -3.124e-09, "w02": 1.228e-10},
    {"Index": 27, "FWHM1": 0.09197, "err1": 0.0051, "FWHM2": 0.01261, "err2": 0.000048, "w01": -0.002891, "w02": 0.0001222},
    {"Index": 28, "FWHM1": 0.09085, "err1": 0.0048, "FWHM2": 0.01283, "err2": 0.000049, "w01": 0.002837, "w02": -0.0001261},
    {"Index": 29, "FWHM1": 0.07733, "err1": 0.0036, "FWHM2": 0.01167, "err2": 0.000056, "w01": 2.289e-09, "w02": -1.13e-10},
    {"Index": 30, "FWHM1": 0.1041, "err1": 0.0053, "FWHM2": 0.01313, "err2": 0.000047, "w01": -3.207e-09, "w02": 1.061e-10},
    {"Index": 31, "FWHM1": 0.08696, "err1": 0.0039, "FWHM2": 0.01295, "err2": 0.000051, "w01": -1.934e-09, "w02": 8.262e-11},
    {"Index": 32, "FWHM1": 0.1045, "err1": 0.0048, "FWHM2": 0.01385, "err2": 0.000046, "w01": -2.332e-09, "w02": 8.279e-11},
    {"Index": 33, "FWHM1": 0.07894, "err1": 0.0031, "FWHM2": 0.01259, "err2": 0.000055, "w01": 1.533e-09, "w02": -8.466e-11},
    {"Index": 34, "FWHM1": 0.06719, "err1": 0.0023, "FWHM2": 0.01202, "err2": 0.000062, "w01": 1.138e-09, "w02": -8.49e-11},
    {"Index": 35, "FWHM1": 0.09925, "err1": 0.0038, "FWHM2": 0.01432, "err2": 0.000047, "w01": 1.585e-09, "w02": -6.541e-11},
    {"Index": 36, "FWHM1": 0.08783, "err1": 0.0031, "FWHM2": 0.01426, "err2": 0.000051, "w01": -1.171e-09, "w02": 5.874e-11},
    {"Index": 37, "FWHM1": 0.07444, "err1": 0.0024, "FWHM2": 0.01343, "err2": 0.000056, "w01": 8.185e-10, "w02": -5.603e-11},
    {"Index": 38, "FWHM1": 0.08843, "err1": 0.0027, "FWHM2": 0.01454, "err2": 0.000048, "w01": 7.422e-10, "w02": -4.215e-11},
    {"Index": 39, "FWHM1": 0.06698, "err1": 0.0018, "FWHM2": 0.01306, "err2": 0.00006, "w01": 3.886e-10, "w02": -3.935e-11},
    {"Index": 40, "FWHM1": 0.07365, "err1": 0.0021, "FWHM2": 0.01379, "err2": 0.000058, "w01": -6.445e-10, "w02": 5.176e-11},
    {"Index": 41, "FWHM1": 0.07034, "err1": 0.0018, "FWHM2": 0.01382, "err2": 0.000058, "w01": -2.451e-10, "w02": 2.817e-11},
    {"Index": 42, "FWHM1": 0.08324, "err1": 0.0022, "FWHM2": 0.01534, "err2": 0.00005, "w01": 4.078e-10, "w02": -3.068e-11},
    {"Index": 43, "FWHM1": 0.08466, "err1": 0.0023, "FWHM2": 0.01553, "err2": 0.000052, "w01": -5.02e-10, "w02": 3.597e-11},
    {"Index": 44, "FWHM1": 0.06106, "err1": 0.0013, "FWHM2": 0.01333, "err2": 0.000067, "w01": 1.753e-10, "w02": -3.071e-11},
    {"Index": 45, "FWHM1": 0.0615, "err1": 0.0013, "FWHM2": 0.01348, "err2": 0.000067, "w01": -6.569e-11, "w02": 2.04e-11},
    {"Index": 46, "FWHM1": 0.08997, "err1": 0.0021, "FWHM2": 0.01673, "err2": 0.000049, "w01": -1.963e-10, "w02": 1.869e-11},
    {"Index": 47, "FWHM1": 0.08237, "err1": 0.0018, "FWHM2": 0.01584, "err2": 0.000052, "w01": 3.292e-11, "w02": -1.097e-11},
    {"Index": 48, "FWHM1": 0.07972, "err1": 0.0017, "FWHM2": 0.01555, "err2": 0.000054, "w01": 6.433e-11, "w02": -1.39e-11},
    {"Index": 49, "FWHM1": 0.07686, "err1": 0.0016, "FWHM2": 0.01587, "err2": 0.000055, "w01": -5.81e-11, "w02": 1.49e-11},
    {"Index": 50, "FWHM1": 0.07353, "err1": 0.0015, "FWHM2": 0.01531, "err2": 0.00006, "w01": 7.728e-11, "w02": -1.771e-11},
    {"Index": 51, "FWHM1": 0.09589, "err1": 0.0018, "FWHM2": 0.0174, "err2": 0.000044, "w01": 8.666e-11, "w02": -1.185e-11},
    {"Index": 52, "FWHM1": 0.06843, "err1": 0.0013, "FWHM2": 0.01502, "err2": 0.000065, "w01": -2.015e-10, "w02": 8.012e-12},
    {"Index": 53, "FWHM1": 0.06112, "err1": 0.00097, "FWHM2": 0.01452, "err2": 0.000072, "w01": 1.67e-10, "w02": -6.744e-12},
    {"Index": 54, "FWHM1": 0.07806, "err1": 0.0014, "FWHM2": 0.01632, "err2": 0.000058, "w01": 2.015e-10, "w02": -5.502e-12},
    {"Index": 55, "FWHM1": 0.05953, "err1": 0.0009, "FWHM2": 0.01431, "err2": 0.000075, "w01": -1.919e-10, "w02": 1.191e-11},
    {"Index": 56, "FWHM1": 0.06543, "err1": 0.001, "FWHM2": 0.0153, "err2": 0.000068, "w01": 3.647e-11, "w02": 1.503e-11},
    {"Index": 57, "FWHM1": 0.06477, "err1": 0.00095, "FWHM2": 0.01542, "err2": 0.000068, "w01": -7.73e-11, "w02": -7.864e-12},
    {"Index": 58, "FWHM1": 0.09075, "err1": 0.0016, "FWHM2": 0.01831, "err2": 0.000051, "w01": -1.528e-10, "w02": 1.302e-12},
    {"Index": 59, "FWHM1": 0.06557, "err1": 0.00093, "FWHM2": 0.01566, "err2": 0.000069, "w01": -4.676e-11, "w02": -1.292e-11},
    {"Index": 60, "FWHM1": 0.08813, "err1": 0.0014, "FWHM2": 0.018, "err2": 0.000051, "w01": -1.461e-10, "w02": 6.418e-13},
    {"Index": 61, "FWHM1": 0.06375, "err1": 0.00089, "FWHM2": 0.01539, "err2": 0.000073, "w01": -2.531e-11, "w02": -1.814e-11},
    {"Index": 62, "FWHM1": 0.1027, "err1": 0.0017, "FWHM2": 0.01907, "err2": 0.000047, "w01": -2.465e-11, "w02": -5.962e-12},
    {"Index": 63, "FWHM1": 0.1029, "err1": 0.0018, "FWHM2": 0.01914, "err2": 0.00005, "w01": 9.771e-12, "w02": 6.849e-12},
    {"Index": 64, "FWHM1": 0.06837, "err1": 0.00095, "FWHM2": 0.01636, "err2": 0.000073, "w01": 4.157e-10, "w02": -4.038e-11},
    {"Index": 65, "FWHM1": 0.06363, "err1": 0.0009, "FWHM2": 0.01529, "err2": 0.000076, "w01": 4.027e-10, "w02": -4.274e-11},
    {"Index": 66, "FWHM1": 0.07321, "err1": 0.001, "FWHM2": 0.01693, "err2": 0.000067, "w01": -5.463e-10, "w02": 5.173e-11},
    {"Index": 67, "FWHM1": 0.07054, "err1": 0.00094, "FWHM2": 0.01641, "err2": 0.000068, "w01": 5.045e-10, "w02": -5.046e-11},
    {"Index": 68, "FWHM1": 0.09391, "err1": 0.0013, "FWHM2": 0.01907, "err2": 0.000048, "w01": 8.74e-11, "w02": 4.453e-12},
    {"Index": 69, "FWHM1": 0.07281, "err1": 0.00098, "FWHM2": 0.01705, "err2": 0.000069, "w01": 5.145e-10, "w02": -5.02e-11},
    {"Index": 70, "FWHM1": 0.07804, "err1": 0.0011, "FWHM2": 0.01773, "err2": 0.000068, "w01": -6.061e-10, "w02": 5.786e-11},
    {"Index": 71, "FWHM1": 0.08073, "err1": 0.0011, "FWHM2": 0.01729, "err2": 0.00006, "w01": -4.048e-10, "w02": 2.98e-11},
    {"Index": 72, "FWHM1": 0.09779, "err1": 0.0013, "FWHM2": 0.01962, "err2": 0.000048, "w01": 4.062e-11, "w02": 9.667e-12},
    {"Index": 73, "FWHM1": 0.06851, "err1": 0.00088, "FWHM2": 0.01619, "err2": 0.000073, "w01": 4.904e-10, "w02": -5.461e-11},
    {"Index": 74, "FWHM1": 0.09831, "err1": 0.0013, "FWHM2": 0.0191, "err2": 0.000048, "w01": 9.103e-11, "w02": 4.419e-12},
    {"Index": 75, "FWHM1": 0.08895, "err1": 0.0011, "FWHM2": 0.01859, "err2": 0.000053, "w01": -5.436e-10, "w02": 4.059e-11},
    {"Index": 76, "FWHM1": 0.08039, "err1": 0.00097, "FWHM2": 0.0183, "err2": 0.000066, "w01": 5.72e-10, "w02": -5.649e-11},
    {"Index": 77, "FWHM1": 0.08069, "err1": 0.00099, "FWHM2": 0.01797, "err2": 0.000066, "w01": 6.307e-10, "w02": -6.373e-11},
    {"Index": 78, "FWHM1": 0.08266, "err1": 0.00096, "FWHM2": 0.01787, "err2": 0.000057, "w01": -5.274e-10, "w02": 4.647e-11},
    {"Index": 79, "FWHM1": 0.07618, "err1": 0.00094, "FWHM2": 0.01721, "err2": 0.000067, "w01": 5.954e-10, "w02": -6.283e-11},
    {"Index": 80, "FWHM1": 0.1008, "err1": 0.0012, "FWHM2": 0.01944, "err2": 0.000048, "w01": -3.521e-10, "w02": 2.004e-11},
    {"Index": 81, "FWHM1": 0.08412, "err1": 0.00098, "FWHM2": 0.01772, "err2": 0.000057, "w01": -5.14e-10, "w02": 4.406e-11},
    {"Index": 82, "FWHM1": 0.09936, "err1": 0.0013, "FWHM2": 0.01931, "err2": 0.000054, "w01": 7.531e-11, "w02": 7.834e-12},
    {"Index": 83, "FWHM1": 0.0779, "err1": 0.00095, "FWHM2": 0.01739, "err2": 0.000069, "w01": 5.914e-10, "w02": -6.214e-11},
    {"Index": 84, "FWHM1": 0.1061, "err1": 0.0012, "FWHM2": 0.01968, "err2": 0.000046, "w01": 1.07e-10, "w02": 2.294e-12},
    {"Index": 85, "FWHM1": 0.1028, "err1": 0.0011, "FWHM2": 0.01936, "err2": 0.000045, "w01": -5.282e-11, "w02": -9.5e-12},
    {"Index": 86, "FWHM1": 0.08082, "err1": 0.001, "FWHM2": 0.01779, "err2": 0.00007, "w01": 6.046e-10, "w02": -6.257e-11},
    {"Index": 87, "FWHM1": 0.1073, "err1": 0.0012, "FWHM2": 0.01956, "err2": 0.000044, "w01": -1.454e-10, "w02": 1.489e-12},
    {"Index": 88, "FWHM1": 0.0994, "err1": 0.0011, "FWHM2": 0.01902, "err2": 0.00005, "w01": 4.633e-10, "w02": -3.18e-11},
    {"Index": 89, "FWHM1": 0.09252, "err1": 0.0011, "FWHM2": 0.01836, "err2": 0.000054, "w01": -3.863e-10, "w02": 2.636e-11},
    {"Index": 90, "FWHM1": 0.1014, "err1": 0.0012, "FWHM2": 0.01921, "err2": 0.000051, "w01": -2.804e-10, "w02": 1.348e-11},
    {"Index": 91, "FWHM1": 0.1022, "err1": 0.0011, "FWHM2": 0.0187, "err2": 0.000049, "w01": 2.606e-10, "w02": -1.249e-11},
    {"Index": 92, "FWHM1": 0.09718, "err1": 0.0011, "FWHM2": 0.01867, "err2": 0.00005, "w01": -5.182e-10, "w02": 3.872e-11},
    {"Index": 93, "FWHM1": 0.08084, "err1": 0.00093, "FWHM2": 0.01698, "err2": 0.000065, "w01": -6.189e-10, "w02": 6.611e-11},
    {"Index": 94, "FWHM1": 0.08931, "err1": 0.00098, "FWHM2": 0.01811, "err2": 0.000058, "w01": 5.769e-10, "w02": -5.256e-11},
    {"Index": 95, "FWHM1": 0.105, "err1": 0.0011, "FWHM2": 0.01911, "err2": 0.000046, "w01": -2.979e-10, "w02": 1.48e-11},
    {"Index": 96, "FWHM1": 0.08733, "err1": 0.001, "FWHM2": 0.01786, "err2": 0.000061, "w01": -6.138e-10, "w02": 5.731e-11},
    {"Index": 97, "FWHM1": 0.07893, "err1": 0.00096, "FWHM2": 0.01684, "err2": 0.000073, "w01": -6.582e-10, "w02": 7.606e-11},
    {"Index": 98, "FWHM1": 0.09194, "err1": 0.0011, "FWHM2": 0.01634, "err2": 0.000054, "w01": 1.126e-10, "w02": 9.854e-12},
    {"Index": 99, "FWHM1": 0.09508, "err1": 0.0011, "FWHM2": 0.01708, "err2": 0.000054, "w01": -5.418e-11, "w02": -1.73e-11},
    {"Index": 100, "FWHM1": 0.09254, "err1": 0.001, "FWHM2": 0.01728, "err2": 0.000052, "w01": 4.355e-10, "w02": -3.233e-11},
    {"Index": 101, "FWHM1": 0.0999, "err1": 0.0011, "FWHM2": 0.01849, "err2": 0.000052, "w01": 2.312e-11, "w02": 1.884e-11},
    {"Index": 102, "FWHM1": 0.08764, "err1": 0.00098, "FWHM2": 0.01746, "err2": 0.000061, "w01": -6.583e-10, "w02": 6.562e-11},
    {"Index": 103, "FWHM1": 0.08849, "err1": 0.001, "FWHM2": 0.01693, "err2": 0.000059, "w01": 4.777e-10, "w02": -3.925e-11},
    {"Index": 104, "FWHM1": 0.09576, "err1": 0.001, "FWHM2": 0.01787, "err2": 0.000051, "w01": -4.997e-10, "w02": 3.733e-11},
    {"Index": 105, "FWHM1": 0.09057, "err1": 0.0011, "FWHM2": 0.0158, "err2": 0.000058, "w01": -4.07e-10, "w02": 2.765e-11},
    {"Index": 106, "FWHM1": 0.08964, "err1": 0.001, "FWHM2": 0.01568, "err2": 0.000053, "w01": -4.461e-10, "w02": 3.335e-11},
    {"Index": 107, "FWHM1": 0.09766, "err1": 0.001, "FWHM2": 0.01794, "err2": 0.00005, "w01": 5.085e-10, "w02": -3.727e-11},
    {"Index": 108, "FWHM1": 0.08669, "err1": 0.001, "FWHM2": 0.01434, "err2": 0.000056, "w01": 3.259e-10, "w02": -1.694e-11},
    {"Index": 109, "FWHM1": 0.09758, "err1": 0.0011, "FWHM2": 0.01822, "err2": 0.000055, "w01": 5.625e-10, "w02": -4.411e-11},
    {"Index": 110, "FWHM1": 0.08132, "err1": 0.001, "FWHM2": 0.0156, "err2": 0.000067, "w01": -6.851e-10, "w02": 7.318e-11},
    {"Index": 111, "FWHM1": 0.08445, "err1": 0.001, "FWHM2": 0.01448, "err2": 0.000056, "w01": -4.169e-10, "w02": 3.045e-11},
    {"Index": 112, "FWHM1": 0.08357, "err1": 0.0011, "FWHM2": 0.01436, "err2": 0.000061, "w01": 1.283e-10, "w02": 1.626e-11},
    {"Index": 113, "FWHM1": 0.08606, "err1": 0.00099, "FWHM2": 0.01498, "err2": 0.000058, "w01": -9.727e-11, "w02": -1.961e-11},
    {"Index": 114, "FWHM1": 0.08837, "err1": 0.001, "FWHM2": 0.01698, "err2": 0.000063, "w01": 6.864e-10, "w02": -6.919e-11},
    {"Index": 115, "FWHM1": 0.1017, "err1": 0.0012, "FWHM2": 0.01789, "err2": 0.000053, "w01": 3.667e-11, "w02": 2.055e-11},
    {"Index": 116, "FWHM1": 0.09356, "err1": 0.0011, "FWHM2": 0.01639, "err2": 0.000058, "w01": 4.705e-12, "w02": 3.12e-11},
    {"Index": 117, "FWHM1": 0.09636, "err1": 0.0011, "FWHM2": 0.0164, "err2": 0.000052, "w01": -4.604e-10, "w02": 3.086e-11},
    {"Index": 118, "FWHM1": 0.1039, "err1": 0.0011, "FWHM2": 0.01799, "err2": 0.000047, "w01": -4.356e-10, "w02": 2.602e-11},
    {"Index": 119, "FWHM1": 0.09152, "err1": 0.0011, "FWHM2": 0.0158, "err2": 0.000059, "w01": -2.205e-11, "w02": -3.092e-11},
    {"Index": 120, "FWHM1": 0.0819, "err1": 0.0012, "FWHM2": 0.01185, "err2": 0.000061, "w01": -4.58e-10, "w02": 3.831e-11},
    {"Index": 121, "FWHM1": 0.09356, "err1": 0.001, "FWHM2": 0.01679, "err2": 0.000055, "w01": -5.824e-10, "w02": 4.933e-11},
    {"Index": 122, "FWHM1": 0.08295, "err1": 0.0011, "FWHM2": 0.01272, "err2": 0.00006, "w01": 2.982e-10, "w02": -7.851e-12},
    {"Index": 123, "FWHM1": 0.09546, "err1": 0.0011, "FWHM2": 0.01622, "err2": 0.000054, "w01": 5.242e-10, "w02": -3.744e-11},
    {"Index": 124, "FWHM1": 0.081, "err1": 0.0012, "FWHM2": 0.01163, "err2": 0.000062, "w01": -4.399e-10, "w02": 3.509e-11},
    {"Index": 125, "FWHM1": 0.09495, "err1": 0.0011, "FWHM2": 0.01704, "err2": 0.000055, "w01": 6.905e-10, "w02": -6.191e-11},
    {"Index": 126, "FWHM1": 0.09186, "err1": 0.001, "FWHM2": 0.01662, "err2": 0.000058, "w01": 6.753e-10, "w02": -6.305e-11},
    {"Index": 127, "FWHM1": 0.09202, "err1": 0.0011, "FWHM2": 0.01514, "err2": 0.000053, "w01": -1.129e-10, "w02": -1.688e-11},
    {"Index": 128, "FWHM1": 0.08666, "err1": 0.0011, "FWHM2": 0.0147, "err2": 0.000058, "w01": 2.33e-11, "w02": 3.534e-11},
    {"Index": 129, "FWHM1": 0.08348, "err1": 0.0012, "FWHM2": 0.01236, "err2": 0.00006, "w01": -3.622e-10, "w02": 1.664e-11},
    {"Index": 130, "FWHM1": 0.0819, "err1": 0.0012, "FWHM2": 0.01241, "err2": 0.000061, "w01": -3.043e-10, "w02": 5.588e-12},
    {"Index": 131, "FWHM1": 0.08385, "err1": 0.0012, "FWHM2": 0.01247, "err2": 0.000061, "w01": 3.279e-10, "w02": -1.092e-11},
    {"Index": 132, "FWHM1": 0.08291, "err1": 0.0012, "FWHM2": 0.01206, "err2": 0.000062, "w01": -2.05e-10, "w02": -1.215e-11},
    {"Index": 133, "FWHM1": 0.08904, "err1": 0.0013, "FWHM2": 0.01306, "err2": 0.000058, "w01": -3.702e-10, "w02": 1.675e-11},
    {"Index": 134, "FWHM1": 0.1007, "err1": 0.0012, "FWHM2": 0.01609, "err2": 0.000052, "w01": -1.663e-10, "w02": -9.097e-12},
    {"Index": 135, "FWHM1": 0.1065, "err1": 0.0013, "FWHM2": 0.01715, "err2": 0.000049, "w01": -3.838e-10, "w02": 1.828e-11},
    {"Index": 136, "FWHM1": 0.08477, "err1": 0.0011, "FWHM2": 0.01398, "err2": 0.00006, "w01": 4.174e-10, "w02": -2.405e-11},
    {"Index": 137, "FWHM1": 0.09833, "err1": 0.0012, "FWHM2": 0.01599, "err2": 0.000052, "w01": -5.087e-10, "w02": 3.173e-11},
    {"Index": 138, "FWHM1": 0.09086, "err1": 0.0011, "FWHM2": 0.01501, "err2": 0.000056, "w01": -5.521e-10, "w02": 4.106e-11},
    {"Index": 139, "FWHM1": 0.08309, "err1": 0.0013, "FWHM2": 0.01132, "err2": 0.000063, "w01": -1.357e-10, "w02": -2.991e-11},
    {"Index": 140, "FWHM1": 0.1054, "err1": 0.0012, "FWHM2": 0.01722, "err2": 0.000049, "w01": 5.104e-10, "w02": -3.081e-11},
    {"Index": 141, "FWHM1": 0.08678, "err1": 0.0015, "FWHM2": 0.01104, "err2": 0.000061, "w01": -7.16e-10, "w02": 6.975e-11},
    {"Index": 142, "FWHM1": 0.09972, "err1": 0.0012, "FWHM2": 0.01562, "err2": 0.00005, "w01": -3.9e-10, "w02": 1.806e-11},
    {"Index": 143, "FWHM1": 0.08383, "err1": 0.0013, "FWHM2": 0.0113, "err2": 0.000063, "w01": 1.99e-10, "w02": 2.062e-11},
    {"Index": 144, "FWHM1": 0.08852, "err1": 0.0013, "FWHM2": 0.01309, "err2": 0.000059, "w01": 3.154e-10, "w02": -4.308e-12},
    {"Index": 145, "FWHM1": 0.08746, "err1": 0.0015, "FWHM2": 0.011, "err2": 0.00006, "w01": 6.849e-10, "w02": -6.193e-11},
    {"Index": 146, "FWHM1": 0.09473, "err1": 0.0011, "FWHM2": 0.01594, "err2": 0.000055, "w01": 2.954e-11, "w02": 3.219e-11},
    {"Index": 147, "FWHM1": 0.09014, "err1": 0.0011, "FWHM2": 0.01539, "err2": 0.00006, "w01": 3.888e-11, "w02": 3.396e-11},
    {"Index": 148, "FWHM1": 0.1007, "err1": 0.0012, "FWHM2": 0.01568, "err2": 0.00005, "w01": 4.091e-10, "w02": -2.031e-11},
    {"Index": 149, "FWHM1": 0.09831, "err1": 0.0012, "FWHM2": 0.01569, "err2": 0.000052, "w01": -4.951e-10, "w02": 2.776e-11},
    {"Index": 150, "FWHM1": 0.08948, "err1": 0.0013, "FWHM2": 0.01177, "err2": 0.000062, "w01": -2.119e-10, "w02": -1.991e-11}
]

df = pd.DataFrame(data)

In [ ]:
plt.figure(figsize=(singlecol, singlecol),dpi=200)
plt.rcParams.update({'lines.markersize':3})
i=10
x=q/10
plt.errorbar(x[i:], df['FWHM1'][i:], yerr=df['err1'][i:], 
             fmt='o', ecolor='red', capsize=5, linestyle='-', 
             label='FWHM1 with Error Bars')

plt.errorbar(x[i:], df['FWHM2'][i:], yerr=df['err2'][i:], 
             fmt='s', ecolor='orange', capsize=5, linestyle='--', 
             label='FWHM2 with Error Bars')

# Formatting
#plt.title('FWHM1 vs Index')
plt.xlabel('q (1/A)')
plt.ylabel('FWHM (meV)')
plt.ylim(0,0.15)
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

### H FWHM

In [ ]:
import pandas as pd

# Standardized data: FWHM1 is always the larger (broad) component
data_block_3 = [
    {"Index": 0, "FWHM1": 0.1779, "err1": 0.0025, "FWHM2": 0.02324, "err2": 0.00014, "w01": -8.626e-10, "w02": 1.181e-10},
    {"Index": 1, "FWHM1": 0.191, "err1": 0.0026, "FWHM2": 0.02452, "err2": 0.00013, "w01": 4.853e-10, "w02": -4.928e-11},
    {"Index": 2, "FWHM1": 0.1846, "err1": 0.0025, "FWHM2": 0.02391, "err2": 0.00013, "w01": -6.709e-10, "w02": 8.288e-11},
    {"Index": 3, "FWHM1": 0.1818, "err1": 0.0029, "FWHM2": 0.0231, "err2": 0.00015, "w01": -8.473e-10, "w02": 1.076e-10},
    {"Index": 4, "FWHM1": 0.1794, "err1": 0.0029, "FWHM2": 0.02284, "err2": 0.00015, "w01": -8.825e-10, "w02": 1.205e-10},
    {"Index": 5, "FWHM1": 0.1693, "err1": 0.0025, "FWHM2": 0.02232, "err2": 0.00015, "w01": 1.101e-09, "w02": -1.646e-10},
    {"Index": 6, "FWHM1": 0.1919, "err1": 0.0026, "FWHM2": 0.02452, "err2": 0.00013, "w01": 4.572e-10, "w02": -4.439e-11},
    {"Index": 7, "FWHM1": 0.1909, "err1": 0.0027, "FWHM2": 0.02428, "err2": 0.00014, "w01": -5.695e-10, "w02": 5.198e-11},
    {"Index": 8, "FWHM1": 0.1929, "err1": 0.0027, "FWHM2": 0.02457, "err2": 0.00013, "w01": 4.662e-10, "w02": -4.06e-11},
    {"Index": 9, "FWHM1": 0.1364, "err1": 0.003, "FWHM2": 0.01829, "err2": 0.00022, "w01": 1.804e-09, "w02": -3.428e-10},
    {"Index": 10, "FWHM1": 0.1538, "err1": 0.0027, "FWHM2": 0.02059, "err2": 0.00018, "w01": 1.437e-09, "w02": -2.4e-10},
    {"Index": 11, "FWHM1": 0.1678, "err1": 0.0026, "FWHM2": 0.02215, "err2": 0.00015, "w01": -1.128e-09, "w02": 1.694e-10},
    {"Index": 12, "FWHM1": 0.1459, "err1": 0.0028, "FWHM2": 0.01964, "err2": 0.00019, "w01": 1.597e-09, "w02": -2.814e-10},
    {"Index": 13, "FWHM1": 0.1939, "err1": 0.0027, "FWHM2": 0.02471, "err2": 0.00013, "w01": 3.98e-10, "w02": -3.513e-11},
    {"Index": 14, "FWHM1": 0.1794, "err1": 0.0025, "FWHM2": 0.02335, "err2": 0.00013, "w01": 8.459e-10, "w02": -1.114e-10},
    {"Index": 15, "FWHM1": 0.1761, "err1": 0.003, "FWHM2": 0.02243, "err2": 0.00016, "w01": 9.937e-10, "w02": -1.418e-10},
    {"Index": 16, "FWHM1": 0.1784, "err1": 0.0029, "FWHM2": 0.02269, "err2": 0.00016, "w01": -9.49e-10, "w02": 1.295e-10},
    {"Index": 17, "FWHM1": 0.14, "err1": 0.003, "FWHM2": 0.01867, "err2": 0.00021, "w01": 1.792e-09, "w02": -3.325e-10},
    {"Index": 18, "FWHM1": 0.1671, "err1": 0.0031, "FWHM2": 0.02141, "err2": 0.00018, "w01": 1.232e-09, "w02": -1.887e-10},
    {"Index": 19, "FWHM1": 0.1343, "err1": 0.0028, "FWHM2": 0.01827, "err2": 0.00022, "w01": -1.794e-09, "w02": 3.438e-10},
    {"Index": 20, "FWHM1": 0.1719, "err1": 0.003, "FWHM2": 0.02197, "err2": 0.00016, "w01": 1.101e-09, "w02": -1.624e-10},
    {"Index": 21, "FWHM1": 0.1939, "err1": 0.0027, "FWHM2": 0.02465, "err2": 0.00013, "w01": 5.142e-10, "w02": -3.714e-11},
    {"Index": 22, "FWHM1": 0.1471, "err1": 0.0027, "FWHM2": 0.01982, "err2": 0.00019, "w01": -1.585e-09, "w02": 2.776e-10},
    {"Index": 23, "FWHM1": 0.183, "err1": 0.0023, "FWHM2": 0.0237, "err2": 0.00012, "w01": -7.435e-10, "w02": 9.617e-11},
    {"Index": 24, "FWHM1": 0.189, "err1": 0.0027, "FWHM2": 0.02427, "err2": 0.00014, "w01": 5.589e-10, "w02": -6.084e-11},
    {"Index": 25, "FWHM1": 0.1418, "err1": 0.003, "FWHM2": 0.0188, "err2": 0.00021, "w01": 1.771e-09, "w02": -3.22e-10},
    {"Index": 26, "FWHM1": 0.1372, "err1": 0.0029, "FWHM2": 0.01852, "err2": 0.00021, "w01": 1.73e-09, "w02": -3.215e-10},
    {"Index": 27, "FWHM1": 0.1834, "err1": 0.0025, "FWHM2": 0.02372, "err2": 0.00013, "w01": -7.442e-10, "w02": 9.164e-11},
    {"Index": 28, "FWHM1": 0.1838, "err1": 0.0024, "FWHM2": 0.02375, "err2": 0.00013, "w01": 7.256e-10, "w02": -9.06e-11},
    {"Index": 29, "FWHM1": 0.149, "err1": 0.0027, "FWHM2": 0.02008, "err2": 0.00018, "w01": -1.538e-09, "w02": 2.645e-10},
    {"Index": 30, "FWHM1": 0.1677, "err1": 0.003, "FWHM2": 0.02145, "err2": 0.00017, "w01": 1.231e-09, "w02": -1.881e-10},
    {"Index": 31, "FWHM1": 0.1664, "err1": 0.0026, "FWHM2": 0.02196, "err2": 0.00015, "w01": -1.166e-09, "w02": 1.756e-10},
    {"Index": 32, "FWHM1": 0.146, "err1": 0.0031, "FWHM2": 0.0192, "err2": 0.00021, "w01": 1.752e-09, "w02": -3.106e-10},
    {"Index": 33, "FWHM1": 0.1813, "err1": 0.0027, "FWHM2": 0.02304, "err2": 0.00014, "w01": 8.158e-10, "w02": -1.071e-10},
    {"Index": 34, "FWHM1": 0.1454, "err1": 0.0028, "FWHM2": 0.01965, "err2": 0.00019, "w01": -1.584e-09, "w02": 2.777e-10},
    {"Index": 35, "FWHM1": 0.1434, "err1": 0.003, "FWHM2": 0.01893, "err2": 0.00021, "w01": -1.759e-09, "w02": 3.158e-10},
    {"Index": 36, "FWHM1": 0.1751, "err1": 0.0025, "FWHM2": 0.02286, "err2": 0.00014, "w01": 9.535e-10, "w02": -1.33e-10},
    {"Index": 37, "FWHM1": 0.1657, "err1": 0.0026, "FWHM2": 0.0219, "err2": 0.00015, "w01": -1.181e-09, "w02": 1.797e-10},
    {"Index": 38, "FWHM1": 0.1454, "err1": 0.003, "FWHM2": 0.01921, "err2": 0.0002, "w01": 1.737e-09, "w02": -3.081e-10},
    {"Index": 39, "FWHM1": 0.1439, "err1": 0.0028, "FWHM2": 0.01926, "err2": 0.0002, "w01": -1.678e-09, "w02": 3.007e-10},
    {"Index": 40, "FWHM1": 0.1927, "err1": 0.0025, "FWHM2": 0.02456, "err2": 0.00013, "w01": 4.568e-10, "w02": -4.378e-11},
    {"Index": 41, "FWHM1": 0.1418, "err1": 0.0028, "FWHM2": 0.01899, "err2": 0.0002, "w01": 1.737e-09, "w02": -3.171e-10},
    {"Index": 42, "FWHM1": 0.1728, "err1": 0.0024, "FWHM2": 0.02259, "err2": 0.00014, "w01": 1.04e-09, "w02": -1.49e-10},
    {"Index": 43, "FWHM1": 0.1904, "err1": 0.0024, "FWHM2": 0.02434, "err2": 0.00012, "w01": 5.51e-10, "w02": -5.951e-11},
    {"Index": 44, "FWHM1": 0.1519, "err1": 0.0027, "FWHM2": 0.02039, "err2": 0.00018, "w01": -1.481e-09, "w02": 2.484e-10},
    {"Index": 45, "FWHM1": 0.1935, "err1": 0.0026, "FWHM2": 0.02456, "err2": 0.00013, "w01": 4.338e-10, "w02": -4.011e-11},
    {"Index": 46, "FWHM1": 0.183, "err1": 0.0026, "FWHM2": 0.02358, "err2": 0.00013, "w01": 7.553e-10, "w02": -9.458e-11},
    {"Index": 47, "FWHM1": 0.1435, "err1": 0.0029, "FWHM2": 0.01902, "err2": 0.0002, "w01": 1.76e-09, "w02": -3.159e-10},
    {"Index": 48, "FWHM1": 0.1427, "err1": 0.0029, "FWHM2": 0.01895, "err2": 0.0002, "w01": 1.705e-09, "w02": -3.032e-10},
    {"Index": 49, "FWHM1": 0.1704, "err1": 0.0024, "FWHM2": 0.02235, "err2": 0.00014, "w01": 1.095e-09, "w02": -1.594e-10},
    {"Index": 50, "FWHM1": 0.194, "err1": 0.0027, "FWHM2": 0.0246, "err2": 0.00013, "w01": 4.171e-10, "w02": -3.767e-11},
    {"Index": 51, "FWHM1": 0.159, "err1": 0.003, "FWHM2": 0.02049, "err2": 0.00018, "w01": -1.474e-09, "w02": 2.403e-10},
    {"Index": 52, "FWHM1": 0.136, "err1": 0.0029, "FWHM2": 0.01833, "err2": 0.00021, "w01": -1.805e-09, "w02": 3.37e-10},
    {"Index": 53, "FWHM1": 0.1447, "err1": 0.0027, "FWHM2": 0.01956, "err2": 0.00019, "w01": 1.596e-09, "w02": -2.791e-10},
    {"Index": 54, "FWHM1": 0.1407, "err1": 0.0029, "FWHM2": 0.01878, "err2": 0.0002, "w01": -1.763e-09, "w02": 3.195e-10},
    {"Index": 55, "FWHM1": 0.143, "err1": 0.0028, "FWHM2": 0.01938, "err2": 0.00019, "w01": -1.607e-09, "w02": 2.829e-10},
    {"Index": 56, "FWHM1": 0.1923, "err1": 0.0024, "FWHM2": 0.02443, "err2": 0.00012, "w01": -4.703e-10, "w02": 4.703e-11},
    {"Index": 57, "FWHM1": 0.1483, "err1": 0.0027, "FWHM2": 0.02, "err2": 0.00018, "w01": -1.542e-09, "w02": 2.632e-10},
    {"Index": 58, "FWHM1": 0.1768, "err1": 0.0024, "FWHM2": 0.02292, "err2": 0.00013, "w01": -9.477e-10, "w02": 1.298e-10},
    {"Index": 59, "FWHM1": 0.1507, "err1": 0.0027, "FWHM2": 0.02025, "err2": 0.00018, "w01": 1.494e-09, "w02": -2.5e-10},
    {"Index": 60, "FWHM1": 0.1668, "err1": 0.0024, "FWHM2": 0.02193, "err2": 0.00014, "w01": -1.212e-09, "w02": 1.843e-10},
    {"Index": 61, "FWHM1": 0.1494, "err1": 0.0027, "FWHM2": 0.02014, "err2": 0.00018, "w01": -1.515e-09, "w02": 2.561e-10},
    {"Index": 62, "FWHM1": 0.151, "err1": 0.0029, "FWHM2": 0.0197, "err2": 0.00019, "w01": 1.632e-09, "w02": -2.758e-10},
    {"Index": 63, "FWHM1": 0.1493, "err1": 0.003, "FWHM2": 0.01951, "err2": 0.00019, "w01": 1.639e-09, "w02": -2.777e-10},
    {"Index": 64, "FWHM1": 0.1499, "err1": 0.0026, "FWHM2": 0.02011, "err2": 0.00017, "w01": 1.561e-09, "w02": -2.663e-10},
    {"Index": 65, "FWHM1": 0.1426, "err1": 0.0028, "FWHM2": 0.01918, "err2": 0.00019, "w01": 1.67e-09, "w02": -2.958e-10},
    {"Index": 66, "FWHM1": 0.1881, "err1": 0.0025, "FWHM2": 0.02382, "err2": 0.00012, "w01": -6.347e-10, "w02": 7.251e-11},
    {"Index": 67, "FWHM1": 0.1925, "err1": 0.0024, "FWHM2": 0.02446, "err2": 0.00012, "w01": 4.805e-10, "w02": -4.637e-11},
    {"Index": 68, "FWHM1": 0.1916, "err1": 0.0024, "FWHM2": 0.0243, "err2": 0.00012, "w01": 5.47e-10, "w02": -5.842e-11},
    {"Index": 69, "FWHM1": 0.1468, "err1": 0.0027, "FWHM2": 0.01963, "err2": 0.00018, "w01": -1.625e-09, "w02": 2.823e-10},
    {"Index": 70, "FWHM1": 0.1889, "err1": 0.0025, "FWHM2": 0.02392, "err2": 0.00012, "w01": 4.215e-11, "w02": -5.687e-11},
    {"Index": 71, "FWHM1": 0.1797, "err1": 0.0027, "FWHM2": 0.02273, "err2": 0.00014, "w01": 9.122e-10, "w02": -1.201e-10},
    {"Index": 72, "FWHM1": 0.1799, "err1": 0.0023, "FWHM2": 0.02312, "err2": 0.00012, "w01": 1.581e-09, "w02": -1.354e-10},
    {"Index": 73, "FWHM1": 0.1564, "err1": 0.0026, "FWHM2": 0.02091, "err2": 0.00016, "w01": -1.36e-09, "w02": 2.158e-10},
    {"Index": 74, "FWHM1": 0.1756, "err1": 0.0027, "FWHM2": 0.0222, "err2": 0.00014, "w01": 1.057e-09, "w02": -1.477e-10},
    {"Index": 75, "FWHM1": 0.1498, "err1": 0.0027, "FWHM2": 0.01976, "err2": 0.00018, "w01": -1.639e-09, "w02": 2.789e-10},
    {"Index": 76, "FWHM1": 0.1494, "err1": 0.0026, "FWHM2": 0.02006, "err2": 0.00017, "w01": -1.542e-09, "w02": 2.611e-10},
    {"Index": 77, "FWHM1": 0.1916, "err1": 0.0025, "FWHM2": 0.02428, "err2": 0.00012, "w01": 3.855e-12, "w02": 4.354e-11},
    {"Index": 78, "FWHM1": 0.1934, "err1": 0.0024, "FWHM2": 0.02448, "err2": 0.00012, "w01": -4.854e-10, "w02": 4.516e-11},
    {"Index": 79, "FWHM1": 0.1459, "err1": 0.0027, "FWHM2": 0.0195, "err2": 0.00018, "w01": -1.624e-09, "w02": 2.811e-10},
    {"Index": 80, "FWHM1": 0.1541, "err1": 0.0028, "FWHM2": 0.02012, "err2": 0.00017, "w01": -1.549e-09, "w02": 2.549e-10},
    {"Index": 81, "FWHM1": 0.1823, "err1": 0.0026, "FWHM2": 0.02303, "err2": 0.00013, "w01": -8.201e-10, "w02": 1.046e-10},
    {"Index": 82, "FWHM1": 0.189, "err1": 0.0023, "FWHM2": 0.02391, "err2": 0.00011, "w01": 6.893e-10, "w02": -8.065e-11},
    {"Index": 83, "FWHM1": 0.1607, "err1": 0.0024, "FWHM2": 0.02136, "err2": 0.00015, "w01": -1.327e-09, "w02": 2.073e-10},
    {"Index": 84, "FWHM1": 0.1757, "err1": 0.0027, "FWHM2": 0.02216, "err2": 0.00014, "w01": 1.495e-09, "w02": -1.56e-10},
    {"Index": 85, "FWHM1": 0.1754, "err1": 0.0026, "FWHM2": 0.02215, "err2": 0.00014, "w01": 1.071e-09, "w02": -1.498e-10},
    {"Index": 86, "FWHM1": 0.1575, "err1": 0.0025, "FWHM2": 0.02102, "err2": 0.00016, "w01": -1.351e-09, "w02": 2.13e-10},
    {"Index": 87, "FWHM1": 0.1695, "err1": 0.0029, "FWHM2": 0.02135, "err2": 0.00016, "w01": -2.122e-09, "w02": 2.02e-10},
    {"Index": 88, "FWHM1": 0.1532, "err1": 0.0028, "FWHM2": 0.01997, "err2": 0.00017, "w01": 1.591e-09, "w02": -2.626e-10},
    {"Index": 89, "FWHM1": 0.194, "err1": 0.0025, "FWHM2": 0.02448, "err2": 0.00012, "w01": -5.817e-10, "w02": 4.623e-11},
    {"Index": 90, "FWHM1": 0.1806, "err1": 0.0023, "FWHM2": 0.02307, "err2": 0.00012, "w01": 1.009e-09, "w02": -1.244e-10},
    {"Index": 91, "FWHM1": 0.1662, "err1": 0.0028, "FWHM2": 0.02105, "err2": 0.00016, "w01": -1.363e-09, "w02": 2.064e-10},
    {"Index": 92, "FWHM1": 0.1574, "err1": 0.0026, "FWHM2": 0.02044, "err2": 0.00016, "w01": 1.522e-09, "w02": -2.442e-10},
    {"Index": 93, "FWHM1": 0.1922, "err1": 0.0024, "FWHM2": 0.02433, "err2": 0.00011, "w01": 5.001e-10, "w02": -4.981e-11},
    {"Index": 94, "FWHM1": 0.1878, "err1": 0.0024, "FWHM2": 0.02366, "err2": 0.00012, "w01": 6.649e-10, "w02": -7.51e-11},
    {"Index": 95, "FWHM1": 0.1789, "err1": 0.0026, "FWHM2": 0.02241, "err2": 0.00013, "w01": 9.549e-10, "w02": -1.327e-10},
    {"Index": 96, "FWHM1": 0.165, "err1": 0.0024, "FWHM2": 0.02174, "err2": 0.00014, "w01": 1.252e-09, "w02": -1.847e-10},
    {"Index": 97, "FWHM1": 0.1603, "err1": 0.0024, "FWHM2": 0.02134, "err2": 0.00015, "w01": 1.324e-09, "w02": -2.055e-10},
    {"Index": 98, "FWHM1": 0.1664, "err1": 0.0027, "FWHM2": 0.021, "err2": 0.00015, "w01": -1.389e-09, "w02": 2.086e-10},
    {"Index": 99, "FWHM1": 0.1645, "err1": 0.0026, "FWHM2": 0.02101, "err2": 0.00015, "w01": -1.386e-09, "w02": 2.116e-10},
    {"Index": 100, "FWHM1": 0.1934, "err1": 0.0024, "FWHM2": 0.02418, "err2": 0.00011, "w01": -9.711e-12, "w02": -5.61e-11},
    {"Index": 101, "FWHM1": 0.1735, "err1": 0.0022, "FWHM2": 0.02244, "err2": 0.00012, "w01": 1.105e-09, "w02": -1.565e-10},
    {"Index": 102, "FWHM1": 0.19, "err1": 0.0023, "FWHM2": 0.02399, "err2": 0.00011, "w01": -5.816e-10, "w02": 6.181e-11},
    {"Index": 103, "FWHM1": 0.1933, "err1": 0.0024, "FWHM2": 0.02442, "err2": 0.00012, "w01": 4.949e-10, "w02": -4.721e-11},
    {"Index": 104, "FWHM1": 0.1849, "err1": 0.0023, "FWHM2": 0.02331, "err2": 0.00012, "w01": 7.503e-10, "w02": -9.145e-11},
    {"Index": 105, "FWHM1": 0.1642, "err1": 0.0026, "FWHM2": 0.02087, "err2": 0.00015, "w01": -1.466e-09, "w02": 2.193e-10},
    {"Index": 106, "FWHM1": 0.1774, "err1": 0.0025, "FWHM2": 0.02214, "err2": 0.00013, "w01": 1.08e-09, "w02": -1.471e-10},
    {"Index": 107, "FWHM1": 0.1858, "err1": 0.0024, "FWHM2": 0.0233, "err2": 0.00012, "w01": 7.708e-10, "w02": -9.271e-11},
    {"Index": 108, "FWHM1": 0.172, "err1": 0.0027, "FWHM2": 0.02139, "err2": 0.00014, "w01": -1.278e-09, "w02": 1.832e-10},
    {"Index": 109, "FWHM1": 0.1673, "err1": 0.0023, "FWHM2": 0.0219, "err2": 0.00013, "w01": -2.601e-10, "w02": 1.617e-10},
    {"Index": 110, "FWHM1": 0.1911, "err1": 0.0023, "FWHM2": 0.0242, "err2": 0.00011, "w01": -5.769e-10, "w02": 5.562e-11},
    {"Index": 111, "FWHM1": 0.1946, "err1": 0.0024, "FWHM2": 0.0241, "err2": 0.00011, "w01": 6.302e-10, "w02": -6.579e-11},
    {"Index": 112, "FWHM1": 0.1833, "err1": 0.0022, "FWHM2": 0.02315, "err2": 0.00011, "w01": 9.226e-10, "w02": -1.189e-10},
    {"Index": 113, "FWHM1": 0.1926, "err1": 0.0024, "FWHM2": 0.02394, "err2": 0.00011, "w01": -6.48e-10, "w02": 7.569e-11},
    {"Index": 114, "FWHM1": 0.1936, "err1": 0.0023, "FWHM2": 0.02431, "err2": 0.00011, "w01": 5.127e-10, "w02": -4.99e-11},
    {"Index": 115, "FWHM1": 0.1726, "err1": 0.0022, "FWHM2": 0.02229, "err2": 0.00012, "w01": -1.177e-09, "w02": 1.687e-10},
    {"Index": 116, "FWHM1": 0.1627, "err1": 0.0025, "FWHM2": 0.02087, "err2": 0.00015, "w01": 1.433e-09, "w02": -2.205e-10},
    {"Index": 117, "FWHM1": 0.1833, "err1": 0.0024, "FWHM2": 0.02268, "err2": 0.00012, "w01": 9.224e-10, "w02": -1.185e-10},
    {"Index": 118, "FWHM1": 0.1845, "err1": 0.0024, "FWHM2": 0.02312, "err2": 0.00012, "w01": -7.928e-10, "w02": 9.508e-11},
    {"Index": 119, "FWHM1": 0.1632, "err1": 0.0025, "FWHM2": 0.02086, "err2": 0.00014, "w01": -1.448e-09, "w02": 2.235e-10},
    {"Index": 120, "FWHM1": 0.1887, "err1": 0.0023, "FWHM2": 0.02336, "err2": 0.00011, "w01": 8.385e-10, "w02": -1.03e-10},
    {"Index": 121, "FWHM1": 0.1898, "err1": 0.0022, "FWHM2": 0.02383, "err2": 0.00011, "w01": -6.358e-10, "w02": 6.931e-11},
    {"Index": 122, "FWHM1": 0.176, "err1": 0.0026, "FWHM2": 0.02168, "err2": 0.00013, "w01": 1.208e-09, "w02": -1.673e-10},
    {"Index": 123, "FWHM1": 0.1849, "err1": 0.0023, "FWHM2": 0.0229, "err2": 0.00011, "w01": -8.517e-10, "w02": 1.082e-10},
    {"Index": 124, "FWHM1": 0.1929, "err1": 0.0024, "FWHM2": 0.02366, "err2": 0.00011, "w01": 7.614e-10, "w02": -8.582e-11},
    {"Index": 125, "FWHM1": 0.1921, "err1": 0.0023, "FWHM2": 0.02408, "err2": 0.00011, "w01": -5.016e-11, "w02": 3.726e-12},
    {"Index": 126, "FWHM1": 0.1903, "err1": 0.0022, "FWHM2": 0.0239, "err2": 0.00011, "w01": 6.053e-10, "w02": -6.483e-11},
    {"Index": 127, "FWHM1": 0.1968, "err1": 0.0024, "FWHM2": 0.02424, "err2": 0.00011, "w01": 5.714e-10, "w02": -5.729e-11},
    {"Index": 128, "FWHM1": 0.1936, "err1": 0.0023, "FWHM2": 0.02446, "err2": 0.00011, "w01": -5.073e-10, "w02": 4.785e-11},
    {"Index": 129, "FWHM1": 0.1862, "err1": 0.0022, "FWHM2": 0.02316, "err2": 0.00011, "w01": -1.215e-09, "w02": 1.227e-10},
    {"Index": 130, "FWHM1": 0.1848, "err1": 0.0022, "FWHM2": 0.02308, "err2": 0.00011, "w01": 9.717e-10, "w02": -1.25e-10},
    {"Index": 131, "FWHM1": 0.1831, "err1": 0.0024, "FWHM2": 0.02235, "err2": 0.00012, "w01": 1.046e-09, "w02": -1.371e-10},
    {"Index": 132, "FWHM1": 0.1952, "err1": 0.0023, "FWHM2": 0.02379, "err2": 0.00011, "w01": -8.391e-10, "w02": 8.373e-11},
    {"Index": 133, "FWHM1": 0.1854, "err1": 0.0023, "FWHM2": 0.02262, "err2": 0.00011, "w01": 9.778e-10, "w02": -1.241e-10},
    {"Index": 134, "FWHM1": 0.1773, "err1": 0.0022, "FWHM2": 0.02262, "err2": 0.00012, "w01": 1.086e-09, "w02": -1.488e-10},
    {"Index": 135, "FWHM1": 0.1745, "err1": 0.0022, "FWHM2": 0.02247, "err2": 0.00012, "w01": -1.133e-09, "w02": 1.581e-10},
    {"Index": 136, "FWHM1": 0.1609, "err1": 0.0022, "FWHM2": 0.02145, "err2": 0.00013, "w01": -1.327e-09, "w02": 2.003e-10},
    {"Index": 137, "FWHM1": 0.1883, "err1": 0.0023, "FWHM2": 0.0232, "err2": 0.00011, "w01": 8.235e-10, "w02": -9.668e-11},
    {"Index": 138, "FWHM1": 0.1926, "err1": 0.0022, "FWHM2": 0.02424, "err2": 0.00011, "w01": -5.496e-10, "w02": 5.468e-11},
    {"Index": 139, "FWHM1": 0.1975, "err1": 0.0023, "FWHM2": 0.02391, "err2": 0.00011, "w01": -6.905e-10, "w02": 7.61e-11},
    {"Index": 140, "FWHM1": 0.1885, "err1": 0.0022, "FWHM2": 0.02332, "err2": 0.00011, "w01": -7.652e-10, "w02": 8.842e-11},
    {"Index": 141, "FWHM1": 0.1976, "err1": 0.0024, "FWHM2": 0.02389, "err2": 0.00011, "w01": 9.621e-10, "w02": -8.106e-11},
    {"Index": 142, "FWHM1": 0.1962, "err1": 0.0024, "FWHM2": 0.02458, "err2": 0.00011, "w01": 4.864e-10, "w02": -4.498e-11},
    {"Index": 143, "FWHM1": 0.1927, "err1": 0.0022, "FWHM2": 0.0234, "err2": 0.00011, "w01": -8.693e-10, "w02": 1.057e-10},
    {"Index": 144, "FWHM1": 0.1771, "err1": 0.0022, "FWHM2": 0.02205, "err2": 0.00011, "w01": 1.217e-09, "w02": -1.702e-10},
    {"Index": 145, "FWHM1": 0.1927, "err1": 0.0022, "FWHM2": 0.02346, "err2": 0.00011, "w01": -8.709e-10, "w02": 1.064e-10},
    {"Index": 146, "FWHM1": 0.1665, "err1": 0.0021, "FWHM2": 0.02171, "err2": 0.00012, "w01": 1.273e-09, "w02": -1.891e-10},
    {"Index": 147, "FWHM1": 0.1665, "err1": 0.0021, "FWHM2": 0.02172, "err2": 0.00012, "w01": 1.277e-09, "w02": -1.891e-10},
    {"Index": 148, "FWHM1": 0.1983, "err1": 0.0023, "FWHM2": 0.02432, "err2": 0.00011, "w01": 2.096e-12, "w02": -5.048e-11},
    {"Index": 149, "FWHM1": 0.1648, "err1": 0.0021, "FWHM2": 0.02165, "err2": 0.00013, "w01": -1.441e-09, "w02": 1.935e-10},
    {"Index": 150, "FWHM1": 0.196, "err1": 0.0022, "FWHM2": 0.02348, "err2": 0.0001, "w01": -8.558e-10, "w02": 1.037e-10}
]

df = pd.DataFrame(data_block_3)

In [ ]:
plt.figure(figsize=(singlecol, singlecol),dpi=200)
plt.rcParams.update({'lines.markersize':3})
i=10
x=q/10
plt.errorbar(x[i:], df['FWHM1'][i:], yerr=df['err1'][i:], 
             fmt='o', ecolor='red', capsize=5, linestyle='-', 
             label='FWHM1 with Error Bars')

plt.errorbar(x[i:], df['FWHM2'][i:], yerr=df['err2'][i:], 
             fmt='s', ecolor='orange', capsize=5, linestyle='--', 
             label='FWHM2 with Error Bars')

# Formatting
#plt.title('FWHM1 vs Index')
plt.xlabel('q (1/A)')
plt.ylabel('FWHM (meV)')
#plt.ylim(0,0.15)
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

### Total FWHM

In [ ]:
import pandas as pd

data = [
    {"Index": 0, "FWHM1": 0.121, "err1": 0.0017, "FWHM2": 0.01344, "err2": 8e-05, "w01": 9.854e-10, "w02": -9.965e-11},
    {"Index": 1, "FWHM1": 0.1311, "err1": 0.0017, "FWHM2": 0.01445, "err2": 7.7e-05, "w01": -9.232e-10, "w02": 9.063e-11},
    {"Index": 2, "FWHM1": 0.1289, "err1": 0.0017, "FWHM2": 0.01444, "err2": 7.9e-05, "w01": 9.679e-10, "w02": -9.982e-11},
    {"Index": 3, "FWHM1": 0.1271, "err1": 0.0019, "FWHM2": 0.0142, "err2": 8.8e-05, "w01": 1.221e-09, "w02": -1.491e-10},
    {"Index": 4, "FWHM1": 0.1275, "err1": 0.0019, "FWHM2": 0.01436, "err2": 9e-05, "w01": 1.246e-09, "w02": -1.554e-10},
    {"Index": 5, "FWHM1": 0.1236, "err1": 0.0017, "FWHM2": 0.01439, "err2": 9e-05, "w01": -1.142e-09, "w02": 1.376e-10},
    {"Index": 6, "FWHM1": 0.1397, "err1": 0.0017, "FWHM2": 0.01588, "err2": 8e-05, "w01": -9.892e-10, "w02": 1.103e-10},
    {"Index": 7, "FWHM1": 0.1383, "err1": 0.0018, "FWHM2": 0.01583, "err2": 8.5e-05, "w01": 1.094e-09, "w02": -1.315e-10},
    {"Index": 8, "FWHM1": 0.1431, "err1": 0.0018, "FWHM2": 0.01642, "err2": 8.3e-05, "w01": -9.903e-10, "w02": 1.134e-10},
    {"Index": 9, "FWHM1": 0.1087, "err1": 0.002, "FWHM2": 0.01314, "err2": 0.00012, "w01": -1.473e-09, "w02": 2.212e-10},
    {"Index": 10, "FWHM1": 0.1197, "err1": 0.0019, "FWHM2": 0.01461, "err2": 0.00011, "w01": -1.306e-09, "w02": 1.824e-10},
    {"Index": 11, "FWHM1": 0.1318, "err1": 0.0019, "FWHM2": 0.01594, "err2": 0.0001, "w01": -1.197e-09, "w02": 1.535e-10},
    {"Index": 12, "FWHM1": 0.1166, "err1": 0.0019, "FWHM2": 0.01446, "err2": 0.00012, "w01": 1.405e-09, "w02": -2.091e-10},
    {"Index": 13, "FWHM1": 0.1545, "err1": 0.0019, "FWHM2": 0.01812, "err2": 8.7e-05, "w01": -8.469e-10, "w02": 9.05e-11},
    {"Index": 14, "FWHM1": 0.1466, "err1": 0.0019, "FWHM2": 0.01765, "err2": 9.4e-05, "w01": 1.024e-09, "w02": -1.209e-10},
    {"Index": 15, "FWHM1": 0.1431, "err1": 0.0022, "FWHM2": 0.01705, "err2": 0.00011, "w01": 1.278e-09, "w02": -1.704e-10},
    {"Index": 16, "FWHM1": 0.1461, "err1": 0.0022, "FWHM2": 0.01744, "err2": 0.00011, "w01": 1.228e-09, "w02": -1.624e-10},
    {"Index": 17, "FWHM1": 0.1193, "err1": 0.0022, "FWHM2": 0.015, "err2": 0.00014, "w01": 1.64e-09, "w02": -2.614e-10},
    {"Index": 18, "FWHM1": 0.1412, "err1": 0.0024, "FWHM2": 0.01709, "err2": 0.00012, "w01": -1.375e-09, "w02": 1.915e-10},
    {"Index": 19, "FWHM1": 0.1134, "err1": 0.0021, "FWHM2": 0.01469, "err2": 0.00015, "w01": -1.614e-09, "w02": 2.732e-10},
    {"Index": 20, "FWHM1": 0.1468, "err1": 0.0023, "FWHM2": 0.01783, "err2": 0.00012, "w01": -1.293e-09, "w02": 1.756e-10},
    {"Index": 21, "FWHM1": 0.1661, "err1": 0.0021, "FWHM2": 0.01999, "err2": 9.7e-05, "w01": -7.289e-10, "w02": 7.686e-11},
    {"Index": 22, "FWHM1": 0.126, "err1": 0.0021, "FWHM2": 0.01634, "err2": 0.00014, "w01": -1.53e-09, "w02": 2.433e-10},
    {"Index": 23, "FWHM1": 0.1612, "err1": 0.002, "FWHM2": 0.01988, "err2": 9.5e-05, "w01": 9.021e-10, "w02": -1.059e-10},
    {"Index": 24, "FWHM1": 0.1672, "err1": 0.0022, "FWHM2": 0.02047, "err2": 0.0001, "w01": 7.395e-10, "w02": -7.785e-11},
    {"Index": 25, "FWHM1": 0.1284, "err1": 0.0025, "FWHM2": 0.01648, "err2": 0.00016, "w01": -1.684e-09, "w02": 2.713e-10},
    {"Index": 26, "FWHM1": 0.1212, "err1": 0.0023, "FWHM2": 0.01594, "err2": 0.00016, "w01": 1.644e-09, "w02": -2.775e-10},
    {"Index": 27, "FWHM1": 0.1667, "err1": 0.0022, "FWHM2": 0.02071, "err2": 0.0001, "w01": -8.192e-10, "w02": 9.219e-11},
    {"Index": 28, "FWHM1": 0.1676, "err1": 0.0021, "FWHM2": 0.02086, "err2": 0.0001, "w01": 8.256e-10, "w02": -9.321e-11},
    {"Index": 29, "FWHM1": 0.1323, "err1": 0.0023, "FWHM2": 0.01747, "err2": 0.00015, "w01": -1.539e-09, "w02": 2.456e-10},
    {"Index": 30, "FWHM1": 0.1538, "err1": 0.0026, "FWHM2": 0.01915, "err2": 0.00014, "w01": 1.318e-09, "w02": -1.826e-10},
    {"Index": 31, "FWHM1": 0.1519, "err1": 0.0023, "FWHM2": 0.01957, "err2": 0.00013, "w01": 1.214e-09, "w02": -1.672e-10},
    {"Index": 32, "FWHM1": 0.1368, "err1": 0.0027, "FWHM2": 0.0177, "err2": 0.00017, "w01": 1.696e-09, "w02": -2.723e-10},
    {"Index": 33, "FWHM1": 0.1658, "err1": 0.0024, "FWHM2": 0.02059, "err2": 0.00012, "w01": 9.726e-10, "w02": -1.246e-10},
    {"Index": 34, "FWHM1": 0.132, "err1": 0.0024, "FWHM2": 0.01767, "err2": 0.00016, "w01": -1.582e-09, "w02": 2.594e-10},
    {"Index": 35, "FWHM1": 0.1359, "err1": 0.0028, "FWHM2": 0.01774, "err2": 0.00017, "w01": -1.712e-09, "w02": 2.773e-10},
    {"Index": 36, "FWHM1": 0.1642, "err1": 0.0023, "FWHM2": 0.02103, "err2": 0.00012, "w01": -9.969e-10, "w02": 1.269e-10},
    {"Index": 37, "FWHM1": 0.154, "err1": 0.0023, "FWHM2": 0.02008, "err2": 0.00013, "w01": 1.234e-09, "w02": -1.735e-10},
    {"Index": 38, "FWHM1": 0.1381, "err1": 0.0027, "FWHM2": 0.01812, "err2": 0.00017, "w01": 1.7e-09, "w02": -2.769e-10},
    {"Index": 39, "FWHM1": 0.1341, "err1": 0.0025, "FWHM2": 0.01789, "err2": 0.00017, "w01": -1.667e-09, "w02": 2.803e-10},
    {"Index": 40, "FWHM1": 0.1812, "err1": 0.0023, "FWHM2": 0.0227, "err2": 0.00011, "w01": 5.69e-10, "w02": -5.623e-11},
    {"Index": 41, "FWHM1": 0.1338, "err1": 0.0026, "FWHM2": 0.0179, "err2": 0.00017, "w01": 1.72e-09, "w02": -2.924e-10},
    {"Index": 42, "FWHM1": 0.165, "err1": 0.0023, "FWHM2": 0.02135, "err2": 0.00012, "w01": -1.06e-09, "w02": 1.405e-10},
    {"Index": 43, "FWHM1": 0.1822, "err1": 0.0023, "FWHM2": 0.02295, "err2": 0.00011, "w01": 6.02e-10, "w02": -6.086e-11},
    {"Index": 44, "FWHM1": 0.143, "err1": 0.0025, "FWHM2": 0.0192, "err2": 0.00016, "w01": -1.495e-09, "w02": 2.388e-10},
    {"Index": 45, "FWHM1": 0.1835, "err1": 0.0024, "FWHM2": 0.02306, "err2": 0.00011, "w01": -5.589e-10, "w02": 5.509e-11},
    {"Index": 46, "FWHM1": 0.1771, "err1": 0.0024, "FWHM2": 0.02261, "err2": 0.00012, "w01": 7.734e-10, "w02": -8.856e-11},
    {"Index": 47, "FWHM1": 0.1386, "err1": 0.0028, "FWHM2": 0.01844, "err2": 0.00018, "w01": 1.729e-09, "w02": -2.899e-10},
    {"Index": 48, "FWHM1": 0.1376, "err1": 0.0028, "FWHM2": 0.01835, "err2": 0.00018, "w01": -1.681e-09, "w02": 2.792e-10},
    {"Index": 49, "FWHM1": 0.1641, "err1": 0.0023, "FWHM2": 0.02146, "err2": 0.00013, "w01": -1.112e-09, "w02": 1.517e-10},
    {"Index": 50, "FWHM1": 0.1866, "err1": 0.0025, "FWHM2": 0.02348, "err2": 0.00012, "w01": -4.808e-10, "w02": 4.366e-11},
    {"Index": 51, "FWHM1": 0.155, "err1": 0.0028, "FWHM2": 0.02002, "err2": 0.00016, "w01": 1.466e-09, "w02": -2.223e-10},
    {"Index": 52, "FWHM1": 0.131, "err1": 0.0027, "FWHM2": 0.01781, "err2": 0.00019, "w01": 1.771e-09, "w02": -3.151e-10},
    {"Index": 53, "FWHM1": 0.139, "err1": 0.0026, "FWHM2": 0.01895, "err2": 0.00018, "w01": -1.599e-09, "w02": 2.676e-10},
    {"Index": 54, "FWHM1": 0.1366, "err1": 0.0028, "FWHM2": 0.01838, "err2": 0.00019, "w01": 1.742e-09, "w02": -2.981e-10},
    {"Index": 55, "FWHM1": 0.1374, "err1": 0.0026, "FWHM2": 0.01881, "err2": 0.00018, "w01": 1.598e-09, "w02": -2.723e-10},
    {"Index": 56, "FWHM1": 0.1862, "err1": 0.0023, "FWHM2": 0.02361, "err2": 0.00011, "w01": 5.432e-10, "w02": -5.381e-11},
    {"Index": 57, "FWHM1": 0.1434, "err1": 0.0026, "FWHM2": 0.01951, "err2": 0.00017, "w01": -1.543e-09, "w02": 2.528e-10},
    {"Index": 58, "FWHM1": 0.173, "err1": 0.0023, "FWHM2": 0.02245, "err2": 0.00012, "w01": -9.56e-10, "w02": 1.223e-10},
    {"Index": 59, "FWHM1": 0.1459, "err1": 0.0026, "FWHM2": 0.01979, "err2": 0.00017, "w01": 1.5e-09, "w02": -2.411e-10},
    {"Index": 60, "FWHM1": 0.1631, "err1": 0.0023, "FWHM2": 0.02153, "err2": 0.00013, "w01": 1.211e-09, "w02": -1.738e-10},
    {"Index": 61, "FWHM1": 0.1447, "err1": 0.0026, "FWHM2": 0.0197, "err2": 0.00017, "w01": 1.517e-09, "w02": -2.48e-10},
    {"Index": 62, "FWHM1": 0.1485, "err1": 0.0029, "FWHM2": 0.01955, "err2": 0.00017, "w01": 1.603e-09, "w02": -2.575e-10},
    {"Index": 63, "FWHM1": 0.1469, "err1": 0.0029, "FWHM2": 0.01938, "err2": 0.00018, "w01": 1.612e-09, "w02": -2.596e-10},
    {"Index": 64, "FWHM1": 0.1458, "err1": 0.0025, "FWHM2": 0.01977, "err2": 0.00017, "w01": -1.566e-09, "w02": 2.58e-10},
    {"Index": 65, "FWHM1": 0.1381, "err1": 0.0027, "FWHM2": 0.01879, "err2": 0.00018, "w01": 1.669e-09, "w02": -2.868e-10},
    {"Index": 66, "FWHM1": 0.1838, "err1": 0.0024, "FWHM2": 0.02335, "err2": 0.00011, "w01": 6.666e-10, "w02": -7.532e-11},
    {"Index": 67, "FWHM1": 0.1879, "err1": 0.0023, "FWHM2": 0.02393, "err2": 0.00011, "w01": -5.234e-10, "w02": 5.055e-11},
    {"Index": 68, "FWHM1": 0.1884, "err1": 0.0023, "FWHM2": 0.02392, "err2": 0.00011, "w01": 5.513e-10, "w02": -5.615e-11},
    {"Index": 69, "FWHM1": 0.1432, "err1": 0.0026, "FWHM2": 0.01938, "err2": 0.00017, "w01": 1.627e-09, "w02": -2.732e-10},
    {"Index": 70, "FWHM1": 0.1853, "err1": 0.0024, "FWHM2": 0.02356, "err2": 0.00012, "w01": -6.102e-10, "w02": 6.585e-11},
    {"Index": 71, "FWHM1": 0.1757, "err1": 0.0026, "FWHM2": 0.02233, "err2": 0.00013, "w01": -9.264e-10, "w02": 1.199e-10},
    {"Index": 72, "FWHM1": 0.1771, "err1": 0.0023, "FWHM2": 0.02287, "err2": 0.00012, "w01": -9.031e-10, "w02": 1.153e-10},
    {"Index": 73, "FWHM1": 0.1524, "err1": 0.0025, "FWHM2": 0.02056, "err2": 0.00016, "w01": -1.379e-09, "w02": 2.119e-10},
    {"Index": 74, "FWHM1": 0.1726, "err1": 0.0027, "FWHM2": 0.02195, "err2": 0.00014, "w01": 1.06e-09, "w02": -1.43e-10},
    {"Index": 75, "FWHM1": 0.1469, "err1": 0.0027, "FWHM2": 0.01959, "err2": 0.00017, "w01": -1.637e-09, "w02": 2.692e-10},
    {"Index": 76, "FWHM1": 0.1464, "err1": 0.0026, "FWHM2": 0.01988, "err2": 0.00017, "w01": 1.764e-09, "w02": -2.62e-10},
    {"Index": 77, "FWHM1": 0.1883, "err1": 0.0024, "FWHM2": 0.02396, "err2": 0.00012, "w01": -5.25e-10, "w02": 5.221e-11},
    {"Index": 78, "FWHM1": 0.1899, "err1": 0.0023, "FWHM2": 0.02411, "err2": 0.00011, "w01": 4.88e-10, "w02": -4.667e-11},
    {"Index": 79, "FWHM1": 0.1425, "err1": 0.0026, "FWHM2": 0.01926, "err2": 0.00018, "w01": 1.636e-09, "w02": -2.75e-10},
    {"Index": 80, "FWHM1": 0.1516, "err1": 0.0027, "FWHM2": 0.01998, "err2": 0.00017, "w01": -1.551e-09, "w02": 2.46e-10},
    {"Index": 81, "FWHM1": 0.1788, "err1": 0.0025, "FWHM2": 0.0227, "err2": 0.00013, "w01": 8.448e-10, "w02": -1.054e-10},
    {"Index": 82, "FWHM1": 0.1862, "err1": 0.0022, "FWHM2": 0.02364, "err2": 0.00011, "w01": 6.91e-10, "w02": -7.783e-11},
    {"Index": 83, "FWHM1": 0.1573, "err1": 0.0024, "FWHM2": 0.02109, "err2": 0.00014, "w01": 1.337e-09, "w02": -2.041e-10},
    {"Index": 84, "FWHM1": 0.1731, "err1": 0.0027, "FWHM2": 0.02196, "err2": 0.00014, "w01": 1.065e-09, "w02": -1.437e-10},
    {"Index": 85, "FWHM1": 0.1727, "err1": 0.0026, "FWHM2": 0.02194, "err2": 0.00013, "w01": 1.078e-09, "w02": -1.462e-10},
    {"Index": 86, "FWHM1": 0.1575, "err1": 0.0025, "FWHM2": 0.02102, "err2": 0.00016, "w01": -1.351e-09, "w02": 2.13e-10},
    {"Index": 87, "FWHM1": 0.1669, "err1": 0.0028, "FWHM2": 0.02118, "err2": 0.00015, "w01": 1.251e-09, "w02": -1.792e-10},
    {"Index": 88, "FWHM1": 0.1505, "err1": 0.0027, "FWHM2": 0.0198, "err2": 0.00017, "w01": 1.59e-09, "w02": -2.558e-10},
    {"Index": 89, "FWHM1": 0.1909, "err1": 0.0024, "FWHM2": 0.02417, "err2": 0.00011, "w01": 4.867e-10, "w02": -4.478e-11},
    {"Index": 90, "FWHM1": 0.1779, "err1": 0.0022, "FWHM2": 0.02284, "err2": 0.00011, "w01": 9.3e-10, "w02": -1.189e-10},
    {"Index": 91, "FWHM1": 0.1632, "err1": 0.0028, "FWHM2": 0.02082, "err2": 0.00015, "w01": 1.378e-09, "w02": -2.023e-10},
    {"Index": 92, "FWHM1": 0.1545, "err1": 0.0026, "FWHM2": 0.02024, "err2": 0.00016, "w01": -1.532e-09, "w02": 2.394e-10},
    {"Index": 93, "FWHM1": 0.1889, "err1": 0.0023, "FWHM2": 0.02399, "err2": 0.00011, "w01": -5.241e-10, "w02": 5.256e-11},
    {"Index": 94, "FWHM1": 0.1848, "err1": 0.0023, "FWHM2": 0.02339, "err2": 0.00011, "w01": 6.788e-10, "w02": -7.644e-11},
    {"Index": 95, "FWHM1": 0.1762, "err1": 0.0026, "FWHM2": 0.02218, "err2": 0.00013, "w01": -1.004e-09, "w02": 1.315e-10},
    {"Index": 96, "FWHM1": 0.162, "err1": 0.0023, "FWHM2": 0.02148, "err2": 0.00014, "w01": 1.241e-09, "w02": -1.821e-10},
    {"Index": 97, "FWHM1": 0.1569, "err1": 0.0023, "FWHM2": 0.02105, "err2": 0.00014, "w01": -1.341e-09, "w02": 2.05e-10},
    {"Index": 98, "FWHM1": 0.1624, "err1": 0.0027, "FWHM2": 0.02062, "err2": 0.00015, "w01": -1.407e-09, "w02": 2.083e-10},
    {"Index": 99, "FWHM1": 0.1609, "err1": 0.0025, "FWHM2": 0.02067, "err2": 0.00014, "w01": -1.395e-09, "w02": 2.1e-10},
    {"Index": 100, "FWHM1": 0.1901, "err1": 0.0023, "FWHM2": 0.02382, "err2": 0.00011, "w01": -5.988e-10, "w02": 6.376e-11},
    {"Index": 101, "FWHM1": 0.1708, "err1": 0.0022, "FWHM2": 0.02219, "err2": 0.00012, "w01": 1.116e-09, "w02": -1.533e-10},
    {"Index": 102, "FWHM1": 0.1871, "err1": 0.0023, "FWHM2": 0.0237, "err2": 0.00011, "w01": 5.987e-10, "w02": -6.399e-11},
    {"Index": 103, "FWHM1": 0.1901, "err1": 0.0023, "FWHM2": 0.02406, "err2": 0.00011, "w01": 5.042e-10, "w02": -4.882e-11},
    {"Index": 104, "FWHM1": 0.182, "err1": 0.0023, "FWHM2": 0.02301, "err2": 0.00011, "w01": 7.74e-10, "w02": -9.247e-11},
    {"Index": 105, "FWHM1": 0.16, "err1": 0.0026, "FWHM2": 0.02045, "err2": 0.00015, "w01": -1.448e-09, "w02": 2.194e-10},
    {"Index": 106, "FWHM1": 0.1732, "err1": 0.0024, "FWHM2": 0.0217, "err2": 0.00013, "w01": 1.116e-09, "w02": -1.502e-10},
    {"Index": 107, "FWHM1": 0.1829, "err1": 0.0023, "FWHM2": 0.02301, "err2": 0.00011, "w01": -7.999e-10, "w02": 9.405e-11},
    {"Index": 108, "FWHM1": 0.1669, "err1": 0.0026, "FWHM2": 0.02086, "err2": 0.00014, "w01": 1.317e-09, "w02": -1.883e-10},
    {"Index": 109, "FWHM1": 0.1645, "err1": 0.0023, "FWHM2": 0.02164, "err2": 0.00013, "w01": 1.222e-09, "w02": -1.754e-10},
    {"Index": 110, "FWHM1": 0.1876, "err1": 0.0023, "FWHM2": 0.0238, "err2": 0.00011, "w01": -5.794e-10, "w02": 5.938e-11},
    {"Index": 111, "FWHM1": 0.19, "err1": 0.0023, "FWHM2": 0.02358, "err2": 0.00011, "w01": 6.47e-10, "w02": -6.956e-11},
    {"Index": 112, "FWHM1": 0.1787, "err1": 0.0022, "FWHM2": 0.02263, "err2": 0.00011, "w01": -9.548e-10, "w02": 1.223e-10},
    {"Index": 113, "FWHM1": 0.1884, "err1": 0.0023, "FWHM2": 0.02347, "err2": 0.00011, "w01": 7.072e-10, "w02": -7.917e-11},
    {"Index": 114, "FWHM1": 0.1907, "err1": 0.0022, "FWHM2": 0.02401, "err2": 0.00011, "w01": -5.314e-10, "w02": 5.22e-11},
    {"Index": 115, "FWHM1": 0.1698, "err1": 0.0022, "FWHM2": 0.02201, "err2": 0.00012, "w01": 1.201e-09, "w02": -1.668e-10},
    {"Index": 116, "FWHM1": 0.1588, "err1": 0.0025, "FWHM2": 0.02048, "err2": 0.00014, "w01": -1.461e-09, "w02": 2.222e-10},
    {"Index": 117, "FWHM1": 0.1797, "err1": 0.0023, "FWHM2": 0.02229, "err2": 0.00012, "w01": -9.684e-10, "w02": 1.216e-10},
    {"Index": 118, "FWHM1": 0.1818, "err1": 0.0024, "FWHM2": 0.02283, "err2": 0.00012, "w01": 8.162e-10, "w02": -9.608e-11},
    {"Index": 119, "FWHM1": 0.159, "err1": 0.0025, "FWHM2": 0.02043, "err2": 0.00014, "w01": 1.492e-09, "w02": -2.267e-10},
    {"Index": 120, "FWHM1": 0.1823, "err1": 0.0022, "FWHM2": 0.02261, "err2": 0.00011, "w01": 9.114e-10, "w02": -1.125e-10},
    {"Index": 121, "FWHM1": 0.1868, "err1": 0.0021, "FWHM2": 0.02349, "err2": 0.0001, "w01": 6.525e-10, "w02": -7.154e-11},
    {"Index": 122, "FWHM1": 0.1701, "err1": 0.0025, "FWHM2": 0.02103, "err2": 0.00013, "w01": -1.273e-09, "w02": 1.779e-10},
    {"Index": 123, "FWHM1": 0.1814, "err1": 0.0023, "FWHM2": 0.0225, "err2": 0.00011, "w01": 9.06e-10, "w02": -1.121e-10},
    {"Index": 124, "FWHM1": 0.1867, "err1": 0.0022, "FWHM2": 0.02294, "err2": 0.00011, "w01": -8.184e-10, "w02": 9.554e-11},
    {"Index": 125, "FWHM1": 0.1893, "err1": 0.0022, "FWHM2": 0.02377, "err2": 0.00011, "w01": -5.906e-10, "w02": 5.927e-11},
    {"Index": 126, "FWHM1": 0.1874, "err1": 0.0021, "FWHM2": 0.02358, "err2": 0.0001, "w01": 6.332e-10, "w02": -6.721e-11},
    {"Index": 127, "FWHM1": 0.1931, "err1": 0.0023, "FWHM2": 0.02379, "err2": 0.00011, "w01": 5.878e-10, "w02": -5.948e-11},
    {"Index": 128, "FWHM1": 0.1899, "err1": 0.0022, "FWHM2": 0.02397, "err2": 0.00011, "w01": 5.388e-10, "w02": -5.215e-11},
    {"Index": 129, "FWHM1": 0.1804, "err1": 0.0021, "FWHM2": 0.02247, "err2": 0.00011, "w01": -9.839e-10, "w02": 1.253e-10},
    {"Index": 130, "FWHM1": 0.1791, "err1": 0.0021, "FWHM2": 0.0224, "err2": 0.00011, "w01": -1.02e-09, "w02": 1.326e-10},
    {"Index": 131, "FWHM1": 0.177, "err1": 0.0023, "FWHM2": 0.02168, "err2": 0.00012, "w01": 1.126e-09, "w02": -1.491e-10},
    {"Index": 132, "FWHM1": 0.1895, "err1": 0.0022, "FWHM2": 0.02314, "err2": 0.0001, "w01": -7.839e-10, "w02": 9.04e-11},
    {"Index": 133, "FWHM1": 0.1796, "err1": 0.0023, "FWHM2": 0.02196, "err2": 0.00011, "w01": 1.042e-09, "w02": -1.342e-10},
    {"Index": 134, "FWHM1": 0.1738, "err1": 0.0022, "FWHM2": 0.0222, "err2": 0.00011, "w01": -1.11e-09, "w02": 1.494e-10},
    {"Index": 135, "FWHM1": 0.1714, "err1": 0.0021, "FWHM2": 0.02211, "err2": 0.00012, "w01": -1.151e-09, "w02": 1.576e-10},
    {"Index": 136, "FWHM1": 0.156, "err1": 0.0022, "FWHM2": 0.02086, "err2": 0.00013, "w01": 1.35e-09, "w02": -2.061e-10},
    {"Index": 137, "FWHM1": 0.1847, "err1": 0.0022, "FWHM2": 0.02277, "err2": 0.00011, "w01": 8.581e-10, "w02": -1.01e-10},
    {"Index": 138, "FWHM1": 0.1892, "err1": 0.0022, "FWHM2": 0.0238, "err2": 0.0001, "w01": 5.707e-10, "w02": -5.778e-11},
    {"Index": 139, "FWHM1": 0.1915, "err1": 0.0022, "FWHM2": 0.02321, "err2": 0.0001, "w01": -7.589e-10, "w02": 8.639e-11},
    {"Index": 140, "FWHM1": 0.1855, "err1": 0.0022, "FWHM2": 0.02298, "err2": 0.0001, "w01": -7.92e-10, "w02": 9.073e-11},
    {"Index": 141, "FWHM1": 0.1911, "err1": 0.0022, "FWHM2": 0.02312, "err2": 0.0001, "w01": 7.763e-10, "w02": -8.877e-11},
    {"Index": 142, "FWHM1": 0.193, "err1": 0.0023, "FWHM2": 0.02414, "err2": 0.00011, "w01": 5.042e-10, "w02": -4.662e-11},
    {"Index": 143, "FWHM1": 0.1867, "err1": 0.0021, "FWHM2": 0.02271, "err2": 0.0001, "w01": 9.389e-10, "w02": -1.16e-10},
    {"Index": 144, "FWHM1": 0.171, "err1": 0.0022, "FWHM2": 0.02133, "err2": 0.00011, "w01": 1.29e-09, "w02": -1.811e-10},
    {"Index": 145, "FWHM1": 0.1861, "err1": 0.0021, "FWHM2": 0.02267, "err2": 0.0001, "w01": 9.508e-10, "w02": -1.181e-10},
    {"Index": 146, "FWHM1": 0.1623, "err1": 0.0021, "FWHM2": 0.02122, "err2": 0.00012, "w01": -1.32e-09, "w02": 1.947e-10},
    {"Index": 147, "FWHM1": 0.162, "err1": 0.0021, "FWHM2": 0.02121, "err2": 0.00012, "w01": -1.323e-09, "w02": 1.955e-10},
    {"Index": 148, "FWHM1": 0.1951, "err1": 0.0022, "FWHM2": 0.0239, "err2": 0.0001, "w01": 5.842e-10, "w02": -5.77e-11},
    {"Index": 149, "FWHM1": 0.1604, "err1": 0.0021, "FWHM2": 0.02111, "err2": 0.00013, "w01": -1.311e-09, "w02": 1.95e-10},
    {"Index": 150, "FWHM1": 0.1906, "err1": 0.0021, "FWHM2": 0.02286, "err2": 0.0001, "w01": 9.155e-10, "w02": -1.123e-10}
]

df = pd.DataFrame(data)

In [ ]:
plt.figure(figsize=(singlecol, singlecol),dpi=200)
plt.rcParams.update({'lines.markersize':3})
i=10
x=q/10
plt.errorbar(x[i:], df['FWHM1'][i:], yerr=df['err1'][i:], 
             fmt='o', ecolor='red', capsize=5, linestyle='-', 
             label='FWHM1 with Error Bars')

plt.errorbar(x[i:], df['FWHM2'][i:], yerr=df['err2'][i:], 
             fmt='s', ecolor='orange', capsize=5, linestyle='--', 
             label='FWHM2 with Error Bars')

# Formatting
#plt.title('FWHM1 vs Index')
plt.xlabel('q (1/A)')
plt.ylabel('FWHM (meV)')
#plt.ylim(0,0.15)
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

In [ ]:
data = [
    {"Index": 0, "FWHM1": 0.01407, "error1": 0.00012, "FWHM2": 0.1091, "error2": 0.002, "omega1": -2.01e-10, "omega2": 1.28e-09},
    {"Index": 1, "FWHM1": 0.01517, "error1": 9.10e-05, "FWHM2": 0.1159, "error2": 0.0016, "omega1": -1.28e-10, "omega2": 9.10e-10},
    {"Index": 2, "FWHM1": 0.01477, "error1": 9.50e-05, "FWHM2": 0.1122, "error2": 0.0016, "omega1": -1.59e-10, "omega2": 1.07e-09},
    {"Index": 3, "FWHM1": 0.0146, "error1": 9.90e-05, "FWHM2": 0.1106, "error2": 0.0017, "omega1": 1.68e-10, "omega2": -1.12e-09},
    {"Index": 4, "FWHM1": 0.01567, "error1": 8.10e-05, "FWHM2": 0.1209, "error2": 0.0015, "omega1": -9.29e-11, "omega2": 7.48e-10},
    {"Index": 5, "FWHM1": 0.01198, "error1": 0.00014, "FWHM2": 0.08841, "error2": 0.0019, "omega1": -3.37e-10, "omega2": 1.78e-09},
    {"Index": 6, "FWHM1": 0.01487, "error1": 9.60e-05, "FWHM2": 0.1134, "error2": 0.0017, "omega1": -1.45e-10, "omega2": 1.02e-09},
    {"Index": 7, "FWHM1": 0.01322, "error1": 0.00012, "FWHM2": 0.09806, "error2": 0.0018, "omega1": 2.55e-10, "omega2": -1.50e-09},
    {"Index": 8, "FWHM1": 0.01405, "error1": 0.00011, "FWHM2": 0.11, "error2": 0.002, "omega1": 1.25e-09, "omega2": -1.91e-10},
    {"Index": 9, "FWHM1": 0.01632, "error1": 9.00e-05, "FWHM2": 0.1289, "error2": 0.0019, "omega1": -2.90e-11, "omega2": 4.12e-10},
    {"Index": 10, "FWHM1": 0.01545, "error1": 8.30e-05, "FWHM2": 0.12, "error2": 0.0016, "omega1": -1.03e-10, "omega2": 7.95e-10},
    {"Index": 11, "FWHM1": 0.0159, "error1": 8.10e-05, "FWHM2": 0.1253, "error2": 0.0016, "omega1": -5.29e-10, "omega2": 5.57e-11},
    {"Index": 12, "FWHM1": 0.01597, "error1": 8.00e-05, "FWHM2": 0.1258, "error2": 0.0016, "omega1": 5.06e-10, "omega2": -5.20e-11},
    {"Index": 13, "FWHM1": 0.01607, "error1": 8.10e-05, "FWHM2": 0.1268, "error2": 0.0017, "omega1": 5.04e-10, "omega2": -5.10e-11},
    {"Index": 14, "FWHM1": 0.01607, "error1": 7.80e-05, "FWHM2": 0.127, "error2": 0.0016, "omega1": -5.18e-10, "omega2": 5.27e-11},
    {"Index": 15, "FWHM1": 0.01282, "error1": 0.00012, "FWHM2": 0.09658, "error2": 0.0018, "omega1": -1.65e-09, "omega2": 2.85e-10},
    {"Index": 16, "FWHM1": 0.0161, "error1": 8.00e-05, "FWHM2": 0.1279, "error2": 0.0017, "omega1": -4.80e-11, "omega2": 4.90e-10},
    {"Index": 17, "FWHM1": 0.01331, "error1": 0.00011, "FWHM2": 0.09869, "error2": 0.0017, "omega1": -2.55e-10, "omega2": 1.52e-09},
    {"Index": 18, "FWHM1": 0.01558, "error1": 7.60e-05, "FWHM2": 0.1234, "error2": 0.0015, "omega1": -7.62e-11, "omega2": 6.62e-10},
    {"Index": 19, "FWHM1": 0.01615, "error1": 7.80e-05, "FWHM2": 0.1286, "error2": 0.0016, "omega1": -4.35e-11, "omega2": 4.61e-10},
    {"Index": 20, "FWHM1": 0.01546, "error1": 7.60e-05, "FWHM2": 0.1227, "error2": 0.0015, "omega1": -8.11e-10, "omega2": 9.96e-11},
    {"Index": 21, "FWHM1": 0.01602, "error1": 7.40e-05, "FWHM2": 0.1275, "error2": 0.0015, "omega1": 5.22e-11, "omega2": -5.25e-10},
    {"Index": 22, "FWHM1": 0.01594, "error1": 7.50e-05, "FWHM2": 0.1264, "error2": 0.0016, "omega1": 5.32e-10, "omega2": -5.33e-11},
    {"Index": 23, "FWHM1": 0.0159, "error1": 7.50e-05, "FWHM2": 0.1282, "error2": 0.0016, "omega1": -6.46e-11, "omega2": 6.23e-10},
    {"Index": 24, "FWHM1": 0.01547, "error1": 7.30e-05, "FWHM2": 0.1231, "error2": 0.0015, "omega1": 7.10e-10, "omega2": -8.22e-11},
    {"Index": 25, "FWHM1": 0.01464, "error1": 8.20e-05, "FWHM2": 0.1131, "error2": 0.0015, "omega1": 1.16e-09, "omega2": -1.63e-10},
    {"Index": 26, "FWHM1": 0.0143, "error1": 8.20e-05, "FWHM2": 0.1156, "error2": 0.0016, "omega1": -1.79e-10, "omega2": 1.27e-09},
    {"Index": 27, "FWHM1": 0.0146, "error1": 7.60e-05, "FWHM2": 0.1208, "error2": 0.0016, "omega1": 1.11e-09, "omega2": -1.47e-10},
    {"Index": 28, "FWHM1": 0.01473, "error1": 7.80e-05, "FWHM2": 0.1147, "error2": 0.0014, "omega1": -1.15e-09, "omega2": 1.61e-10},
    {"Index": 29, "FWHM1": 0.01546, "error1": 6.80e-05, "FWHM2": 0.1252, "error2": 0.0014, "omega1": -8.24e-11, "omega2": 7.29e-10},
    {"Index": 30, "FWHM1": 0.01597, "error1": 7.00e-05, "FWHM2": 0.1272, "error2": 0.0015, "omega1": -5.53e-10, "omega2": 5.47e-11}
]
df = pd.DataFrame(data)

In [ ]:
plt.figure(figsize=(singlecol, singlecol),dpi=200)
plt.errorbar(df['Index'], df['FWHM1'], yerr=df['error1'], 
             fmt='o', ecolor='red', capsize=5, linestyle='-', 
             label='FWHM1 with Error Bars')

plt.errorbar(df['Index'], df['FWHM2'], yerr=df['error2'], 
             fmt='s', ecolor='orange', capsize=5, linestyle='--', 
             label='FWHM2 with Error Bars')

# Formatting
#plt.title('FWHM1 vs Index')
plt.xlabel('Index')
plt.ylabel('FWHM (meV)')
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

In [ ]:
data = [
    {"Index": 1, "FWHM1": 0.4569, "err1": 0.42, "FWHM2": 0.009121, "err2": 0.000046, "w01": -2.937e-07, "w02": -4.825e-10},
    {"Index": 2, "FWHM1": 0.2587, "err1": 0.088, "FWHM2": 0.009871, "err2": 0.000045, "w01": -4.618e-08, "w02": 3.579e-10},
    {"Index": 3, "FWHM1": 0.1722, "err1": 0.03, "FWHM2": 0.01055, "err2": 0.000046, "w01": -2.348e-08, "w02": 2.664e-10},
    {"Index": 4, "FWHM1": 0.1404, "err1": 0.015, "FWHM2": 0.01163, "err2": 0.000046, "w01": -1.111e-08, "w02": 1.869e-10},
    {"Index": 5, "FWHM1": 0.0868, "err1": 0.0052, "FWHM2": 0.01116, "err2": 0.000053, "w01": 3.59e-09, "w02": -1.329e-10},
    {"Index": 6, "FWHM1": 0.09766, "err1": 0.005, "FWHM2": 0.01317, "err2": 0.000049, "w01": 2.63e-09, "w02": -9.407e-11},
    {"Index": 7, "FWHM1": 0.06535, "err1": 0.0022, "FWHM2": 0.01199, "err2": 0.000063, "w01": 1.043e-09, "w02": -8.296e-11},
    {"Index": 8, "FWHM1": 0.0912, "err1": 0.0027, "FWHM2": 0.01484, "err2": 0.000048, "w01": -9.653e-10, "w02": 5.44e-11},
    {"Index": 9, "FWHM1": 0.07082, "err1": 0.0017, "FWHM2": 0.01444, "err2": 0.00006, "w01": -2.243e-10, "w02": 2.876e-11},
    {"Index": 10, "FWHM1": 0.09274, "err1": 0.0019, "FWHM2": 0.01758, "err2": 0.000048, "w01": 5.386e-11, "w02": -1.099e-11},
    {"Index": 11, "FWHM1": 0.06289, "err1": 0.001, "FWHM2": 0.01483, "err2": 0.000073, "w01": -4.977e-11, "w02": -1.353e-11},
    {"Index": 12, "FWHM1": 0.06847, "err1": 0.001, "FWHM2": 0.01603, "err2": 0.000069, "w01": 4.526e-10, "w02": -4.105e-11},
    {"Index": 13, "FWHM1": 0.09248, "err1": 0.0013, "FWHM2": 0.01882, "err2": 0.00005, "w01": 2.073e-10, "w02": -6.172e-12},
    {"Index": 14, "FWHM1": 0.09552, "err1": 0.0013, "FWHM2": 0.01929, "err2": 0.000049, "w01": 8.079e-11, "w02": 5.454e-12},
    {"Index": 15, "FWHM1": 0.08072, "err1": 0.0011, "FWHM2": 0.01776, "err2": 0.000066, "w01": 0.0004758, "w02": -7.979e-05},
    {"Index": 16, "FWHM1": 0.102, "err1": 0.0013, "FWHM2": 0.01978, "err2": 0.000049, "w01": -9.395e-11, "w02": -4.118e-12},
    {"Index": 17, "FWHM1": 0.07931, "err1": 0.00094, "FWHM2": 0.01772, "err2": 0.00007, "w01": -5.995e-10, "w02": 6.579e-11},
    {"Index": 18, "FWHM1": 0.09201, "err1": 0.00099, "FWHM2": 0.01872, "err2": 0.000055, "w01": -5.717e-10, "w02": 4.911e-11},
    {"Index": 19, "FWHM1": 0.1046, "err1": 0.0012, "FWHM2": 0.0194, "err2": 0.000051, "w01": -2.286e-10, "w02": 8.377e-12},
    {"Index": 20, "FWHM1": 0.09011, "err1": 0.0011, "FWHM2": 0.01647, "err2": 0.000057, "w01": 1.135e-10, "w02": 1.033e-11},
    {"Index": 21, "FWHM1": 0.0959, "err1": 0.001, "FWHM2": 0.0179, "err2": 0.000052, "w01": 4.341e-10, "w02": -3.247e-11},
    {"Index": 22, "FWHM1": 0.08383, "err1": 0.00099, "FWHM2": 0.01612, "err2": 0.000065, "w01": -6.806e-10, "w02": 7.178e-11},
    {"Index": 23, "FWHM1": 0.08592, "err1": 0.00099, "FWHM2": 0.01493, "err2": 0.000057, "w01": -6.871e-11, "w02": -2.437e-11},
    {"Index": 24, "FWHM1": 0.09693, "err1": 0.0011, "FWHM2": 0.01677, "err2": 0.000051, "w01": -4.688e-10, "w02": 3.091e-11},
    {"Index": 25, "FWHM1": 0.1054, "err1": 0.0012, "FWHM2": 0.01806, "err2": 0.00005, "w01": 2.346e-11, "w02": 2.275e-11},
    {"Index": 26, "FWHM1": 0.08496, "err1": 0.0013, "FWHM2": 0.01159, "err2": 0.000061, "w01": -4.267e-11, "w02": -3.93e-11},
    {"Index": 27, "FWHM1": 0.08498, "err1": 0.0013, "FWHM2": 0.01106, "err2": 0.000061, "w01": 1.812e-11, "w02": 4.85e-11},
    {"Index": 28, "FWHM1": 0.1063, "err1": 0.0011, "FWHM2": 0.0173, "err2": 0.000047, "w01": 4.493e-10, "w02": -2.577e-11},
    {"Index": 29, "FWHM1": 0.1007, "err1": 0.0012, "FWHM2": 0.01643, "err2": 0.00005, "w01": -5.772e-10, "w02": 3.839e-11},
    {"Index": 30, "FWHM1": 0.08853, "err1": 0.0012, "FWHM2": 0.01422, "err2": 0.00006, "w01": 1.083e-10, "w02": 2.535e-11}
]

import pandas as pd
df = pd.DataFrame(data)

In [ ]:
x.shape

In [ ]:
plt.figure(figsize=(singlecol, singlecol),dpi=200)
plt.errorbar(df['Index'][5:], df['FWHM1'][5:], yerr=df['err1'][5:], 
             fmt='o', ecolor='red', capsize=5, linestyle='-', 
             label='FWHM1 with Error Bars')

plt.errorbar(df['Index'][5:], df['FWHM2'][5:], yerr=df['err2'][5:], 
             fmt='s', ecolor='orange', capsize=5, linestyle='--', 
             label='FWHM2 with Error Bars')

# Formatting
#plt.title('FWHM1 vs Index')
plt.xlabel('Index')
plt.ylabel('FWHM (meV)')
plt.tight_layout()
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
plt.figure(figsize=(6,4))
gama_ls=[]
# ------- Lorentzian model (center fixed at zero) -------
def lorentz(w, A, gamma):
    return (A/np.pi) * (gamma / (w**2 + gamma**2))

for i in range(Na_disf.shape[0]):
    S = Na_disf[i]
# ------- Fit the data -------
    popt, pcov = curve_fit(lorentz, omega, S, p0=[np.max(S), 0.1])
    
    A_fit, gamma_fit = popt
    FWHM = 2 * gamma_fit
    
    print("Fitted Lorentzian parameters:")
    print(f"A     = {A_fit}")
    print(f"Gamma = {gamma_fit}")
    print(f"FWHM  = {FWHM}")
    gama_ls.append(gamma_fit)
    # ------- Plot -------
    plt.scatter(omega, S, s=10, color="k", label="Data")
    plt.plot(omega, lorentz(omega, A_fit, gamma_fit),
             linewidth=2, label="Lorentz Fit")
    plt.xlabel("Energy transfer ω")
    plt.ylabel("S(Q, ω)")
    plt.title("Lorentz-Only Fit to QENS Spectrum")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.xlim(-3,3)
    #plt.yscale('log')
    plt.show()
    


In [ ]:
plt.plot(gama_ls[15:])

In [ ]:
Nq = Na_disf.shape[0]
results = []
for iq in range(Nq):
    S = O_disf[iq]
    popt, pcov, derived = fit_spectrum(omega, S, p0=None, bounds=None, use_resolution=False)
    results.append(derived)

Gammas = np.array([r['Gamma'] for r in results])
FWHMs = np.array([r['FWHM'] for r in results])
taus = np.array([r['tau_ps'] for r in results])

# plot FWHM vs Q (you need Q array)
plt.figure()
plt.plot(q/10, FWHMs, 'o-')
plt.xlabel('Q (Å⁻¹)')
plt.ylabel('FWHM (meV)')
plt.grid(True)
plt.show()

In [ ]:
plt.plot(q/10,taus)

In [ ]:
Nq = Na_disf.shape[0]
results = []
for iq in range(Nq):
    S = Na_disf[iq]
    popt, pcov, derived = fit_spectrum(omega, S, p0=None, bounds=None, use_resolution=False)
    results.append(derived)

Gammas = np.array([r['Gamma'] for r in results])
FWHMs = np.array([r['FWHM'] for r in results])
taus = np.array([r['tau_ps'] for r in results])

# plot FWHM vs Q (you need Q array)
plt.figure()
plt.plot(q/10, FWHMs, 'o-')
plt.xlabel('Q (Å⁻¹)')
plt.ylabel('FWHM (meV)')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# -------- Lorentzian Model --------
def lorentz(w, A, gamma, w0):
    return (A/np.pi) * (gamma / ((w - w0)**2 + gamma**2))

# -------- Example data (replace with your data) --------
# w  = frequency transfer (in meV or THz)
# S  = dynamic structure factor at fixed Q
# w, S = your_arrays_here
# Example:
# w = np.linspace(-1, 1, 200)
# S = lorentz(w, 1.0, 0.05, 0) + 0.02*np.random.randn(len(w))

# -------- Fit --------
popt, pcov = curve_fit(lorentz, omega, S[0], p0=[1.0, 0.005, 0.0])
A_fit, gamma_fit, w0_fit = popt

FWHM = 2 * gamma_fit

print("Fitted parameters:")
print(f"A     = {A_fit}")
print(f"Gamma = {gamma_fit}")
print(f"ω0    = {w0_fit}")
print(f"FWHM  = {FWHM}")

# -------- Plot --------
plt.figure(figsize=(6,4))
plt.scatter(w, S, s=10, label="Data", alpha=0.6)
plt.plot(w, lorentz(w, *popt), linewidth=2, label="Lorentzian fit")
plt.xlabel("Energy transfer ω")
plt.ylabel("S(Q, ω)")
plt.title("Lorentzian Fit to QENS Spectrum")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


### f(q,t)

In [ ]:
file_path = "./hydration/QENS/400K_vac/400K_12_DISF.mda"

with h5py.File(file_path, "r") as f:
    total_F = f["disf/f(q,t)/total"][:]
    Na_F = f["disf/f(q,t)/Na"][:]
    O_F = f["disf/f(q,t)/O"][:]
    H_F = f["disf/f(q,t)/H"][:]
    Fe_F = f["disf/f(q,t)/Fe"][:]
    Mn_F = f["disf/f(q,t)/Mn"][:]
    C_F = f["disf/f(q,t)/C"][:]
    N_F = f["disf/f(q,t)/N"][:]

    q = f["disf/axes/q"][:]
    time = f["disf/axes/time"][:]

    weight = f["metadata/inputs/weights"]

In [ ]:
time

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_F, "Total"),
    (Na_F,  "Na"),
    (H_F,  "H"),
    (O_F, "O")
]

indices = range(25,190,5)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*3, singlecol*3))
axes = axes.ravel()
j=1
for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(time[j:], data[i, j:]/data[i, j], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("F(q, t)")
    ax.set_xscale('log')
    #ax.set_xlim(0,2000)
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (Fe_F,  "Fe"),
    (Mn_F,  "Mn"),
    (C_F,  "C"),
    (N_F,  "N")
]

indices = range(150)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(t, data[i, :], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("F(q, t)")
    #ax.set_xlim(0,2000)
    ax.grid(True)


plt.tight_layout()
plt.show()


### 1 Exponential fit

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

indices = []
tau1_list = []
tau1_err_list = []
A1_list = []

for i in range(25, 190, 5):
    F =  H_F[i, 1:] / H_F[i, 1:2] 
    t = time[1:]

    # ---- Model ----
    def single_exp(t, A, tau):
        return A * np.exp(-t / tau)

    # ---- Initial guesses ----
    t_min = t[t > 0].min() if np.any(t > 0) else t[1]
    t_max = t.max()

    p0 = [F[0], np.sqrt(t_min * t_max)]

    bounds_lower = [0, t_min * 0.1]
    bounds_upper = [2, t_max * 10]

    try:
        popt, pcov = curve_fit(
            single_exp,
            t, F,
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A1_fit, tau1_fit = popt
        A1_err, tau1_err = np.sqrt(np.diag(pcov))

    except Exception as e:
        print(f"Fit failed at index {i}: {e}")
        continue

    # ---- Fit curve ----
    fit_total = single_exp(t, *popt)

    # ---- Plot ----
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2),
                             gridspec_kw={'height_ratios': [2, 1]})
    ax = axes[0]
    ax.plot(t, F, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(t, fit_total, '-', lw=2.5, color='red', label=f'Exp (τ={tau1_fit:.3g})')
    ax.set_xlabel('Time t (ps)')
    ax.set_ylabel('F(q, t)')
    ax.set_xscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Single Exponential Fit — q = {round(q[i]/10, 3)}')

    ax2 = axes[1]
    ax2.plot(t, F - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Time t (ps)')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ---- Convert τ → FWHM in meV ----
    hbar = 0.6582119  # meV·ps
    FWHM1 = 2 * hbar / tau1_fit

    # ---- Collect results ----
    indices.append(i)
    tau1_list.append(tau1_fit)
    tau1_err_list.append(tau1_err)
    A1_list.append(A1_fit)

    print(f"\n{'='*50}  Q index {i}")
    print(f"Exp:  A={A1_fit:.4g}±{A1_err:.2g}  τ={tau1_fit:.4g}±{tau1_err:.2g} ps  FWHM={FWHM1:.4g} meV")

# ---- FWHM vs q plot ----
hbar = 0.6582119
FWHM1_list         = [2 * hbar / t for t in tau1_list]
FWHM1_err_list_meV = [2 * hbar * e / t**2 for e, t in zip(tau1_err_list, tau1_list)]

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='o-', color='green',
            capsize=4, label='FWHM Exp 1')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
#ax.set_ylim(0, 0.15)
ax.legend(fontsize=16, frameon=False)
plt.tight_layout()
plt.show()

### 2 Exponential fit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ============================================================
# Settings and data
# ============================================================
total_F = np.asarray(total_F, dtype=float)
time = np.asarray(time, dtype=float)

q_values = np.asarray(q, dtype=float)  # Use q / 10 if q is in nm^-1
fit_indices = range(20, 190)

tau_initial = np.array([12.0, 725.0])  # ps
hbar = 0.6582119569                    # meV·ps

assert total_F.ndim == 2
assert total_F.shape[1] == len(time)
assert total_F.shape[0] == len(q_values)

t = time[1:]
assert len(t) > 5
assert np.all(np.isfinite(t))
assert np.all(np.diff(t) > 0)

t0 = t[0]
dt = t - t0

tau_min = np.median(np.diff(t)) * 1e-6
assert np.all(tau_initial > tau_min)

# ============================================================
# Model
# ============================================================
def exponential(dt, A, tau):
    return A * np.exp(-dt / tau)


def double_exp_plateau(dt, A1, tau1, A2, tau2, C):
    return (
        exponential(dt, A1, tau1)
        + exponential(dt, A2, tau2)
        + C
    )


# ============================================================
# Storage
# ============================================================
nq = total_F.shape[0]

Tau = np.full((nq, 2), np.nan)
Tau_err = np.full((nq, 2), np.nan)

Amplitude_F = np.full((nq, 2), np.nan)
Plateau = np.full(nq, np.nan)
Plateau_err = np.full(nq, np.nan)

# Parameter order: [A1, tau1, A2, tau2, C]
fit_parameters_F = np.full((nq, 5), np.nan)
RMSE_F = np.full(nq, np.nan)

failed_indices_F = []

# ============================================================
# Fit selected q values
# ============================================================
for i in fit_indices:
    denominator = total_F[i, 1]

    if not np.isfinite(denominator) or denominator <= 0:
        failed_indices_F.append(i)
        print(f"Skipped i={i}: invalid normalization value.")
        continue

    # Requested preprocessing
    F = total_F[i, 1:] / total_F[i, 1:2]

    mask = np.isfinite(F)
    x = dt[mask]
    y = F[mask]

    if len(y) <= 5:
        failed_indices_F.append(i)
        print(f"Skipped i={i}: too few valid points.")
        continue

    # Initial plateau estimate from the final 10% of valid points.
    # This is only a starting guess; C remains free.
    n_tail = max(1, len(y) // 10)
    C0 = np.clip(np.median(y[-n_tail:]), 0.01, 0.95)
    A0 = (1.0 - C0) / 2.0

    p0 = [
        A0, tau_initial[0],
        A0, tau_initial[1],
        C0,
    ]

    try:
        popt, pcov = curve_fit(
            double_exp_plateau,
            x,
            y,
            p0=p0,
            bounds=(
                [0, tau_min, 0, tau_min, 0],
                [np.inf, np.inf, np.inf, np.inf, 1],
            ),
            method="trf",
            x_scale="jac",
            max_nfev=20000,
        )

    except (RuntimeError, ValueError) as exc:
        failed_indices_F.append(i)
        print(f"Fit failed at i={i}: {exc}")
        continue

    perr = np.sqrt(np.maximum(np.diag(pcov), 0))

    # Sort the two exponential components: fast → slow
    order = np.argsort(popt[[1, 3]])

    Tau[i] = popt[[1, 3]][order]
    Tau_err[i] = perr[[1, 3]][order]
    Amplitude_F[i] = popt[[0, 2]][order]

    Plateau[i] = popt[4]
    Plateau_err[i] = perr[4]

    fit_parameters_F[i] = [
        Amplitude_F[i, 0], Tau[i, 0],
        Amplitude_F[i, 1], Tau[i, 1],
        Plateau[i],
    ]

    residual = y - double_exp_plateau(x, *fit_parameters_F[i])
    RMSE_F[i] = np.sqrt(np.mean(residual**2))

# ============================================================
# Convert tau (ps) to Gamma (meV, HWHM)
# ============================================================
Gamma = np.full_like(Tau, np.nan)
Gamma_err = np.full_like(Tau_err, np.nan)

valid = np.isfinite(Tau) & (Tau > 0)

Gamma[valid] = hbar / Tau[valid]
Gamma_err[valid] = hbar * Tau_err[valid] / Tau[valid]**2

# Lists remain aligned with original spectrum indices.
# Unfitted or failed entries contain NaN.
Tau1_list, Tau2_list = Tau.T.tolist()
Tau1_err_list, Tau2_err_list = Tau_err.T.tolist()

Gamma1_list, Gamma2_list = Gamma.T.tolist()
Gamma1_err_list, Gamma2_err_list = Gamma_err.T.tolist()

A1_list, A2_list = Amplitude_F.T.tolist()
C_list = Plateau.tolist()
C_err_list = Plateau_err.tolist()

print("Failed fit indices:", failed_indices_F)

# Successful selected indices, ordered by q
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(fit_parameters_F[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

for i in idx:
    print(
        f"i={i:3d}, q={q_values[i]:.4f}: "
        f"tau1={Tau[i, 0]:.4g} ps, "
        f"tau2={Tau[i, 1]:.4g} ps, "
        f"C={Plateau[i]:.4f}, "
        f"RMSE={RMSE_F[i]:.3g}"
    )

In [ ]:
# ---- Convert tau (ps) to Gamma (meV, HWHM) ----
hbar = 0.6582119569  # meV·ps

Gamma = np.full_like(Tau, np.nan)
Gamma_err = np.full_like(Tau_err, np.nan)

valid = np.isfinite(Tau) & (Tau > 0)

Gamma[valid] = hbar / Tau[valid]
Gamma_err[valid] = hbar * Tau_err[valid] / Tau[valid]**2

# Lists aligned with original spectrum indices
Gamma1_list, Gamma2_list = Gamma.T.tolist()
Gamma1_err_list, Gamma2_err_list = Gamma_err.T.tolist()

# Successful selected indices, sorted by q
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(Gamma[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

# ---- Plot Gamma vs q ----
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["Fast / broad", "Intermediate", "Slow / narrow"]

fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))

for k in [0]:
    ax.plot(
        q_values[idx], Gamma[idx, k],
        "o-", markersize=4,
        color=colors[k], label=labels[k],
    )

    valid_err = np.isfinite(Gamma_err[idx, k])
    j = idx[valid_err]

    ax.errorbar(
        q_values[j], Gamma[j, k],
        yerr=Gamma_err[j, k],
        fmt="none", capsize=3,
        ecolor=colors[k],
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"$\Gamma$ (meV, HWHM)")
#ax.set_yscale("log")  # Remove for a linear scale
ax.legend()
#ax.set_ylim(0,0.5)
fig.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Plot individual fits and residuals
# ============================================================
for i in range(25,190,10):
    if not np.all(np.isfinite(fit_parameters_F[i])):
        continue

    F = total_F[i, 1:] / total_F[i, 1:2]
    mask = np.isfinite(F)

    t_plot = t[mask]
    x = dt[mask]
    y = F[mask]

    components = np.array([
        exponential(x, Amplitude_F[i, k], Tau[i, k])
        for k in range(2)
    ])

    total_fit = components.sum(axis=0) + Plateau[i]
    residual = y - total_fit

    fig, (ax, ax_err) = plt.subplots(
        2, 1,
        figsize=(7, 6),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1],
            "hspace": 0.08,
        },
    )

    ax.plot(
        t_plot, y,
        ".", color="0.5", markersize=3,
        label="Normalized F",
    )
    ax.plot(
        t_plot, total_fit,
        "k-", linewidth=2,
        label="Total fit",
    )

    for k in range(2):
        ax.plot(
            t_plot, components[k],
            "--", color=colors[k],
            label=(
                fr"Exp {k + 1}: $A$={Amplitude_F[i, k]:.3f}, "
                fr"$\tau$={Tau[i, k]:.4g} ps"
            ),
        )

    ax.axhline(
        Plateau[i],
        color="tab:green", linestyle=":",
        label=fr"Plateau: $C$={Plateau[i]:.3f}",
    )

    ax.set_ylabel(r"$F(q,t)/F(q,t_0)$")
    ax.set_title(
        fr"$q={q_values[i]:.3f}\ \mathrm{{\AA}}^{{-1}}$"
        f"  (i={i})"
    )
    ax.legend(fontsize=9)

    ax_err.axhline(0, color="black", linewidth=1)
    ax_err.plot(
        t_plot, residual,
        ".", color="tab:red", markersize=3,
    )
    ax_err.set_ylabel("Data − fit")
    ax_err.set_xlabel("Time (ps)")
    ax_err.text(
        0.98, 0.93,
        f"RMSE = {RMSE_F[i]:.3g}",
        transform=ax_err.transAxes,
        ha="right", va="top", fontsize=9,
    )

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

indices = []
tau1_list = []
tau2_list = []
tau1_err_list = []
tau2_err_list = []
A1_list = []
A2_list = []

for i in range(25, 190, 10):
    F =  H_F[i, 1:] / total_F[i, 1:2] 
    t = time[1:]

    # ---- Helper models ----
    def exp_decay(t, A, tau):
        return A * np.exp(-t / tau)

    def double_exp(t, A1, tau1, A2, tau2):
        return exp_decay(t, A1, tau1) + exp_decay(t, A2, tau2)

    # ---- Initial guesses ----
    t_min = t[t > 0].min() if np.any(t > 0) else t[1]
    t_max = t.max()

    tau1_0 = t_min * 5
    tau2_0 = t_max / 5

    A0 = F[0] / 2.0

    p0 = [A0, tau1_0, A0, tau2_0]

    bounds_lower = [0, t_min * 0.1,  0, t_min * 0.1]
    bounds_upper = [2,  t_max * 10,  2,  t_max * 10]

    try:
        popt, pcov = curve_fit(
            double_exp,
            t, F,
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A1_fit, tau1_fit, A2_fit, tau2_fit = popt
        perr = np.sqrt(np.diag(pcov))
        A1_err, tau1_err, A2_err, tau2_err = perr

    except Exception as e:
        print(f"Fit failed at index {i}: {e}")
        continue

    # ---- Fit curves ----
    fit_total = double_exp(t, *popt)
    exp1_fit  = exp_decay(t, A1_fit, tau1_fit)
    exp2_fit  = exp_decay(t, A2_fit, tau2_fit)

    # ---- Plot ----
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2),
                             gridspec_kw={'height_ratios': [2, 1]})
    ax = axes[0]
    ax.plot(t, F, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(t, fit_total, '-',  lw=2.5, color='red',    label='Double Exp (total)')
    ax.plot(t, exp1_fit,  '--', lw=1.5, color='green',  label=f'Exp 1 (τ={tau1_fit:.3g})')
    ax.plot(t, exp2_fit,  '--', lw=1.5, color='purple', label=f'Exp 2 (τ={tau2_fit:.3g})')
    ax.set_xlabel('Time t (ps)')
    ax.set_ylabel('F(q, t)')
    ax.set_xscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Double Exponential Fit — q = {round(q[i]/10, 3)}')

    ax2 = axes[1]
    ax2.plot(t, F - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Time t (ps)')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ---- Convert τ → FWHM in meV ----
    hbar = 0.6582119  # meV·ps
    FWHM1 = 2 * hbar / tau1_fit
    FWHM2 = 2 * hbar / tau2_fit

    # ---- Collect results ----
    indices.append(i)
    tau1_list.append(tau1_fit);  tau1_err_list.append(tau1_err)
    tau2_list.append(tau2_fit);  tau2_err_list.append(tau2_err)
    A1_list.append(A1_fit)
    A2_list.append(A2_fit)

    print(f"\n{'='*50}  Q index {i}")
    print(f"Exp 1:  A={A1_fit:.4g}±{A1_err:.2g}  τ={tau1_fit:.4g}±{tau1_err:.2g} ps  FWHM={FWHM1:.4g} meV")
    print(f"Exp 2:  A={A2_fit:.4g}±{A2_err:.2g}  τ={tau2_fit:.4g}±{tau2_err:.2g} ps  FWHM={FWHM2:.4g} meV")

# ---- FWHM vs q plot ----
hbar = 0.6582119
FWHM1_list     = [2 * hbar / t for t in tau1_list]
FWHM2_list     = [2 * hbar / t for t in tau2_list]
FWHM1_err_list_meV = [2 * hbar * e / t**2 for e, t in zip(tau1_err_list, tau1_list)]
FWHM2_err_list_meV = [2 * hbar * e / t**2 for e, t in zip(tau2_err_list, tau2_list)]

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='o-', color='green',
            capsize=4, label='FWHM Exp 1')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list_meV, fmt='s-', color='purple',
            capsize=4, label='FWHM Exp 2')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.set_ylim(0, 0.15)
ax.legend(fontsize=16, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
3.14*2/1.25


In [ ]:
H_F.shape

### 3 Exponential fit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ============================================================
# Settings and data
# ============================================================
total_F = np.asarray(H_F, dtype=float)
time = np.asarray(time, dtype=float)

q_values = np.asarray(q/10, dtype=float)  # Use q / 10 if q is in nm^-1
time_unit = "ps"                      # Must match the units of time

fit_indices = range(25,191)       # 20, 30, ..., 180
tau_initial = np.array([12.0, 70.0, 3000.0])

assert total_F.ndim == 2
assert total_F.shape[1] == len(time)
assert total_F.shape[0] == len(q_values)

t = time[1:]
t0 = t[0]
dt = t - t0

assert len(t) > 6
assert np.all(np.isfinite(t))
assert np.all(np.diff(t) > 0)

time_step = np.median(np.diff(t))
tau_min = time_step * 1e-6

assert np.all(tau_initial > tau_min)

# ============================================================
# Model: three exponential decays
# ============================================================
def exponential(dt, A, tau):
    return A * np.exp(-dt / tau)


def triple_exp(dt, A1, tau1, A2, tau2, A3, tau3):
    return (
        exponential(dt, A1, tau1)
        + exponential(dt, A2, tau2)
        + exponential(dt, A3, tau3)
    )


# ============================================================
# Storage
# ============================================================
nq = total_F.shape[0]

Tau = np.full((nq, 3), np.nan)
Tau_err = np.full((nq, 3), np.nan)
Amplitude_F = np.full((nq, 3), np.nan)

fit_parameters_F = np.full((nq, 6), np.nan)
RMSE_F = np.full(nq, np.nan)

failed_indices_F = []

# ============================================================
# Fit each selected q
# ============================================================
for i in fit_indices:
    denominator = total_F[i, 1]

    if not np.isfinite(denominator) or denominator <= 0:
        failed_indices_F.append(i)
        print(f"Skipped i={i}: invalid normalization value.")
        continue

    # Requested preprocessing
    F = total_F[i, 1:] / total_F[i, 1:2]

    mask = np.isfinite(F)
    x = dt[mask]
    y = F[mask]

    if len(y) <= 6:
        failed_indices_F.append(i)
        print(f"Skipped i={i}: too few valid points.")
        continue

    # Parameter order: [A1, tau1, A2, tau2, A3, tau3]
    p0 = np.empty(6)
    p0[0::2] = 1 / 3
    p0[1::2] = tau_initial

    try:
        popt, pcov = curve_fit(
            triple_exp,
            x,
            y,
            p0=p0,
            bounds=(
                [0, tau_min] * 3,
                [np.inf] * 6,
            ),
            method="trf",
            x_scale="jac",
            max_nfev=20000,
        )

    except (RuntimeError, ValueError) as exc:
        failed_indices_F.append(i)
        print(f"Fit failed at i={i}: {exc}")
        continue

    perr = np.sqrt(np.maximum(np.diag(pcov), 0))

    # Sort components by increasing tau: fast → slow
    order = np.argsort(popt[1::2])

    Tau[i] = popt[1::2][order]
    Tau_err[i] = perr[1::2][order]
    Amplitude_F[i] = popt[0::2][order]

    fit_parameters_F[i, 0::2] = Amplitude_F[i]
    fit_parameters_F[i, 1::2] = Tau[i]

    residual = y - triple_exp(x, *fit_parameters_F[i])
    RMSE_F[i] = np.sqrt(np.mean(residual**2))

# ============================================================
# Parameter lists, aligned with original spectrum indices
# Unfitted or failed indices contain NaN.
# ============================================================
Tau1_list, Tau2_list, Tau3_list = Tau.T.tolist()

Tau1_err_list, Tau2_err_list, Tau3_err_list = Tau_err.T.tolist()

A1_list, A2_list, A3_list = Amplitude_F.T.tolist()

print("Failed fit indices:", failed_indices_F)

# Successful selected indices, sorted by q
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(fit_parameters_F[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

# Print fitted values
print(f"\nDecay times in {time_unit}")
for i in idx:
    print(
        f"i={i:3d}, q={q_values[i]:.4f}: "
        f"tau1={Tau[i, 0]:.4g}, "
        f"tau2={Tau[i, 1]:.4g}, "
        f"tau3={Tau[i, 2]:.4g}, "
        f"RMSE={RMSE_F[i]:.3g}"
    )

In [ ]:
Tau

In [ ]:
# ---- Convert tau (ps) to Gamma (meV, HWHM) ----
hbar = 0.6582119569  # meV·ps

Gamma = np.full_like(Tau, np.nan)
Gamma_err = np.full_like(Tau_err, np.nan)

valid = np.isfinite(Tau) & (Tau > 0)

Gamma[valid] = hbar / Tau[valid] 
Gamma_err[valid] = hbar * Tau_err[valid] / Tau[valid]**2

# Lists aligned with original spectrum indices
Gamma1_list, Gamma2_list, Gamma3_list = Gamma.T.tolist()
Gamma1_err_list, Gamma2_err_list, Gamma3_err_list = Gamma_err.T.tolist()

# Successful selected indices, sorted by q
idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(Gamma[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

# ---- Plot Gamma vs q ----
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["Fast / broad", "Intermediate", "Slow / narrow"]

fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))

for k in [0]:
    ax.plot(
        q_values[idx], Gamma[idx, k],
        "o-", markersize=4,
        color=colors[k], label=labels[k],
    )

    valid_err = np.isfinite(Gamma_err[idx, k])
    j = idx[valid_err]

    ax.errorbar(
        q_values[j], Gamma[j, k],
        yerr=Gamma_err[j, k],
        fmt="none", capsize=3,
        ecolor=colors[k],
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(r"$\Gamma$ (meV)")
#ax.set_yscale("log")  # Remove for a linear scale
ax.legend()
#ax.set_ylim(0.04,0.07)
fig.tight_layout()
plt.show()

In [ ]:
t_plot

In [ ]:
np.savetxt("./hydration/QENS/400K_vac/time_masked.txt", t_plot, fmt="%.6f", delimiter="\t")

In [ ]:
#np.savetxt("./hydration/QENS/400K_vac/Iqt_masked.txt", F[mask], fmt="%.6f", delimiter="\t")
array = np.loadtxt("./hydration/QENS/400K_vac/Iqt_masked.txt", delimiter="\t")

In [ ]:
array.shape

In [ ]:
# ============================================================
# Plot each normalized spectrum, fitted components, residuals
# ============================================================
for i in range(25,190,10):
    if not np.all(np.isfinite(fit_parameters_F[i])):
        continue

    F = total_F[i, 1:] / total_F[i, 1:2]
    mask = np.isfinite(F)

    t_plot = t[mask]
    x = dt[mask]
    y = F[mask]

    components = np.array([
        exponential(x, Amplitude_F[i, k], Tau[i, k])
        for k in range(3)
    ])

    total_fit = components.sum(axis=0)
    residual = y - total_fit

    fig, (ax, ax_err) = plt.subplots(
        2,
        1,
        figsize=(7, 6),
        sharex=True,
        gridspec_kw={
            "height_ratios": [3, 1],
            "hspace": 0.08,
        },
    )

    # Normalized data and total fit
    ax.plot(
        t_plot, y,
        ".", color="r", markersize=5,
        label="Normalized F",
    )
    ax.plot(
        t_plot, total_fit,
        "k-", linewidth=2,
        label="Total fit",
    )

    # Individual exponential components
    for k in range(3):
        ax.plot(
            t_plot,
            components[k],
            "--",
            color=colors[k],
            label=(
                fr"{labels[k]}: $A$={Amplitude_F[i, k]:.3f}, "
                fr"$\tau$={Tau[i, k]:.4g} {time_unit}"
            ),
        )

    ax.set_ylabel(r"$F(q,t)/F(q,t_0)$")
    ax.set_title(
        fr"$q={q_values[i]:.3f}\ \mathrm{{\AA}}^{{-1}}$"
        f"  (i={i})"
    )
    ax.legend(fontsize=9)
    ax.set_yscale('log')
    ax.set_ylim(y.min()*0.9, 1)

    # Residuals
    ax_err.axhline(0, color="black", linewidth=1)
    ax_err.plot(
        t_plot, residual,
        ".", color="tab:red", markersize=3,
    )
    ax_err.set_ylabel("Data − fit")
    ax_err.set_xlabel(f"Time ({time_unit})")
    ax_err.text(
        0.98, 0.93,
        f"RMSE = {RMSE_F[i]:.3g}",
        transform=ax_err.transAxes,
        ha="right",
        va="top",
        fontsize=9,
    )
    plt.savefig(f'./hydration/Iqt_fit_q_{q_values[i]:.3f}.png')

In [ ]:
colors = ["tab:blue", "tab:orange", "tab:green"]
labels = ["Fast", "Intermediate", "Slow"]

time_unit = "ps"  # Change if time uses another unit.

idx = np.array(list(fit_indices), dtype=int)
idx = idx[np.all(np.isfinite(Tau[idx]), axis=1)]
idx = idx[np.argsort(q_values[idx])]

fig, ax = plt.subplots(figsize=(7, 5))

for k in range(3):
    ax.errorbar(
        q_values[idx], Tau[idx, k],
        yerr=Tau_err[idx, k],
        fmt="o-", markersize=3, capsize=2,
        color=colors[k], label=labels[k],
    )

ax.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax.set_ylabel(fr"$\tau$ ({time_unit})")
ax.set_yscale("log")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
for i in fit_indices:
    if not np.all(np.isfinite(fit_parameters_F[i])):
        continue

    F = total_F[i, 1:] / total_F[i, 1:2]
    mask = np.isfinite(F)

    t_plot = t[mask]
    x = dt[mask]
    y = F[mask]

    components = np.array([
        exponential(x, Amplitude_F[i, k], Tau[i, k])
        for k in range(3)
    ])

    total_fit = components.sum(axis=0)
    residual = y - total_fit

    fig, (ax, ax_err) = plt.subplots(
        2, 1, figsize=(7, 6),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
    )

    ax.plot(t_plot, y, ".", color="0.5", markersize=3, label="Normalized F")
    ax.plot(t_plot, total_fit, "k-", linewidth=2, label="Total fit")

    for k in range(3):
        ax.plot(
            t_plot, components[k], "--", color=colors[k],
            label=fr"{labels[k]}: $\tau$ = {Tau[i, k]:.4g} {time_unit}",
        )

    ax.set_ylabel(r"$F(q,t)/F(q,t_0)$")
    ax.set_title(
        fr"$q = {q_values[i]:.3f}\ \mathrm{{\AA}}^{{-1}}$  (i = {i})"
    )
    ax.legend(fontsize=9)

    ax_err.axhline(0, color="black", linewidth=1)
    ax_err.plot(t_plot, residual, ".", color="tab:red", markersize=3)
    ax_err.set_ylabel("Data − fit")
    ax_err.set_xlabel(f"Time ({time_unit})")
    ax_err.text(
        0.98, 0.93, f"RMSE = {RMSE_F[i]:.3g}",
        transform=ax_err.transAxes,
        ha="right", va="top", fontsize=9,
    )

    plt.show()
    plt.close(fig)

In [ ]:
%matplotlib widget
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

indices = []
tau1_list = []
tau2_list = []
tau3_list = []
tau1_err_list = []
tau2_err_list = []
tau3_err_list = []
A1_list = []
A2_list = []
A3_list = []
hbar = 0.6582119
for i in range(50, 190,10):
    F =  total_F[i, 1:] / total_F[i, 1:2] 
    t = time[1:]

    # ---- Helper models ----
    def exp_decay(t, A, tau):
        return A * np.exp(-t / tau)

    def triple_exp(t, A1, tau1, A2, tau2, A3, tau3):
        return exp_decay(t, A1, tau1) + exp_decay(t, A2, tau2) + exp_decay(t, A3, tau3)

    # ---- Initial guesses ----
    # Three timescales spanning the available time range
    t_min = t[t > 0].min() if np.any(t > 0) else t[1]
    t_max = t.max()

    tau1_0 = t_min * 5          # fast
    tau2_0 = np.sqrt(t_min * t_max)  # intermediate
    tau3_0 = t_max / 5          # slowa

    A0 = F[0] / 3.0  # split initial amplitude equally

    p0 = [A0, 10, A0, 500, A0, 1000]

    bounds_lower = [0, t_min * 0,   0, t_min * 0,   0, t_min * 0]
    bounds_upper = [2,  20,   2,  800,   2,  t_max * 10]

    try:
        popt, pcov = curve_fit(
            triple_exp,
            t, F,
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=50000
        )
        A1_fit, tau1_fit, A2_fit, tau2_fit, A3_fit, tau3_fit = popt
        perr = np.sqrt(np.diag(pcov))
        A1_err, tau1_err, A2_err, tau2_err, A3_err, tau3_err = perr

    except Exception as e:
        print(f"Fit failed at index {i}: {e}")
        continue

    # ---- Fit curves ----
    fit_total  = triple_exp(t, *popt)
    exp1_fit   = exp_decay(t, A1_fit, tau1_fit)
    exp2_fit   = exp_decay(t, A2_fit, tau2_fit)
    exp3_fit   = exp_decay(t, A3_fit, tau3_fit)

    # ---- Plot ----
    
    fig, axes = plt.subplots(2, 1, figsize=(singlecol*2, singlecol*2),
                             gridspec_kw={'height_ratios': [2, 1]})
    ax = axes[0]
    ax.plot(t, F, 'k.', ms=4, alpha=0.6, label='Data')
    ax.plot(t, fit_total, '-',  lw=2.5, color='red',    label='Triple Exp (total)')
    ax.plot(t, exp1_fit,  '--', lw=1.5, color='green',  label=f'Exp 1 (τ={tau1_fit:.3g}) Gamma={hbar/tau1_fit:.3g}')
    ax.plot(t, exp2_fit,  '--', lw=1.5, color='purple', label=f'Exp 2 (τ={tau2_fit:.3g}) Gamma={hbar/tau2_fit:.3g}')
    ax.plot(t, exp3_fit,  '--', lw=1.5, color='orange', label=f'Exp 3 (τ={tau3_fit:.3g}) Gamma={hbar/tau3_fit:.3g}')
    ax.set_xlabel('Time t (ps)')
    ax.set_ylabel('F(q, t)')
    ax.set_xscale('log')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Triple Exponential Fit — q = {round(q[i]/10, 3)}')

    ax2 = axes[1]
    ax2.plot(t, F - fit_total, '.', ms=4, color='blue', alpha=0.6)
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_xlabel('Time t (ps)')
    ax2.set_ylabel('Residual')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # ---- Convert τ → FWHM in meV via ℏ/τ ----
    # FWHM of a Lorentzian in energy = 2ℏ/τ  (ℏ = 0.6582 meV·ps)
    hbar = 0.6582119  # meV·ps
    FWHM1 = 2 * hbar / tau1_fit
    FWHM2 = 2 * hbar / tau2_fit
    FWHM3 = 2 * hbar / tau3_fit
    FWHM1_err = FWHM1 * (tau1_err / tau1_fit)
    FWHM2_err = FWHM2 * (tau2_err / tau2_fit)
    FWHM3_err = FWHM3 * (tau3_err / tau3_fit)

    # ---- Collect results ----
    indices.append(i)
    tau1_list.append(tau1_fit);  tau1_err_list.append(tau1_err)
    tau2_list.append(tau2_fit);  tau2_err_list.append(tau2_err)
    tau3_list.append(tau3_fit);  tau3_err_list.append(tau3_err)
    A1_list.append(A1_fit)
    A2_list.append(A2_fit)
    A3_list.append(A3_fit)

    FWHM1_list = [2 * 0.6582119 / t for t in tau1_list]
    FWHM2_list = [2 * 0.6582119 / t for t in tau2_list]
    FWHM3_list = [2 * 0.6582119 / t for t in tau3_list]
    FWHM1_err_list_meV = [2 * 0.6582119 * e / t**2 for e, t in zip(tau1_err_list, tau1_list)]
    FWHM2_err_list_meV = [2 * 0.6582119 * e / t**2 for e, t in zip(tau2_err_list, tau2_list)]
    FWHM3_err_list_meV = [2 * 0.6582119 * e / t**2 for e, t in zip(tau3_err_list, tau3_list)]

    print(f"\n{'='*50}  Q index {i}")
    print(f"Exp 1:  A={A1_fit:.4g}±{A1_err:.2g}  τ={tau1_fit:.4g}±{tau1_err:.2g} ps  FWHM={FWHM1:.4g} meV")
    print(f"Exp 2:  A={A2_fit:.4g}±{A2_err:.2g}  τ={tau2_fit:.4g}±{tau2_err:.2g} ps  FWHM={FWHM2:.4g} meV")
    print(f"Exp 3:  A={A3_fit:.4g}±{A3_err:.2g}  τ={tau3_fit:.4g}±{tau3_err:.2g} ps  FWHM={FWHM3:.4g} meV")

In [ ]:
plt.figure()
plt.scatter(q[indices]/10,tau1_list)
plt.scatter(q[indices]/10,tau2_list)
#plt.plot(tau3_list)

In [ ]:
# ---- FWHM vs q plot ----
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='o-', color='green', capsize=4, label='FWHM Exp 2')
#ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list_meV, fmt='s-', color='purple', capsize=4, label='FWHM Exp 2')
ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list_meV, fmt='s-', color='orange',capsize=4, label='FWHM Exp 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
#ax.set_ylim(0, 0.01)
ax.legend(fontsize=16, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ---- FWHM vs q plot ----
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='o-', color='green', capsize=4, label='FWHM Exp 2')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list_meV, fmt='s-', color='purple', capsize=4, label='FWHM Exp 2')
#ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list_meV, fmt='s-', color='orange',capsize=4, label='FWHM Exp 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.set_ylim(0, 0.01)
ax.legend(fontsize=16, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ---- FWHM vs q plot ----
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(q[indices]/10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='o-', color='green',
#            capsize=4, label='FWHM Exp 1')
ax.errorbar(q[indices]/10, FWHM2_list, yerr=FWHM2_err_list_meV, fmt='s-', color='purple',
            capsize=4, label='FWHM Exp 2')
#ax.errorbar(q[indices]/10, FWHM3_list, yerr=FWHM3_err_list_meV, fmt='s-', color='orange',
#            capsize=4, label='FWHM Exp 3')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.set_ylim(0, 0.02)
ax.legend(fontsize=16, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

### 4 Exponential fit

In [ ]:
%matplotlib widget

from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt

indices = []

tau1_list, tau2_list, tau3_list, tau4_list = [], [], [], []
tau1_err_list, tau2_err_list, tau3_err_list, tau4_err_list = [], [], [], []
A1_list, A2_list, A3_list, A4_list = [], [], [], []

FWHM1_list, FWHM2_list, FWHM3_list, FWHM4_list = [], [], [], []
FWHM1_err_list_meV, FWHM2_err_list_meV = [], []
FWHM3_err_list_meV, FWHM4_err_list_meV = [], []

tau_lists = [tau1_list, tau2_list, tau3_list, tau4_list]
tau_err_lists = [
    tau1_err_list, tau2_err_list,
    tau3_err_list, tau4_err_list
]
A_lists = [A1_list, A2_list, A3_list, A4_list]
FWHM_lists = [FWHM1_list, FWHM2_list, FWHM3_list, FWHM4_list]
FWHM_err_lists = [
    FWHM1_err_list_meV, FWHM2_err_list_meV,
    FWHM3_err_list_meV, FWHM4_err_list_meV
]

hbar = 0.6582119  # meV·ps
plot_fits = True


# ---- Helper models ----
def exp_decay(t, A, tau):
    return A * np.exp(-t / tau)


def quadruple_exp(t, A1, tau1, A2, tau2, A3, tau3, A4, tau4):
    return (
        exp_decay(t, A1, tau1)
        + exp_decay(t, A2, tau2)
        + exp_decay(t, A3, tau3)
        + exp_decay(t, A4, tau4)
    )


for i in range(45, 190, 10):
    normalization = H_F[i, 1]

    if not np.isfinite(normalization) or normalization == 0:
        print(f"Skipped index {i}: invalid normalization.")
        continue

    F = np.asarray(total_F[i, 1:] / normalization, dtype=float)
    t = np.asarray(time[1:], dtype=float)

    # ---- Keep finite data at nonnegative times ----
    fit_mask = np.isfinite(t) & np.isfinite(F) & (t >= 0)
    t = t[fit_mask]
    F = F[fit_mask]

    if len(t) <= 8 or not np.any(t > 0):
        print(f"Skipped index {i}: insufficient valid data.")
        continue

    t_max = t.max()

    # ---- Initial guesses ----
    A0 = F[0] / 4.0

    # Four starting timescales in ps; adjust to your data.
    # Four starting timescales in ps; adjust to your data.
    tau_guesses = np.array([hbar/2, hbar/0.25, hbar/0.025, hbar/0.0005])

    # ---- Bounds ----
    # Parameter order: A1, tau1, A2, tau2, A3, tau3, A4, tau4
    bounds_lower = [
        0, 1e-10,
        0, 1e-10,
        0, 1e-10,
        0, 1e-10
    ]

    bounds_upper = [
        2, 20,
        2, 200,
        2, 800,
        2, np.inf
    ]

    # Keep starting guesses strictly inside the bounds.
    tau_upper = np.asarray(bounds_upper[1::2])
    tau_guesses = np.minimum(tau_guesses, 0.9 * tau_upper)
    A0 = np.clip(A0, 1e-8, 2 - 1e-8)

    p0 = [
        A0, tau_guesses[0],
        A0, tau_guesses[1],
        A0, tau_guesses[2],
        A0, tau_guesses[3]
    ]

    try:
        popt, pcov = curve_fit(
            quadruple_exp,
            t, F,
            p0=p0,
            bounds=(bounds_lower, bounds_upper),
            maxfev=100000
        )

        # Each row contains [A, tau].
        params = popt.reshape(4, 2)
        errors = np.sqrt(np.diag(pcov)).reshape(4, 2)

    except (RuntimeError, ValueError) as e:
        print(f"Fit failed at index {i}: {e}")
        continue

    # ---- Fit curves ----
    fit_total = quadruple_exp(t, *popt)
    component_curves = [
        exp_decay(t, *component_params)
        for component_params in params
    ]

    # ---- Optional plot ----
    if plot_fits:
        fig, axes = plt.subplots(
            2, 1,
            figsize=(8, 7),
            gridspec_kw={"height_ratios": [2, 1]},
            sharex=True
        )

        ax, ax2 = axes
        positive_t = t > 0

        ax.plot(
            t[positive_t], F[positive_t],
            "k.", ms=4, alpha=0.6, label="Data"
        )
        ax.plot(
            t[positive_t], fit_total[positive_t],
            "-", lw=2.5, color="red",
            label="Four exponentials (total)"
        )

        colors = ["orange", "red", "green", "purple"]

        for j, (curve, color) in enumerate(zip(component_curves, colors)):
            ax.plot(
                t[positive_t], curve[positive_t],
                "--", lw=1.5, color=color,
                label=f"Exp {j + 1} (τ={params[j, 1]:.3g} ps Gamma = {round(hbar/params[j, 1],5)})"
            )

        ax.set_ylabel("F(q, t)")
        ax.set_xscale("log")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.set_title(
            f"Four-Exponential Fit — q = {q[i] / 10:.3f}"
        )

        ax2.plot(
            t[positive_t], (F - fit_total)[positive_t],
            ".", ms=4, color="blue", alpha=0.6
        )
        ax2.axhline(0, color="k", lw=0.8, ls="--")
        ax2.set_xlabel("Time t (ps)")
        ax2.set_ylabel("Residual")
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # ---- Convert tau to Lorentzian FWHM ----
    # Assumes time is in ps:
    # FWHM = 2*hbar/tau, in meV
    taus = params[:, 1]
    tau_errors = errors[:, 1]

    FWHMs = 2 * hbar / taus
    FWHM_errors = 2 * hbar * tau_errors / taus**2

    # ---- Collect results ----
    indices.append(i)

    for j in range(4):
        A_lists[j].append(params[j, 0])
        tau_lists[j].append(taus[j])
        tau_err_lists[j].append(tau_errors[j])
        FWHM_lists[j].append(FWHMs[j])
        FWHM_err_lists[j].append(FWHM_errors[j])

    # ---- Print results ----
    print(f"\n{'=' * 50}  Q index {i}")

    for j in range(4):
        print(
            f"Exp {j + 1}:  "
            f"A={params[j, 0]:.4g}±{errors[j, 0]:.2g}  "
            f"τ={taus[j]:.4g}±{tau_errors[j]:.2g} ps  "
            f"FWHM={FWHMs[j]:.4g}±{FWHM_errors[j]:.2g} meV"
        )

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))
#ax.errorbar(np.asarray(q)[indices] / 10, FWHM4_list, yerr=FWHM4_err_list_meV, fmt='o-', color='purple', capsize=4, label='FWHM Lorentzian 3')
#ax.errorbar(np.asarray(q)[indices] / 10, FWHM3_list, yerr=FWHM3_err_list_meV, fmt='s-', color='green', capsize=4, label='FWHM Lorentzian 2')
#ax.errorbar(np.asarray(q)[indices] / 10, FWHM2_list, yerr=FWHM2_err_list_meV, fmt='s-', color='green',capsize=4, label='FWHM Lorentzian 1')
ax.errorbar(np.asarray(q)[indices] / 10, FWHM1_list, yerr=FWHM1_err_list_meV, fmt='s-', color='orange',capsize=4, label='FWHM Lorentzian 0')
ax.set_xlabel('q (1/Å)')
ax.set_ylabel('FWHM (meV)')
ax.legend(fontsize=16, frameon=False)
#ax.grid(True, alpha=0.3)
#ax.set_title('FWHM vs Q index')
plt.tight_layout()
plt.show()
#plt.ylim(0,0.5)

In [ ]:
plt.rcParams.update({'font.size': 16})

fig, ax = plt.subplots(figsize=(singlecol*2, singlecol*2))

ax.errorbar(
    np.asarray(q)[indices] / 10,
    FWHM4_list,
    yerr=FWHM4_err_list_meV,
    fmt='s-',
    color='red',
    capsize=4,
    label='Component 2'
)

ax.set_xlabel(r'$Q$ ($\mathrm{\AA}^{-1}$)')
ax.set_ylabel('FWHM (meV)')
#ax.set_ylim(0, 2)
ax.legend(fontsize=16, frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
if __name__ == "__main__":
    # ── Load your data here ──────────────────────────────────────────────────
    # import numpy as np
    # total_F = np.load("your_file.npy")   # or however you load it
    # t       = np.load("your_time.npy")
    ls=range(25,190,5)
    data = H_F[ls, 1:] / H_F[ls, 1:2]   # shape (T, N_Q)
    time = t[1:]                                    # shape (T,)
    q_ls = q[ls] / 10                             # shape (N_Q,) in Å⁻¹

    # ── Synthetic demo (remove when using real data) ─────────────────────────
    '''
    np.random.seed(0)
    time = np.logspace(-2, 2, 200)          # 0.01 … 100 ps  shape (T,)
    q_ls = np.linspace(0.3, 1.8, 8)        # 8 Q points in Å⁻¹  shape (N_Q,)
    N_Q  = len(q_ls)
    D1, D2 = 0.01, 0.15                    # Å²/ps  (slow & fast)
    data = np.zeros((N_Q, len(time)))       # shape (N_Q, T)
    for j, qi in enumerate(q_ls):
        G1 = D1 * qi**2
        G2 = D2 * qi**2
        clean = 0.6 * np.exp(-G1 * time) + 0.4 * np.exp(-G2 * time)
        data[j, :] = clean + 0.005 * np.random.randn(len(time))
    data = np.clip(data, 1e-6, None)
    '''
    # ─────────────────────────────────────────────────────────────────────────

    print("Fitting F(q,t) …")
    fit_results = run_fitting(time, data, q_ls)

    print("\nPlotting γ vs Q …")
    plot_results(fit_results, q_ls)

    print("\nPlotting fit curves …")
    plot_fits(time, data, fit_results, q_ls)

In [ ]:
if __name__ == "__main__":
    # ── Load your data here ──────────────────────────────────────────────────
    # import numpy as np
    # total_F = np.load("your_file.npy")   # or however you load it
    # t       = np.load("your_time.npy")
    ls=range(25,190,5)
    data = Na_F[ls, 1:] / Na_F[ls, 1:2]   # shape (T, N_Q)
    time = t[1:]                                    # shape (T,)
    q_ls = q[ls] / 10                             # shape (N_Q,) in Å⁻¹

    # ── Synthetic demo (remove when using real data) ─────────────────────────
    '''
    np.random.seed(0)
    time = np.logspace(-2, 2, 200)          # 0.01 … 100 ps  shape (T,)
    q_ls = np.linspace(0.3, 1.8, 8)        # 8 Q points in Å⁻¹  shape (N_Q,)
    N_Q  = len(q_ls)
    D1, D2 = 0.01, 0.15                    # Å²/ps  (slow & fast)
    data = np.zeros((N_Q, len(time)))       # shape (N_Q, T)
    for j, qi in enumerate(q_ls):
        G1 = D1 * qi**2
        G2 = D2 * qi**2
        clean = 0.6 * np.exp(-G1 * time) + 0.4 * np.exp(-G2 * time)
        data[j, :] = clean + 0.005 * np.random.randn(len(time))
    data = np.clip(data, 1e-6, None)
    '''
    # ─────────────────────────────────────────────────────────────────────────

    print("Fitting F(q,t) …")
    fit_results = run_fitting(time, data, q_ls)

    print("\nPlotting γ vs Q …")
    plot_results(fit_results, q_ls)

    print("\nPlotting fit curves …")
    plot_fits(time, data, fit_results, q_ls)

In [ ]:
if __name__ == "__main__":
    # ── Load your data here ──────────────────────────────────────────────────
    # import numpy as np
    # total_F = np.load("your_file.npy")   # or however you load it
    # t       = np.load("your_time.npy")
    ls=range(25,190,5)
    data = Na_F[ls, 1:] / Na_F[ls, 1:2]   # shape (T, N_Q)
    time = t[1:]                                    # shape (T,)
    q_ls = q[ls] / 10                             # shape (N_Q,) in Å⁻¹

    # ── Synthetic demo (remove when using real data) ─────────────────────────
    '''
    np.random.seed(0)
    time = np.logspace(-2, 2, 200)          # 0.01 … 100 ps  shape (T,)
    q_ls = np.linspace(0.3, 1.8, 8)        # 8 Q points in Å⁻¹  shape (N_Q,)
    N_Q  = len(q_ls)
    D1, D2 = 0.01, 0.15                    # Å²/ps  (slow & fast)
    data = np.zeros((N_Q, len(time)))       # shape (N_Q, T)
    for j, qi in enumerate(q_ls):
        G1 = D1 * qi**2
        G2 = D2 * qi**2
        clean = 0.6 * np.exp(-G1 * time) + 0.4 * np.exp(-G2 * time)
        data[j, :] = clean + 0.005 * np.random.randn(len(time))
    data = np.clip(data, 1e-6, None)
    '''
    # ─────────────────────────────────────────────────────────────────────────

    print("Fitting F(q,t) …")
    fit_results = run_fitting(time, data, q_ls)

    print("\nPlotting γ vs Q …")
    plot_results(fit_results, q_ls)

    print("\nPlotting fit curves …")
    plot_fits(time, data, fit_results, q_ls)

In [ ]:
if __name__ == "__main__":
    # ── Load your data here ──────────────────────────────────────────────────
    # import numpy as np
    # total_F = np.load("your_file.npy")   # or however you load it
    # t       = np.load("your_time.npy")
    ls=range(25,190,5)
    data = H_F[ls, 1:] / H_F[ls, 1:2]   # shape (T, N_Q)
    time = t[1:]                                    # shape (T,)
    q_ls = q[ls] / 10                             # shape (N_Q,) in Å⁻¹

    # ── Synthetic demo (remove when using real data) ─────────────────────────
    '''
    np.random.seed(0)
    time = np.logspace(-2, 2, 200)          # 0.01 … 100 ps  shape (T,)
    q_ls = np.linspace(0.3, 1.8, 8)        # 8 Q points in Å⁻¹  shape (N_Q,)
    N_Q  = len(q_ls)
    D1, D2 = 0.01, 0.15                    # Å²/ps  (slow & fast)
    data = np.zeros((N_Q, len(time)))       # shape (N_Q, T)
    for j, qi in enumerate(q_ls):
        G1 = D1 * qi**2
        G2 = D2 * qi**2
        clean = 0.6 * np.exp(-G1 * time) + 0.4 * np.exp(-G2 * time)
        data[j, :] = clean + 0.005 * np.random.randn(len(time))
    data = np.clip(data, 1e-6, None)
    '''
    # ─────────────────────────────────────────────────────────────────────────

    print("Fitting F(q,t) …")
    fit_results = run_fitting(time, data, q_ls)

    print("\nPlotting γ vs Q …")
    plot_results(fit_results, q_ls)

    print("\nPlotting fit curves …")
    plot_fits(time, data, fit_results, q_ls)

In [ ]:
# ── ENTRY POINT ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Load your data here ──────────────────────────────────────────────────
    # import numpy as np
    # total_F = np.load("your_file.npy")   # or however you load it
    # t       = np.load("your_time.npy")
    data = total_F[25:] / total_F[25:, 1:2]   # shape (T, N_Q)
    time = t[1:]                                    # shape (T,)
    q_ls = q[25:] / 10                             # shape (N_Q,) in Å⁻¹

    # ── Synthetic demo (remove when using real data) ─────────────────────────
    '''
    np.random.seed(0)
    time = np.logspace(-2, 2, 200)          # 0.01 … 100 ps
    q_ls = np.linspace(0.3, 1.8, 8)        # 8 Q points in Å⁻¹
    N_Q  = len(q_ls)
    D1, D2 = 0.01, 0.15                    # Å²/ps  (slow & fast)
    data = np.zeros((len(time), N_Q))
    for j, qi in enumerate(q_ls):
        G1 = D1 * qi**2
        G2 = D2 * qi**2
        clean = 0.6 * np.exp(-G1 * time) + 0.4 * np.exp(-G2 * time)
        data[:, j] = clean + 0.005 * np.random.randn(len(time))
    data = np.clip(data, 1e-6, None)
    '''
    # ─────────────────────────────────────────────────────────────────────────

    print("Fitting F(q,t) …")
    fit_results = run_fitting(time, data, q_ls)

    print("\nPlotting γ vs Q …")
    plot_results(fit_results, q_ls)

    print("\nPlotting fit curves …")
    plot_fits(time, data, fit_results, q_ls)

In [ ]:
data.shape

In [ ]:
from scipy.optimize import curve_fit
import traceback

# inspect first Q column
fqt = data[:, 0]
print("time[:5]:", time[:5])
print("time[-5:]:", time[-5:])
print("fqt[:5]:", fqt[:5])
print("fqt[-5:]:", fqt[-5:])
print("any NaN:", np.any(np.isnan(fqt)), np.any(np.isnan(time)))
print("any Inf:", np.any(np.isinf(fqt)), np.any(np.isinf(time)))
print("fqt min/max:", fqt.min(), fqt.max())

# try fitting and print the actual exception
for n_comp, (func, param_names) in MODELS.items():
    p0 = initial_guess(n_comp, time, fqt)
    lo, hi = bounds(n_comp)
    print(f"\n--- {n_comp}-component ---")
    print("p0:", p0)
    try:
        popt, pcov = curve_fit(func, time, fqt, p0=p0, bounds=(lo, hi),
                               maxfev=20000, ftol=1e-10, xtol=1e-10)
        print("popt:", popt)
    except Exception as e:
        traceback.print_exc()

In [ ]:
data.shape[0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# --- Data Constants ---
HBAR = 0.65821  # meV * ps

# 1. Define the 3-Exponential Model
def three_exp(t, A1, g1, A2, g2, A3, g3):
    """
    g1, g2, g3 are in ps^-1
    """
    return (A1 * np.exp(-g1 * t) + 
            A2 * np.exp(-g2 * t) + 
            A3 * np.exp(-g3 * t))

# Assuming your data is F_Qt (191, 1250) and time is t (1250,)
# data = F_Qt
# time_axis = t

# Initialize storage for parameters
# We'll store 3 Gammas and 3 Amplitudes for each of the 191 Q-values
results_meV = np.zeros((data.shape[0], 3)) 

# 2. Fitting Loop
for i in range(data.shape[0]):
    y = data[i, :]
    
    # Initial Guess: [A1, g1, A2, g2, A3, g3]
    # It helps to sort guesses from fastest to slowest
    p0 = [0.4, 1.0, 0.3, 0.1, 0.3, 0.01] 
    
    try:
        # Enforce positive bounds for physical reality
        popt, _ = curve_fit(three_exp, time_axis, y, p0=p0, bounds=(0, np.inf))
        
        # Convert the three gammas (indices 1, 3, 5) to meV
        results_meV[i, 0] = popt[1] * HBAR
        results_meV[i, 1] = popt[3] * HBAR
        results_meV[i, 2] = popt[5] * HBAR
    except:
        results_meV[i, :] = np.nan

# 3. Plotting Gamma vs Q
# Replace q_values with your actual Q vector
q_values = np.linspace(0.2, 2.0, 191)

plt.figure(figsize=(8, 6))
plt.plot(q_values, results_meV[:, 0], 'o', label='$\Gamma_1$ (Fast)')
plt.plot(q_values, results_meV[:, 1], 's', label='$\Gamma_2$ (Medium)')
plt.plot(q_values, results_meV[:, 2], '^', label='$\Gamma_3$ (Slow)')

plt.xlabel('Q ($\AA^{-1}$)')
plt.ylabel('$\Gamma$ (meV)')
plt.title('Dispersion of Relaxation Rates (3-Exp Fit)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Total scattering

In [ ]:
with h5py.File("./hydration/QENS/400K_vac/400K_4_DISF_coherent.mda", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
i=10

file_path = f"./hydration/QENS/400K_vac/400K_{i}_DISF_coherent.mda"
with h5py.File(file_path, "r") as f:
    total_coh_S = f["dcsf/s(q,f)/total"][:]

file_path = f"./hydration/QENS/400K_vac/400K_{i}_DISF.mda"
with h5py.File(file_path, "r") as f:
    total_inc_S = f["disf/s(q,f)/total"][:]

omega = np.linspace(-0.51675,0.51675,1499)

In [ ]:
file_path = "./hydration/QENS/400K_9/400K_4_DISF_coherent.mda"

with h5py.File(file_path, "r") as f:
    total_inc_S = f["ndsf/s(q,f)_inc/total"][:]
    C_inc_S = f["ndsf/s(q,f)_inc/C"][:]
    Mn_inc_S = f["ndsf/s(q,f)_inc/Mn"][:]
    Na_inc_S = f["ndsf/s(q,f)_inc/Na"][:]
    N_inc_S = f["ndsf/s(q,f)_inc/N"][:]
    O_inc_S = f["ndsf/s(q,f)_inc/O"][:]
    H_inc_S = f["ndsf/s(q,f)_inc/H"][:]
    Fe_inc_S = f["ndsf/s(q,f)_inc/Fe"][:]

    total_coh_S = f["ndsf/s(q,f)_coh/total"][:]
    total_S = f["ndsf/s(q,f)/total"][:]
    
    total_inc_F = f["ndsf/f(q,t)_inc/total"][:]
    Na_inc_F = f["ndsf/f(q,t)_inc/Na"][:]
    O_inc_F = f["ndsf/f(q,t)_inc/O"][:]
    H_inc_F = f["ndsf/f(q,t)_inc/H"][:]
    Fe_inc_F = f["ndsf/f(q,t)_inc/Fe"][:]
    Mn_inc_F = f["ndsf/f(q,t)_inc/Mn"][:]
    C_inc_F = f["ndsf/f(q,t)_inc/C"][:]
    N_inc_F = f["ndsf/f(q,t)_inc/N"][:]

    total_coh_F = f["ndsf/f(q,t)_coh/total"][:]
    total_F = f["ndsf/f(q,t)/total"][:]


In [ ]:
%matplotlib widget
plt.rcParams.update({'font.size': 8})

import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    #(total_S, "Total"),
    (total_coh_S,  "Coherent"),
    (total_inc_S,  "Incoherent"),
    #(O_inc_S, "O")
]

indices = range(40, 190)   # frames to plot
n = len(indices)
cmap = plt.get_cmap("jet")
fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

def remove_peak_outlier(S_raw):
    """Remove the single highest peak by linear interpolation."""
    S_clean = S_raw.copy()
    i0 = np.argmax(S_raw)
    lo = max(i0 - 0, 0)
    hi = min(i0 + 1, len(S_raw))
    good_idx = np.r_[0:lo, hi:len(S_raw)]
    S_clean[lo:hi] = np.interp(np.arange(lo, hi), good_idx, S_raw[good_idx])
    return S_clean

for ax, (data, label) in zip(axes, datasets):
    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        S_clean = remove_peak_outlier(data[i, :])
        ax.plot(omega, S_clean, color=color, label=f"t={i}")
    ax.set_title(label)
    ax.set_xlabel("Energy transfer (meV)")
    ax.set_ylabel("S(q, w)")
    ax.set_xlim(-0.03, 0.03)
    ax.set_yscale("log")
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_F, "Total"),
    (H_inc_F,  "Coherent"),
    (total_inc_F,  "Incoherent"),
    (Na_inc_F, "O")
]

indices = range(25,190,5)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(t, data[i, :]/data[i, 1], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("F(q, t)")
    ax.set_xscale('log')
    ax.set_ylim(-0.1,1.1)
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
%matplotlib widget
i=70
plt.figure( figsize=(singlecol*2, singlecol*2))
S_clean = remove_peak_outlier(total_S[i,:])
plt.plot(omega, S_clean, label='total', lw=3)
S_clean = remove_peak_outlier(total_coh_S[i, :])
plt.plot(omega, S_clean, label='coherent')
#plt.plot(omega, C_disf[i,:], label='C')
#plt.plot(omega, N_disf[i,:], label='N')
#plt.plot(omega, Mn_disf[i,:], label='Mn')
#plt.plot(omega, Fe_disf[i,:], label='Fe')
#plt.plot(omega, O_disf[i,:], label='O')
S_clean = remove_peak_outlier(total_inc_S[i, :])
plt.plot(omega, S_clean, label='incoherent')
plt.xlim(-0.2,0.2)
plt.ylim(1e-10,1e-7)
plt.plot()
plt.legend()
plt.yscale('log')
plt.xlabel("Energy transfer (meV)")
plt.ylabel('S(q,w)')

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_S, "Total"),
    (total_coh_S,  "Na"),
    (total_inc_S,  "H"),
    (O_inc_S, "O")
]

indices = range(40,190)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(2, 2, figsize=(singlecol*2, singlecol*2))
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(data[i, :], color=color, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("Time (ps)")
    ax.set_ylabel("S(q, w)")
    #ax.set_xlim(0,2000)
    ax.grid(True)


plt.yscale('log')
plt.tight_layout()
plt.show()


## DOS

In [ ]:
with h5py.File("./hydration/QENS/400K_DOS/400K_1_DOS.mda", "r") as f:
    def print_keys(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"[GROUP] {name}")
            print("  keys:", list(obj.keys()))
        elif isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")

    f.visititems(print_keys)

In [ ]:
#file_path = "./hydration/QENS/300K_9/400K_9_DISF_1000.mda"
file_path = "./hydration/QENS/400K_DOS/400K_1_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
    C_DOS = f["dos/C"][:]#*weights['C']
    Mn_DOS = f["dos/Mn"][:]#*weights['Mn']
    Na_DOS = f["dos/Na"][:]#*weights['Na']
    N_DOS = f["dos/N"][:]#*weights['N']
    O_DOS = f["dos/O"][:]#*weights['O']
    H_DOS = f["dos/H"][:]#*weights['H']
    Fe_DOS = f["dos/Fe"][:]#*weights['Fe']

    E = f["dos/axes/time"][:]


### Vacancy-free

In [ ]:
file_path = "./hydration/QENS/300K_DOS/300K_2_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_H = f["dos/total"][:]

file_path = "./hydration/QENS/300K_DOS/300K_1_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_D = f["dos/total"][:]

E = np.linspace(0, 413.4, 1000)
    
plt.figure(figsize=(singlecol*2,singlecol*2))

plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})

plt.plot(E, total_DOS_D, label='x = 1.8, y= 0')
plt.plot(E, total_DOS_H, label='x = 1.75, y= 2')

plt.xlabel("Energy (meV)")
plt.ylabel("Intensity (a.u.)")

plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()
plt.show()
plt.savefig('vDOS.svg')

In [ ]:
%matplotlib widget
plt.figure(figsize=(singlecol*2, singlecol*2))
E = np.linspace(0, 413.4, 1000)
file_path = "./hydration/QENS/400K_DOS/400K_10_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_vac')

file_path = "./hydration/QENS/400K_9/400K_9_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_9')

file_path = "./hydration/QENS/400K_8/400K_8_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_8')

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()

In [ ]:
%matplotlib widget
plt.figure(figsize=(singlecol*2, singlecol*2))
E = np.linspace(0, 413.4, 1000)
file_path = "./hydration/QENS/400K_DOS/400K_1_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_1')

file_path = "./hydration/QENS/400K_DOS/400K_2_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_2')

file_path = "./hydration/QENS/400K_DOS/400K_4_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_4')

file_path = "./hydration/QENS/400K_DOS/400K_5_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_5')

file_path = "./hydration/QENS/400K_DOS/400K_6_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_6')

file_path = "./hydration/QENS/400K_DOS/400K_10_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='400K_10')

file_path = "./hydration/QENS/400K_9/400K_9_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='No_vac')

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()

In [ ]:
%matplotlib widget
plt.figure(figsize=(singlecol*2, singlecol*2))
plt.rcParams.update({'legend.fontsize': 16})
plt.rcParams.update({'font.size': 16})
E = np.linspace(0, 413.4, 1000)

num_Na_list = [224,224,224,192,192,192,1,1,256,240]
num_H2O_list = [416,512,584,312,512,626,1,1,512,520]
color_list = []

k=0
i=10
file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_select_{i}.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
#plt.plot(E,  (total_DOS/total_DOS.max())+0.5*(k),
#         label=f'400K vac = {1-round(num_Na_list[i-1]/256,4)} y = {round(num_H2O_list[i-1]/256,3)} (Coordinated)')

for i in [4,5]:
    file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_{i}.mda"
    with h5py.File(file_path, "r") as f:
        total_DOS = f["dos/total"][:]
    plt.plot(E, (total_DOS/total_DOS.max())+0.35*k, label=f'400 K y = {round(num_H2O_list[i-1]/256,3):.3f} □ = {1-round(num_Na_list[i-1]/256,4)}')
    k += 1

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()

plt.savefig('vDOS_45.svg')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
E = np.linspace(0, 413.4, 1000)

i=10
file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_select_{i}.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_1 = f["dos/H"][:]
plt.plot(E,  total_DOS_1*0.0523*20, label=f'Vacancy × 20')

i=10
file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_invert_{i}.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_1 = f["dos/H"][:]
plt.plot(E,  total_DOS_1*0.9476, label=f'Interstitial')

i=10
file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_{i}.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_2 = f["dos/H"][:]
plt.plot(E, total_DOS_2, label=f'Total')

i=9
file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_{i}.mda"
with h5py.File(file_path, "r") as f:
    total_DOS_2 = f["dos/H"][:]
#plt.plot(E, total_DOS_2, label=f'400K vac = {1-round(num_Na_list[i-1]/256,4)} y = {round(num_H2O_list[i-1]/256,3)}')

file_path = "./hydration/QENS/400K_9/400K_9_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
#plt.plot(E, total_DOS, label='No_vac')

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()

plt.savefig('vDOS_10.svg')

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
E = np.linspace(0, 413.4, 1000)

num_Na_list = [224,224,224,192,192,192,1,1,1,240]
num_H2O_list = [416,512,584,312,512,626,1,1,1,520]

for i in [4,5,10]:
    file_path = f"./hydration/QENS/400K_vac/400K_vac_DOS_{i}.mda"
    with h5py.File(file_path, "r") as f:
        total_DOS = f["dos/total"][:]
    plt.plot(E, total_DOS, label=f'400K vac = {round(num_Na_list[i-1]/256,3)} y = {round(num_H2O_list[i-1]/256,3)}')

file_path = "./hydration/QENS/400K_9/400K_9_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
#plt.plot(E, total_DOS, label='No_vac')

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()

In [ ]:
### 400K_11 (x = 1.8, y = 0)
weight_ls = [0.0009016126660181753,0.060504082522098636,0.060107511067892146,0.4508063330090505,0.42768046073494054]

In [ ]:
#file_path = "./hydration/QENS/300K_9/400K_9_DISF_1000.mda"
file_path = "./hydration/QENS/300K_DOS/300K_1_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS_D = f["dos/total"][:]
    C_DOS_D = f["dos/C"][:]*0.0008513328868254667
    Mn_DOS_D = f["dos/Mn"][:]*0.05712998184213628
    Fe_DOS_D = f["dos/Fe"][:]*0.056755525788372854
    N_DOS_D = f["dos/N"][:]* 0.42566644341287824
    Na_DOS_D = f["dos/Na"][:]*0.4595967160697872

    E = np.linspace(0, 413.4, 1000)
    wave_num = E*8.1

    #E = f["dos/axes/time"][:]


In [ ]:
#file_path = "./hydration/QENS/300K_9/400K_9_DISF_1000.mda"
file_path = "./hydration/QENS/300K_DOS/300K_2_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS_H = f["dos/total"][:]
    C_DOS_H = f["dos/C"][:]*1.8287699980021953e-05
    Mn_DOS_H = f["dos/Mn"][:]*0.0012272237851505454
    Fe_DOS_H = f["dos/Fe"][:]* 0.0012191799986683107
    N_DOS_H = f["dos/N"][:]* 0.009143849990014089
    Na_DOS_H = f["dos/Na"][:]*0.009872714874940243
    O_DOS_H = f["dos/O"][:]*4.87671999467346e-06
    H_DOS_H = f["dos/H"][:]*0.9785138669312522
    
    E = np.linspace(0, 413.4, 1000)
    wave_num = E*8.1

    #E = f["dos/axes/time"][:]


In [ ]:
#file_path = "./hydration/QENS/300K_9/400K_9_DISF_1000.mda"
file_path = "./hydration/QENS/400K_11/400K_11_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS_D = f["dos/total"][:]
    C_DOS_D = f["dos/C"][:]* 0.0009016126660181753
    Mn_DOS_D = f["dos/Mn"][:]*0.060504082522098636
    Fe_DOS_D = f["dos/Fe"][:]*0.060107511067892146
    N_DOS_D = f["dos/N"][:]*0.4508063330090505
    Na_DOS_D = f["dos/Na"][:]*0.42768046073494054

    E = np.linspace(0, 413.4, 1000)
    wave_num = E*8.1

    #E = f["dos/axes/time"][:]


In [ ]:
#file_path = "./hydration/QENS/300K_9/400K_9_DISF_1000.mda"
file_path = "./hydration/QENS/400K_7/400K_7_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
    C_DOS = f["dos/C"][:]#*weights['C']
    Mn_DOS = f["dos/Mn"][:]#*weights['Mn']
    Na_DOS = f["dos/Na"][:]#*weights['Na']
    N_DOS = f["dos/N"][:]#*weights['N']
    O_DOS = f["dos/O"][:]#*weights['O']
    H_DOS = f["dos/H"][:]#*weights['H']
    Fe_DOS = f["dos/Fe"][:]#*weights['Fe']

    E = f["dos/axes/time"][:]
    wave_num = E*8.1


In [ ]:
E = np.linspace(0, 413.4, 1000)
E

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# List of datasets and labels
datasets = [
    (total_DOS, "Total"),
    (Na_DOS,  "Na"),
    #(O_DOS,  "O"),
    (H_DOS, "H"),
    (C_DOS, "C"),
    #(N_DOS, "N"),
    (Fe_DOS, "Fe"),
    (Mn_DOS, "Mn")
]

indices = range(105)   # frames to plot
n = len(indices)

cmap = plt.get_cmap("jet")

fig, axes = plt.subplots(3, 2, figsize=(singlecol*2, singlecol*3), sharex=True)
axes = axes.ravel()

for ax, (data, label) in zip(axes, datasets):

    for idx, i in enumerate(indices):
        color = cmap(idx / (n - 1))
        ax.plot(E*8.1, data, label=f"t={i}")

    ax.set_title(label)
    ax.set_xlabel("wavenumber (cm-1)")
    ax.set_ylabel("S(q, w)")
    #ax.set_xlim(0,150)
    #ax.set_yscale("log")
    ax.grid(True)


plt.tight_layout()
plt.show()


In [ ]:
%matplotlib widget
plt.figure(figsize=(singlecol*2, singlecol*2))
plt.plot(wave_num, total_DOS_H, label='total')
plt.plot(wave_num, total_DOS_D, label='total')
#plt.plot(E, (Na_DOS+H_DOS+O_DOS+C_DOS+N_DOS+Mn_DOS+Fe_DOS), label='Sum')
plt.plot(wave_num, Na_DOS_D, label='Na')
#plt.plot(E, H_DOS, label='H')
#plt.plot(E, O_DOS, label='O')
plt.plot(wave_num, C_DOS_D, label='C')
plt.plot(wave_num, N_DOS_D, label='N')
plt.plot(wave_num, Mn_DOS_D, label='Mn')
plt.plot(wave_num, Fe_DOS_D, label='Fe')
plt.xlabel("wave number (cm-1)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
plt.xlim(400,3500)
plt.tight_layout()

In [ ]:
%matplotlib widget
plt.figure(figsize=(singlecol*2, singlecol*2))
plt.plot(wave_num, total_DOS_D, label='total')
#plt.plot(E, (Na_DOS+H_DOS+O_DOS+C_DOS+N_DOS+Mn_DOS+Fe_DOS), label='Sum')
plt.plot(wave_num, Na_DOS_D, label='Na')
plt.plot(wave_num, H_DOS, label='H')
plt.plot(wave_num, O_DOS, label='O')
plt.plot(wave_num, C_DOS_D, label='C')
plt.plot(wave_num, N_DOS_D, label='N')
plt.plot(wave_num, Mn_DOS_D, label='Mn')
plt.plot(wave_num, Fe_DOS_D, label='Fe')
plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.yticks([])
plt.legend(frameon=False)
#plt.xlim(0,150)
plt.tight_layout()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv('./hydration/hydrated_IR.CSV', sep=';', decimal=',', header=None)
df.columns = ['Wavenumber', 'Transmittance']

# 2. Convert Transmittance (%) to Absorbance
# We use np.where to avoid log10(0) errors if the data contains 0% transmittance
df['Absorbance'] = np.where(
    df['Transmittance'] > 0, 
    2 - np.log10(df['Transmittance']), 
    0  # Or np.nan if you prefer to mask invalid data
)

# 3. Plotting Absorbance
plt.figure(figsize=(10, 6))
plt.plot(df['Wavenumber'], df['Absorbance'], color='darkred', linewidth=1.2)

df = pd.read_csv('./hydration/dehydrated_IR.CSV', sep=';', decimal=',', header=None)
df.columns = ['Wavenumber', 'Transmittance']

# 2. Convert Transmittance (%) to Absorbance
# We use np.where to avoid log10(0) errors if the data contains 0% transmittance
df['Absorbance'] = np.where(
    df['Transmittance'] > 0, 
    2 - np.log10(df['Transmittance']), 
    0  # Or np.nan if you prefer to mask invalid data
)

# 3. Plotting Absorbance
plt.plot(df['Wavenumber'], df['Absorbance'], color='darkred', linewidth=1.2)

# Standard IR formatting: High to Low wavenumber
plt.gca().invert_xaxis()
plt.xlabel(r'Wavenumber ($cm^{-1}$)', fontsize=12)
plt.ylabel('Absorbance', fontsize=12)
#plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### IR

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import h5py

plt.rcParams.update({'font.size': 8})

E = np.linspace(0, 413.4, 1000)
wave_num = E*8.1

fig, (ax1, ax2, ax3) = plt.subplots(
    3, 1,
    figsize=(singlecol*2, singlecol*1.75),
    sharex=True
)


df = pd.read_csv('./hydration/hydrated_IR.CSV', sep=';', decimal=',', header=None)
df.columns = ['Wavenumber', 'Transmittance']

# 2. Convert Transmittance (%) to Absorbance
# We use np.where to avoid log10(0) errors if the data contains 0% transmittance
df['Absorbance'] = np.where(
    df['Transmittance'] > 0, 
    2 - np.log10(df['Transmittance']), 
    0  # Or np.nan if you prefer to mask invalid data
)

ax1.plot(df['Wavenumber'], df['Absorbance'], label='Hydrated', linewidth=1.2)

df = pd.read_csv('./hydration/dehydrated_IR.CSV', sep=';', decimal=',', header=None)
df.columns = ['Wavenumber', 'Transmittance']

# 2. Convert Transmittance (%) to Absorbance
# We use np.where to avoid log10(0) errors if the data contains 0% transmittance
df['Absorbance'] = np.where(
    df['Transmittance'] > 0, 
    2 - np.log10(df['Transmittance']), 
    0  # Or np.nan if you prefer to mask invalid data
)

# 3. Plotting Absorbance
ax1.plot(df['Wavenumber'], df['Absorbance'], label='Dehydrated', linewidth=1.2)

ax1.legend(frameon=False)

file_path = "./hydration/QENS/300K_DOS/300K_2_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS_H = f["dos/total"][:]
    C_DOS_H  = f["dos/C"][:]  * 1.8287699980021953e-05
    Mn_DOS_H = f["dos/Mn"][:] * 0.0012272237851505454
    Fe_DOS_H = f["dos/Fe"][:] * 0.0012191799986683107
    N_DOS_H  = f["dos/N"][:]  * 0.009143849990014089
    Na_DOS_H = f["dos/Na"][:] * 0.009872714874940243
    O_DOS_H  = f["dos/O"][:]  * 4.87671999467346e-06
    H_DOS_H  = f["dos/H"][:]  * 0.9785138669312522

color_ls = ['grey', 'blue', 'red', 'lightpink', 'gold', 'darkorange', 'purple']

data = [
    C_DOS_H * 10000,
    N_DOS_H * 100,
    O_DOS_H * 1e6,
    H_DOS_H * 0.1,
    Na_DOS_H,
    Fe_DOS_H * 100,
    Mn_DOS_H * 10000
]

label = [
    r'C × $10^{4}$',
    r'N × $10^{2}$',
    r'O × $10^{6}$',
    r'H × $10^{-1}$',
    'Na',
    r'Fe × $10^{2}$',
    r'Mn × $10^{4}$'
]

for d, l, c in zip(data, label, color_ls):
    ax2.plot(wave_num, d, label=l, color=c)

ax2.vlines(wave_num[N_DOS_H[400:].argmax()+400], -0.0001, 0.001, linestyle='--')
ax2.text(wave_num[N_DOS_H[400:].argmax()+400]-20, 0.001*0.9, f'{round(wave_num[N_DOS_H[400:].argmax()+400],2)}')

file_path = "./hydration/QENS/300K_DOS/300K_1_DOS.mda"

with h5py.File(file_path, "r") as f:
    total_DOS_D = f["dos/total"][:]
    C_DOS_D  = f["dos/C"][:]  * 0.0008513328868254667
    Mn_DOS_D = f["dos/Mn"][:] * 0.05712998184213628
    Fe_DOS_D = f["dos/Fe"][:] * 0.056755525788372854
    N_DOS_D  = f["dos/N"][:]  * 0.42566644341287824
    Na_DOS_D = f["dos/Na"][:] * 0.4595967160697872

color_ls = ['grey', 'blue', 'gold', 'darkorange', 'purple']

data = [
    C_DOS_D * 1e2,
    N_DOS_D * 1,
    Na_DOS_D,
    Fe_DOS_D * 1,
    Mn_DOS_D * 1
]

label = [
    r'C × $10^{2}$',
    r'N × $10^{2}$',
    'Na',
    'Fe',
    'Mn'
]

for d, l, c in zip(data, label, color_ls):
    ax3.plot(wave_num, d, label=l, color=c)

# =========================
for ax in [ax3]:
    ax.set_xlabel("Wavenumber (cm$^{-1}$)")
    ax.set_xlim(3500, 400)
    ax.set_yticks([])
    ax.legend(frameon=False)

ax3.vlines(wave_num[N_DOS_D[400:].argmax()+400], -0.0001, 0.0008, linestyle='--', color='b')
ax3.text(wave_num[N_DOS_D[400:].argmax()+400]-20, 0.0008*0.9, f'{round(wave_num[N_DOS_D[400:].argmax()+400],2)}')

ax2.set_xlim(3700, 400)
ax2.legend(frameon=False,  ncol=2)
ax1.set_yticks([])
ax2.set_yticks([])
ax1.set_ylim(-0.05, 0.55)
ax2.set_ylim(-0.0001, 0.001)
ax3.set_ylim(-0.0001, 0.0008)
ax1.set_ylabel("Absorbance (a.u.)")
ax2.set_ylabel("Intensity (a.u.)")
ax3.set_ylabel("Intensity (a.u.)")

plt.tight_layout()
plt.savefig('./hydration/IR_DOS.svg', dpi=300)

In [ ]:
plt.figure()
plt.plot(total_DOS_H)

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*2))
plt.plot(E, total_DOS, label='total')
#plt.plot(E, (Na_DOS+H_DOS+O_DOS+C_DOS+N_DOS+Mn_DOS+Fe_DOS), label='Sum')
#plt.plot(E, Na_DOS, label='Na')
plt.plot(E, H_DOS, label='H')
#plt.plot(E, O_DOS, label='O')
#plt.plot(E, C_DOS, label='C')
#plt.plot(E, N_DOS, label='N')
#plt.plot(E, Mn_DOS, label='Mn')
#plt.plot(E, Fe_DOS, label='Fe')
plt.xlim(0,150)
plt.legend()

In [ ]:
plt.figure(figsize=(singlecol*2, singlecol*1.5))

name_ls = ['x = 0.25, y = 2',
           'x = 1, y = 2',
           'x = 1.75, y = 2',
           'x = 1.8, y = 0']
id_ls = [7, 8, 9, 11]

for i in [3,2]:
    file_path = f"./hydration/QENS/400K_{id_ls[i]}/400K_{id_ls[i]}_DOS.mda"
    with h5py.File(file_path, "r") as f:
        total_DOS = f["dos/total"][:]

    plt.plot(E, total_DOS, label=f'{name_ls[i]}')

file_path = f"./hydration/QENS/400K_vac/400K_1_DOS.mda"
with h5py.File(file_path, "r") as f:
    total_DOS = f["dos/total"][:]
plt.plot(E, total_DOS, label='vac')

plt.xlabel("Energy (meV)")#, **axis_font)
plt.ylabel("Intensity (a.u.)")#, **axis_font)
plt.legend(frameon=False)
plt.xlim(0,150)
plt.tight_layout()


In [ ]:
    (Fe_F,  "Fe"),
    (Mn_F,  "Mn"),
    (C_F,  "C"),
    (N_F,  "N"),

In [ ]:
plt.figure()

cmap = plt.get_cmap("viridis")

for i in range(31):
    color = cmap(i /31)
    plt.plot(Na_F[i, :], color=color, label=f"t={i}")

#plt.xlim(100-10,100+10)


In [ ]:
import os
import MDAnalysis as mda
from MDANSE import REGISTRY

# =============================================================================
# INPUTS
# =============================================================================
LAMMPS_TRAJ = "./hydration/400_diffusion_4/dump.400_diffuse_9.trj"  # Your trajectory file
LAMMPS_TOPOLOGY = "./hydration/400_diffusion_4/400K_Na_28_O_32.data" # Or a PDB/GROMACS file
OUTPUT_MDT = "./hydration/QENS/400K_9_convert_trj"

input_lammps_trj =  "./hydration/400_diffusion_4/dump.400_diffuse_9.trj"  # Your input file
output_mdt_file = "./hydration/QENS/400K_9_convert_trj"

# Your specific type mapping
type_mapping = {
    '1': 'C', 
    '2': 'Mn', 
    '3': 'Fe', 
    '4': 'N', 
    '5': 'Na', 
    '6': 'O', 
    '7': 'H'
}

# =============================================================================
# PHASE 1: Pre-process with MDAnalysis
# We read the raw LAMMPS dump and apply the element names.
# =============================================================================
print(f"Reading {input_lammps_trj} with MDAnalysis...")

# Load the universe. 
# 'format="LAMMPSDUMP"' handles standard LAMMPS ASCII dumps.
u = mda.Universe(input_lammps_trj, format="LAMMPSDUMP")

# Apply the mapping
# MDAnalysis typically reads LAMMPS types as strings ('1', '2', etc.)
print("Applying atom type mapping...")
new_names = []
new_types = []
elements = []

for atom in u.atoms:
    # Get the original type (e.g., '1')
    orig_type = atom.type
    # Map to element (e.g., 'C'), default to 'X' if not found
    element = type_mapping.get(orig_type, 'X')
    
    new_names.append(element)
    new_types.append(element)
    elements.append(element)

# Update the Universe topology
u.atoms.names = new_names
u.atoms.types = new_types
# MDAnalysis might need mass guesses for writing formats, but for MDANSE names are key.

# =============================================================================
# PHASE 2: Convert to MDANSE (.mdt)
# MDANSE's most stable conversion path is often from PDB (topology) + DCD (traj)
# because PDBs store element names explicitly.
# =============================================================================
print("Creating intermediate topology/trajectory files...")

temp_pdb = "temp_topology.pdb"
temp_dcd = "temp_trajectory.dcd"

# Write Topology (PDB)
u.atoms.write(temp_pdb)

# Write Trajectory (DCD)
# We iterate to handle memory efficiently for large files
with mda.Writer(temp_dcd, u.atoms.n_atoms) as W:
    for ts in u.trajectory:
        W.write(u)

print("Running MDANSE internal converter (DCD -> MDT)...")

# Setup MDANSE Converter Parameters
# We use the 'dcd' converter which takes a topology (PDB) and trajectory (DCD)
parameters = {
    'trajectory': temp_dcd,
    'topology': temp_pdb,
    'output_files': [output_mdt_file]
}

# Run the MDANSE Conversion Job
converter = REGISTRY['converter']['dcd'](None, parameters)
converter.run()

# Cleanup temporary files
os.remove(temp_pdb)
os.remove(temp_dcd)

print(f"Conversion complete! File saved as: {output_mdt_file}")

In [ ]:
import os
from mdanse.conversion.lammps import LammpsConverter

# --- INPUT FILES ---
LAMMPS_TRAJ = "./hydration/400_diffusion_4/dump.400_diffuse_9.trj"  # Your trajectory file
LAMMPS_TOPOLOGY = "./hydration/400_diffusion_4/400K_Na_28_O_32.data" # Or a PDB/GROMACS file
OUTPUT_MDT = "./hydration/QENS/400K_9_convert_trj"

# --- CONVERTER SETUP ---
print(f"Starting conversion of {LAMMPS_TRAJ}...")

converter = LammpsConverter(
    input_file=LAMMPS_TRAJ,
    # The topology file is crucial for atom types, masses, and box info
    topology_file=LAMMPS_TOPOLOGY,
    output_file=OUTPUT_MDT,
    # Specify the atom type mapping if your LAMMPS dump does not contain atom names
    # Example: lammps_atom_types={'1': 'H', '2': 'O'}
    lammps_atom_types={'1': 'C', '2': 'Mn', '3': 'Fe', '4': 'N', '5': 'Na', '6': 'O', '7': 'H'},  # <--- ADJUST THIS TO YOUR ATOM TYPES
    skip_initial_frames=0
)

# Run the conversion
converter.run()
print(f"Conversion complete. MDT file saved as: {OUTPUT_MDT}")

In [ ]:
mass 1 12.10 # C
mass 2 54.94 # Mn
mass 3 55.85 # Fe
mass 4 14.01 # N
mass 5 22.99 # Na
mass 6 15.9994  # O
mass 7 1.00794  # H


In [ ]:
parameters = {
    "atom_aliases": {
        # MDANSE configuration objects commonly expect a "value" key
        # containing the JSON string that describes the mapping.
        "value": (
            '{"mass=12.10": {"1": "C"}, '
            '"mass=54.94": {"2": "Mn"}, '
            '"mass=55.85": {"3": "Fe"}, '
            '"mass=14.01": {"4": "N"}, '
            '"mass=22.99": {"5": "Na"}, '
            '"mass=15.9994": {"6": "O"}, '
            '"mass=1.00794": {"7": "H"}}'
        )
    },
    "config_file": "./hydration/400_diffusion_4/400K_Na_28_O_32.data",
    "fold": False,
    "lammps_units": "metal",
    "n_steps": "0",
    "output_files": (
        "./hydration/QENS/400K_9_convert_trj",  # filename
        64,                                      # datatype / ? (as you had)
        128,                                     # chunk size
        "gzip",                                  # compression
        "INFO",                                  # logfile output level
        True                                     # write_logs (required)
    ),
    "time_step": "0.001",
    "trajectory_file": "./hydration/400_diffusion_4/dump.400_diffuse_9.trj",
    "trajectory_format": "custom",
}

########################################################
# Setup and run the analysis                           #
########################################################

if __name__ == "__main__":
    lammps = Converter.create("LAMMPS")
    # Progress bars only available if tqdm available.
    # Install with `cli` optional dependency.
    lammps.run(parameters, status=True, prog_bar=True)